# 02 — Translation Pipeline Overview

This notebook establishes the descriptive baseline for the `Digital Humanities` translation run. I use it to answer a practical question before moving into disagreement analysis: what data actually exists, where does coverage fail, and which outputs need review before they can be treated as evidence?

The notebook works at three levels:

- **Coverage**: which services produced translations for which language rows, and how that varies by family, prompt, and service type.
- **Quality signals**: which outputs show structural problems such as missing rationales, refusals, repetition loops, source-term leakage, script mixing, or implausible script choices.
- **Review state**: which automated signals and manual exclusions are written back to the evaluation folder for later notebooks.

I keep community references, machine-translation baselines, and prompt-based LLM outputs separate. Wikipedia is a community translation source, not an MT baseline. Google Translate, EasyNMT, and Lingvanex are direct MT baselines. The LLM services vary by prompt, so most coverage views either collapse them to "any successful prompt" or explicitly show the service/prompt matrix.

This notebook also reads the historic `Digital Humanities` reference-convergence files generated by `scripts/analysis/build_historic_reference_convergence.py`. Those files let me compare the December 13, 2022 flattened artifact and the November 2024 initial prompt-reference artifact without relying on the old output folder.




## Unit of Analysis

The unit of analysis is a language row for the target term `Digital Humanities`. For prompt-invariant sources, one row per language is enough. For LLM services, each language can appear under multiple prompt variants, so I treat service, prompt, and language as separate cells when checking completeness and failures.

I also separate automated and manual review layers:

- **Automated review signals** identify language-level evidence that something may be wrong or analytically fragile.
- **Manual exclusions** are later human decisions saved in `manual_exclusions.csv`; these determine what downstream notebooks exclude from quality analysis, search-term generation, or term correction.


In [1]:
import os
import sys
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

PROJECT_ROOT = Path("..")
if not (PROJECT_ROOT / "scripts" / "utils.py").exists():
    PROJECT_ROOT = Path(".")
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.utils import read_csv_file, get_language_family, save_chart
from scripts.analysis.explore_confidence_within_variant import load_variant_df

DATA_DIR = PROJECT_ROOT / "datasets"
FIGURE_DIR = PROJECT_ROOT / "figures" / "dhq_article"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TERMS = ["Digital Humanities"]
TARGET_TERMS = TERMS
VARIANTS = ["minimal", "fluent_speaker", "github_searcher", "judge"]

COMMUNITY_TRANSLATION_SERVICES = {
    "Wikipedia": "wikipedia_translated_term",
}
MT_BASELINE_SERVICES = {
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
}
PROMPT_INVARIANT_SERVICES = {**COMMUNITY_TRANSLATION_SERVICES, **MT_BASELINE_SERVICES}

# Backward-compatible alias used by older cells; notebook prose avoids calling
# Wikipedia a baseline.
BASELINE_SERVICES = PROMPT_INVARIANT_SERVICES

LLM_SERVICES = {
    "OpenAI":   "openai_translated_term",
    "Claude":   "claude_translated_term",
    "Gemini":   "gemini_translated_term",
    "DeepSeek": "deepseek_translated_term",
    "Llama":    "llama_translated_term",
    "Gemma":    "gemma_translated_term",
    "Qwen":     "qwen_translated_term",
    "Mistral":  "mistral_translated_term",
}
SERVICE_COLS = {**BASELINE_SERVICES, **LLM_SERVICES}



## Prompt Conditions

I use prompt variation to make the pipeline's accumulated design choices visible rather than to identify a single optimal prompt. The first three conditions generate translations independently and vary persona, rationale language, and intended use. The judge runs afterward and synthesizes evidence from those prompts, the direct translation services, and Wikipedia.


In [2]:
from great_tables import GT

prompt_conditions_df = pd.DataFrame([
    {
        'condition': 'Minimal',
        'system_framing': 'None; system message omitted',
        'instruction_evidence': 'Translate the term using the target-language name and ISO code; return a translation and rationale as JSON.',
        'relationship': 'Independent',
        'analytical_purpose': 'Baseline without a persona or application context.',
    },
    {
        'condition': 'Fluent speaker',
        'system_framing': 'Multilingual Digital Humanities scholar',
        'instruction_evidence': 'Translate the term and write the rationale in the target language.',
        'relationship': 'Independent',
        'analytical_purpose': 'Tests scholarly role framing and target-language reasoning.',
    },
    {
        'condition': 'GitHub searcher',
        'system_framing': 'Researcher identifying Digital Humanities communities on GitHub',
        'instruction_evidence': 'Produce a translation for finding repositories, users, issues, and topics related to Digital Humanities.',
        'relationship': 'Independent',
        'analytical_purpose': 'Tests whether intended downstream use changes the proposed term.',
    },
    {
        'condition': 'Judge',
        'system_framing': 'Multilingual Digital Humanities scholar',
        'instruction_evidence': 'Evaluate distinct candidates from the first three prompts, direct translation services, and Wikipedia; select or improve the result.',
        'relationship': 'Dependent synthesis',
        'analytical_purpose': 'Tests synthesis under supplied evidence rather than independent translation.',
    },
])

prompt_conditions_table = (
    GT(prompt_conditions_df)
    .tab_header(
        title='Prompt Conditions in the Translation Experiment',
        subtitle=(
            'The first three conditions generate translations independently; '
            'the judge evaluates candidates produced by earlier prompts and sources.'
        ),
    )
    .cols_label(
        condition='Condition',
        system_framing='System framing',
        instruction_evidence='User instruction and supplied evidence',
        relationship='Relation to other prompts',
        analytical_purpose='Analytical purpose',
    )
    .cols_width(
        condition='125px',
        system_framing='230px',
        instruction_evidence='390px',
        relationship='145px',
        analytical_purpose='290px',
    )
    .tab_options(
        column_labels_font_weight='bold',
        column_labels_vlines_style='solid',
        column_labels_vlines_width='1px',
        column_labels_vlines_color='#A9A9A9',
        table_body_vlines_style='solid',
        table_body_vlines_width='1px',
        table_body_vlines_color='#D0D0D0',
    )
    .tab_source_note(
        source_note=(
            'Source: scripts/candidate_generation/translation_prompts.py. '
            'All conditions request a translated term and rationale in JSON. '
            'Identical judge candidates are grouped so agreement among sources is visible.'
        )
    )
)
display(prompt_conditions_table)
article_figures_dir = FIGURE_DIR
finalized_figures_dir = PROJECT_ROOT / "figures" / "finalized_dhq_article"
finalized_figures_dir.mkdir(parents=True, exist_ok=True)
article_figures_dir.mkdir(parents=True, exist_ok=True)
prompt_conditions_table_path = article_figures_dir / 'prompt_conditions_table.png'
prompt_conditions_table.gtsave(str(prompt_conditions_table_path))
print("Saved article table figure.")


GT(_tbl_data=         condition                                     system_framing  \
0          Minimal                       None; system message omitted   
1   Fluent speaker            Multilingual Digital Humanities scholar   
2  GitHub searcher  Researcher identifying Digital Humanities comm...   
3            Judge            Multilingual Digital Humanities scholar   

                                instruction_evidence         relationship  \
0  Translate the term using the target-language n...          Independent   
1  Translate the term and write the rationale in ...          Independent   
2  Produce a translation for finding repositories...          Independent   
3  Evaluate distinct candidates from the first th...  Dependent synthesis   

                                  analytical_purpose  
0  Baseline without a persona or application cont...  
1  Tests scholarly role framing and target-langua...  
2  Tests whether intended downstream use changes ...  
3  Tests synthesis under supplied evidence rather...  , _body=<great_tables._gt_data.Body object at 0x11c75e900>, _boxhead=Boxhead([ColInfo(var='condition', type=<ColInfoTypeEnum.default: 1>, column_label='Condition', column_align='left', column_width='125px'), ColInfo(var='system_framing', type=<ColInfoTypeEnum.default: 1>, column_label='System framing', column_align='left', column_width='230px'), ColInfo(var='instruction_evidence', type=<ColInfoTypeEnum.default: 1>, column_label='User instruction and supplied evidence', column_align='left', column_width='390px'), ColInfo(var='relationship', type=<ColInfoTypeEnum.default: 1>, column_label='Relation to other prompts', column_align='left', column_width='145px'), ColInfo(var='analytical_purpose', type=<ColInfoTypeEnum.default: 1>, column_label='Analytical purpose', column_align='left', column_width='290px')]), _stub=<great_tables._gt_data.Stub object at 0x11c75e7b0>, _spanners=Spanners([]), _heading=Heading(title='Prompt Conditions in the Translation Experiment', subtitle='The first three conditions generate translations independently; the judge evaluates candidates produced by earlier prompts and sources.', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11c75ee40>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11c79e0d0>, _source_notes=['Source: scripts/candidate_generation/translation_prompts.py. All conditions request a translated term and rationale in JSON. Identical judge candidates are grouped so agreement among sources is visible.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11c75ef90>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value

Saved article table figure.


In [3]:
def load_all_variants(data_dir, term, variants=VARIANTS):
    """Merge per-service files for all variants into one concatenated DataFrame."""
    dfs = []
    term_slug = term.lower().replace(" ", "_")
    for variant in variants:
        df = load_variant_df(data_dir, term_slug, variant)
        if df is not None:
            df["term_source_query"] = term
            dfs.append(df)
        else:
            print(f"  No data for variant: {variant}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

all_dfs = {term: load_all_variants(DATA_DIR, term) for term in TARGET_TERMS}
for term, df in all_dfs.items():
    print(f"{term}: {len(df):,} rows across {df['prompt_variant'].nunique()} variants")

term = TARGET_TERMS[0]
term_slug = term.lower().replace(" ", "_")
all_terms_df = all_dfs[term]
ref = all_terms_df[all_terms_df["prompt_variant"] == VARIANTS[0]].copy()
n_lang = ref["language_code"].nunique()

community_cols = {s: c for s, c in COMMUNITY_TRANSLATION_SERVICES.items() if c in ref.columns}
mt_baseline_cols = {s: c for s, c in MT_BASELINE_SERVICES.items() if c in ref.columns}
prompt_invariant_cols = {**community_cols, **mt_baseline_cols}
llm_cols = {s: c for s, c in LLM_SERVICES.items() if c in all_terms_df.columns}

prompt_invariant_possible = n_lang * len(prompt_invariant_cols)
prompt_invariant_present = int(sum(ref[col].notna().sum() for col in prompt_invariant_cols.values()))

llm_possible = 0
llm_present = 0
llm_variant_rows = []
for variant in VARIANTS:
    vdf = all_terms_df[all_terms_df["prompt_variant"] == variant]
    v_lang = vdf["language_code"].nunique()
    v_possible = v_lang * len(llm_cols)
    v_present = int(sum(vdf[col].notna().sum() for col in llm_cols.values()))
    llm_possible += v_possible
    llm_present += v_present
    llm_variant_rows.append({
        "variant": variant,
        "languages": v_lang,
        "llm_translation_cells_filled": v_present,
        "llm_translation_cells_possible": v_possible,
        "fill_rate": v_present / v_possible if v_possible else 0,
    })

total_translations_present = prompt_invariant_present + llm_present
total_translations_possible = prompt_invariant_possible + llm_possible

inventory_rows = [
    {"unit": "languages", "scope": "minimal variant language rows", "count": n_lang},
    {"unit": "prompt variants", "scope": "loaded variants", "count": all_terms_df["prompt_variant"].nunique()},
    {"unit": "community translation sources", "scope": "Wikipedia interlanguage-link labels", "count": len(community_cols)},
    {"unit": "MT baseline services", "scope": "prompt-invariant MT columns present", "count": len(mt_baseline_cols)},
    {"unit": "LLM services", "scope": "prompt-varying columns present", "count": len(llm_cols)},
    {"unit": "prompt-invariant translation cells", "scope": "community + MT sources, minimal variant only", "count": prompt_invariant_present, "possible": prompt_invariant_possible, "fill_rate": prompt_invariant_present / prompt_invariant_possible if prompt_invariant_possible else 0},
    {"unit": "LLM translation cells", "scope": "language x service x variant", "count": llm_present, "possible": llm_possible, "fill_rate": llm_present / llm_possible if llm_possible else 0},
    {"unit": "total translations", "scope": "prompt-invariant sources once + LLM outputs across all variants", "count": total_translations_present, "possible": total_translations_possible, "fill_rate": total_translations_present / total_translations_possible if total_translations_possible else 0},
]

eval_dir = os.path.join(DATA_DIR, "translated_terms", term_slug, "evaluation")
signals_path = os.path.join(eval_dir, "automated_review_signals.csv")
manual_path = os.path.join(eval_dir, "manual_exclusions.csv")
if os.path.exists(signals_path):
    _signal_rows = len(pd.read_csv(signals_path, converters={"language_code": str}))
    inventory_rows.append({"unit": "automated review signal rows", "scope": "language-level review evidence", "count": _signal_rows})
if os.path.exists(manual_path):
    _manual_rows = len(pd.read_csv(manual_path, dtype=str).fillna(""))
    inventory_rows.append({"unit": "manual review rows", "scope": "manual_exclusions.csv", "count": _manual_rows})

inventory_df = pd.DataFrame(inventory_rows)
print("Data inventory and denominators:")
display(inventory_df)

llm_variant_df = pd.DataFrame(llm_variant_rows)
print("LLM coverage denominator by prompt variant:")
display(llm_variant_df)


Digital Humanities: 3,524 rows across 4 variants
Data inventory and denominators:


,unit,scope,count,possible,fill_rate
0,languages,minimal variant language rows,881,NaN,NaN
1,prompt variants,loaded variants,4,NaN,NaN
2,community translation sources,Wikipedia interlanguage-link labels,1,NaN,NaN
3,MT baseline services,prompt-invariant MT columns present,3,NaN,NaN
4,LLM services,prompt-varying columns present,8,NaN,NaN
5,prompt-invariant translation cells,"community + MT sources, minimal variant only",491,3524.0,0.139330
6,LLM translation cells,language x service x variant,27451,28192.0,0.973716
7,total translations,prompt-invariant sources once + LLM outputs ac...,27942,31716.0,0.881006
8,automated review signal rows,language-level review evidence,881,NaN,NaN
9,manual review rows,manual_exclusions.csv,599,NaN,NaN


LLM coverage denominator by prompt variant:


,variant,languages,llm_translation_cells_filled,llm_translation_cells_possible,fill_rate
0,minimal,881,6750,7048,0.957719
1,fluent_speaker,881,6775,7048,0.961266
2,github_searcher,881,6893,7048,0.978008
3,judge,881,7033,7048,0.997872


## 2.1 Pipeline Coverage

I begin with coverage rather than accuracy. A missing translation, a logged failure, and an implausible but non-empty answer are different failure modes. This section first asks which services produced anything at all, then separates prompt-specific missingness and historic reference convergence.


In [4]:
term = TARGET_TERMS[0]
df = all_dfs[term].copy()
ref = df[df["prompt_variant"] == VARIANTS[0]].copy()
total = ref["language_code"].nunique()

coverage_rows = []

# Community translation and MT baselines are prompt-invariant, so count
# them once from the minimal-variant reference frame.
for service, col in PROMPT_INVARIANT_SERVICES.items():
    n = int(ref[col].notna().sum()) if col in ref.columns else 0
    coverage_rows.append({
        "service": service,
        "service_class": "Community translation" if service in COMMUNITY_TRANSLATION_SERVICES else "MT baseline",
        "languages_covered": n,
        "total_languages": total,
        "coverage_rate": n / total if total else 0,
    })

# LLM services vary by prompt. For this overview snapshot, collapse to any
# successful prompt variant; prompt-specific counts appear in the service × prompt outcome matrix below.
for service, col in LLM_SERVICES.items():
    if col in df.columns:
        any_by_lang = df.groupby("language_code")[col].agg(lambda x: x.notna().any())
        n = int(any_by_lang.sum())
    else:
        n = 0
    coverage_rows.append({
        "service": service,
        "service_class": "LLM: any prompt",
        "languages_covered": n,
        "total_languages": total,
        "coverage_rate": n / total if total else 0,
    })

service_coverage = pd.DataFrame(coverage_rows)
service_order = (
    service_coverage.sort_values(["service_class", "languages_covered"], ascending=[True, False])
    ["service"].tolist()
)

coverage_chart = alt.Chart(service_coverage).mark_bar().encode(
    y=alt.Y("service:N", sort=service_order, title=None),
    x=alt.X("languages_covered:Q", scale=alt.Scale(domain=[0, total]), title="languages covered"),
    color=alt.Color(
        "service_class:N",
        scale=alt.Scale(domain=["Community translation", "MT baseline", "LLM: any prompt"], range=["#388e3c", "#8d6e63", "#1976d2"]),
        title=None,
    ),
    tooltip=[
        "service:N",
        "service_class:N",
        alt.Tooltip("languages_covered:Q", title="languages covered"),
        alt.Tooltip("total_languages:Q", title="total languages"),
        alt.Tooltip("coverage_rate:Q", format=".1%", title="coverage rate"),
    ],
).properties(width=430, height=300, title=f"Service availability snapshot ({total:,} languages)")

coverage_text = coverage_chart.mark_text(align="left", dx=4, fontSize=10).encode(
    text="languages_covered:Q",
    color=alt.value("black"),
)

display(coverage_chart + coverage_text)
display(
    service_coverage.sort_values("languages_covered", ascending=False)
    .style.format({"coverage_rate": "{:.1%}"})
)


alt.LayerChart(...)

,service,service_class,languages_covered,total_languages,coverage_rate
4,OpenAI,LLM: any prompt,881,881,100.0%
5,Claude,LLM: any prompt,881,881,100.0%
6,Gemini,LLM: any prompt,881,881,100.0%
7,DeepSeek,LLM: any prompt,881,881,100.0%
8,Llama,LLM: any prompt,881,881,100.0%
9,Gemma,LLM: any prompt,881,881,100.0%
10,Qwen,LLM: any prompt,881,881,100.0%
11,Mistral,LLM: any prompt,881,881,100.0%
1,Google Translate,MT baseline,242,881,27.5%
3,Lingvanex,MT baseline,109,881,12.4%


### Historic Reference Convergence

The current pipeline separates direct MT baselines, Wikipedia community references, and prompt-based LLM services. For the historic section, I now keep two older artifacts distinct because they answer different questions.

First, `historic_materials/translated_dh_terms.csv` is the December 13, 2022 flattened artifact. For the `Digital Humanities` rows, it lets me compare the pre-existing/Wikipedia-derived `term` field with the historic Google Translate `translated_term` field. Second, `historic_materials/initial_translated_terms.csv` is the November 2024 initial prompt-reference artifact. It lets me compare Wikipedia, Google Translate, EasyNMT, and OpenAI outputs side by side.

I read precomputed convergence tables generated by `scripts/analysis/build_historic_reference_convergence.py`. Keeping this calculation in a script makes it reproducible and keeps it aligned with the downstream exploration workflow. The script hard-filters both files to `term_source == "Digital Humanities"` and writes separate `historic_2022_*.csv` and `historic_2024_*.csv` outputs into `datasets/translated_terms/digital_humanities/evaluation/`.

Agreement here means exact agreement after light Unicode, case, and whitespace normalization. It does not count semantic similarity or near-synonyms. That narrow definition is intentional: I use it to measure convergence between available references, not to judge whether two different translations might both be acceptable.




In [5]:
historic_eval_dir = Path(DATA_DIR) / "translated_terms" / "digital_humanities" / "evaluation"
historic_suffixes = {
    "reference_artifacts": "reference_artifacts.csv",
    "reference_service_presence": "reference_service_presence.csv",
    "pairwise_reference_agreement": "pairwise_reference_agreement.csv",
    "reference_count_summary": "reference_count_summary.csv",
    "all_source_convergence_summary": "all_source_convergence_summary.csv",
    "reference_convergence_detail": "reference_convergence_detail.csv",
}
historic_sources = {
    "historic_2022": "December 13, 2022: Wikipedia/pre-existing term vs historic Google Translate",
    "historic_2024": "November 2024: Wikipedia, GT, EasyNMT, and OpenAI prompt-reference outputs",
}

historic_required = {
    (prefix, name): historic_eval_dir / f"{prefix}_{suffix}"
    for prefix in historic_sources
    for name, suffix in historic_suffixes.items()
}
missing = [str(path) for path in historic_required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing historic convergence output(s). Run this before executing notebook 02:\n"
        "  python scripts/analysis/build_historic_reference_convergence.py --term 'Digital Humanities'\n\n"
        + "\n".join(missing)
    )

def _load_historic_outputs(prefix):
    outputs = {
        name: pd.read_csv(historic_required[(prefix, name)], dtype=str, keep_default_na=False)
        for name in historic_suffixes
    }
    numeric_specs = [
        ("reference_artifacts", ["rows", "languages", "terms", "nonblank_translation_rows"]),
        ("reference_service_presence", ["rows_with_translation", "total_rows", "coverage_rate"]),
        ("pairwise_reference_agreement", ["overlap", "exact_normalized_agreements", "agreement_rate"]),
        ("reference_count_summary", ["n_sources_present", "rows", "sources_agree", "outputs_all_agree", "agreement_rate"]),
        ("all_source_convergence_summary", ["rows", "share"]),
        ("reference_convergence_detail", ["n_sources_present", "n_distinct_translations"]),
    ]
    for frame_name, numeric_cols in numeric_specs:
        frame = outputs[frame_name]
        for col in numeric_cols:
            if col in frame.columns:
                frame[col] = pd.to_numeric(frame[col], errors="coerce")
    detail = outputs["reference_convergence_detail"]
    if "sources_agree" in detail.columns:
        detail["sources_agree"] = detail["sources_agree"].astype(str).str.lower().eq("true")
    return outputs

historic_outputs = {prefix: _load_historic_outputs(prefix) for prefix in historic_sources}

def _convergence_rows(prefix):
    frame = historic_outputs[prefix]["all_source_convergence_summary"].copy()
    return frame.set_index("convergence_category")["rows"].to_dict()

def _article_summary_row(prefix, label):
    artifact = historic_outputs[prefix]["reference_artifacts"].iloc[0]
    presence = historic_outputs[prefix]["reference_service_presence"].copy()
    pairwise = historic_outputs[prefix]["pairwise_reference_agreement"].copy()
    source_count = historic_outputs[prefix]["reference_count_summary"].copy()
    convergence = _convergence_rows(prefix)
    total_rows = int(artifact["rows"])
    all_sources_n = int(source_count["n_sources_present"].max())
    all_sources = source_count[source_count["n_sources_present"].eq(all_sources_n)]
    all_source_overlap = int(all_sources["rows"].iloc[0]) if not all_sources.empty else 0
    all_source_agree = int(all_sources["sources_agree"].iloc[0]) if not all_sources.empty else 0
    all_source_agreement_rate = (all_source_agree / all_source_overlap) if all_source_overlap else 0
    source_coverage = "; ".join(
        f"{row.source}: {int(row.rows_with_translation)}/{int(row.total_rows)} ({row.coverage_rate:.1%})"
        for row in presence.itertuples()
    )
    pairwise_agreement = "; ".join(
        f"{row.comparison}: {int(row.exact_normalized_agreements)}/{int(row.overlap)} ({row.agreement_rate:.1%})"
        for row in pairwise.itertuples()
    )
    return {
        "historic_stage": label,
        "rows": total_rows,
        "source_coverage": source_coverage,
        "rows_with_2plus_sources": int(source_count.loc[source_count["n_sources_present"].ge(2), "rows"].sum()),
        "rows_where_available_sources_agree": int(convergence.get("all available outputs agree", 0)),
        "rows_where_available_sources_diverge": int(convergence.get("outputs diverge", 0)),
        "all_source_overlap": all_source_overlap,
        "all_source_exact_agreement": all_source_agree,
        "all_source_agreement_rate": all_source_agreement_rate,
        "pairwise_agreement": pairwise_agreement,
    }

historic_article_summary = pd.DataFrame([
    _article_summary_row("historic_2022", "December 13, 2022 flattened artifact"),
    _article_summary_row("historic_2024", "November 2024 prompt-reference artifact"),
])

historic_stage_summary = historic_article_summary[[
    "historic_stage",
    "rows",
    "rows_with_2plus_sources",
    "rows_where_available_sources_agree",
    "rows_where_available_sources_diverge",
    "all_source_overlap",
    "all_source_exact_agreement",
    "all_source_agreement_rate",
]].copy()
historic_stage_summary["multi_source_coverage_rate"] = (
    historic_stage_summary["rows_with_2plus_sources"] / historic_stage_summary["rows"]
)
historic_stage_summary = historic_stage_summary[[
    "historic_stage",
    "rows",
    "rows_with_2plus_sources",
    "multi_source_coverage_rate",
    "rows_where_available_sources_agree",
    "rows_where_available_sources_diverge",
    "all_source_overlap",
    "all_source_exact_agreement",
    "all_source_agreement_rate",
]]

print("Article-facing historic stage summary")
display(
    historic_stage_summary.style.format({
        "multi_source_coverage_rate": "{:.1%}",
        "all_source_agreement_rate": "{:.1%}",
    })
)

print("Detailed source coverage and pairwise agreement for prose drafting")
display(
    historic_article_summary.style.format({
        "all_source_agreement_rate": "{:.1%}",
    })
)

historic_2024_coverage = historic_outputs["historic_2024"]["reference_service_presence"].copy()
historic_2024_coverage["missing_rows"] = (
    historic_2024_coverage["total_rows"] - historic_2024_coverage["rows_with_translation"]
)
historic_2024_coverage["coverage_label"] = (
    historic_2024_coverage["rows_with_translation"].astype(int).astype(str)
    + "/"
    + historic_2024_coverage["total_rows"].astype(int).astype(str)
    + " ("
    + (historic_2024_coverage["coverage_rate"] * 100).round(1).astype(str)
    + "%)"
)
historic_2024_source_order = historic_2024_coverage["source"].tolist()

coverage_total = alt.Chart(historic_2024_coverage).mark_bar(
    color="#e5e7eb",
    cornerRadiusEnd=2,
).encode(
    y=alt.Y(
        "source:N",
        title=None,
        sort=historic_2024_source_order,
        axis=alt.Axis(labelLimit=180),
    ),
    x=alt.X(
        "total_rows:Q",
        title="Number of translations",
        scale=alt.Scale(domain=[0, 190]),
    ),
    tooltip=[
        alt.Tooltip("source:N", title="source"),
        alt.Tooltip("rows_with_translation:Q", title="rows with output"),
        alt.Tooltip("missing_rows:Q", title="missing rows"),
        alt.Tooltip("total_rows:Q", title="total rows"),
        alt.Tooltip("coverage_rate:Q", title="coverage", format=".1%"),
    ],
)

coverage_actual = alt.Chart(historic_2024_coverage).mark_bar(
    color="#2563eb",
    cornerRadiusEnd=2,
).encode(
    y=alt.Y("source:N", sort=historic_2024_source_order),
    x=alt.X("rows_with_translation:Q"),
    tooltip=[
        alt.Tooltip("source:N", title="source"),
        alt.Tooltip("rows_with_translation:Q", title="rows with output"),
        alt.Tooltip("missing_rows:Q", title="missing rows"),
        alt.Tooltip("total_rows:Q", title="total rows"),
        alt.Tooltip("coverage_rate:Q", title="coverage", format=".1%"),
    ],
)

coverage_labels = alt.Chart(historic_2024_coverage).mark_text(
    align="left",
    baseline="middle",
    dx=5,
    color="#111827",
    fontSize=12,
).encode(
    y=alt.Y("source:N", sort=historic_2024_source_order),
    x=alt.X("rows_with_translation:Q"),
    text="coverage_label:N",
)

historic_2024_coverage_chart = (
    coverage_total + coverage_actual + coverage_labels
).properties(
    width=560,
    height=190,
    title={
        "text": "November 2024 Source Coverage",
        "subtitle": "Blue shows rows with output; gray shows the 185-row Digital Humanities denominator",
    },
)

def _agreement_heatmap_data():
    frames = []
    for prefix in historic_sources:
        pairwise = historic_outputs[prefix]["pairwise_reference_agreement"].copy()
        pairwise["stage"] = "December 13, 2022" if prefix == "historic_2022" else "November 2024"
        pairwise["label"] = (
            pairwise["exact_normalized_agreements"].astype(int).astype(str)
            + "/"
            + pairwise["overlap"].astype(int).astype(str)
        )
        frames.append(pairwise)
    return pd.concat(frames, ignore_index=True)

historic_pairwise_long = _agreement_heatmap_data()

pairwise_bars = alt.Chart(historic_pairwise_long).mark_bar(color="#1976d2").encode(
    y=alt.Y("comparison:N", title=None, sort="-x", axis=alt.Axis(labelLimit=330)),
    x=alt.X("agreement_rate:Q", title="Exact-agreement rate among overlapping rows", axis=alt.Axis(format="%")),
    tooltip=[
        alt.Tooltip("stage:N", title="stage"),
        alt.Tooltip("comparison:N", title="comparison"),
        alt.Tooltip("overlap:Q", title="overlap"),
        alt.Tooltip("exact_normalized_agreements:Q", title="exact agreements"),
        alt.Tooltip("agreement_rate:Q", title="agreement rate", format=".1%"),
    ],
).properties(width=560, height=220)

pairwise_labels = alt.Chart(historic_pairwise_long).mark_text(
    align="left", baseline="middle", dx=4, color="#111827"
).encode(
    y=alt.Y("comparison:N", sort="-x"),
    x=alt.X("agreement_rate:Q"),
    text="label:N",
)

pairwise_agreement_chart = (pairwise_bars + pairwise_labels).facet(
    row=alt.Row("stage:N", title=None),
    title={
        "text": "Historic Pairwise Agreement",
        "subtitle": "Agreement means exact match after light Unicode, case, and whitespace normalization",
    },
)

def _heatmap_for_2024():
    prefix = "historic_2024"
    pairwise = historic_outputs[prefix]["pairwise_reference_agreement"].copy()
    source_order = historic_outputs[prefix]["reference_service_presence"]["source"].tolist()
    pairwise["label"] = (
        pairwise["exact_normalized_agreements"].astype(int).astype(str)
        + "/"
        + pairwise["overlap"].astype(int).astype(str)
        + " ("
        + (pairwise["agreement_rate"] * 100).round(1).astype(str)
        + "%)"
    )
    matrix_rows = []
    for _, row in pairwise.iterrows():
        for source_a, source_b in [(row["source_a"], row["source_b"]), (row["source_b"], row["source_a"] )]:
            matrix_rows.append({
                "source_a": source_a,
                "source_b": source_b,
                "agreement_rate": row["agreement_rate"],
                "label": row["label"],
                "agreements": int(row["exact_normalized_agreements"]),
                "overlap": int(row["overlap"]),
            })
    matrix = pd.DataFrame(matrix_rows)
    heatmap = alt.Chart(matrix).mark_rect(stroke="white", strokeWidth=1).encode(
        x=alt.X("source_b:N", title=None, sort=source_order, axis=alt.Axis(labelAngle=0, labelLimit=140)),
        y=alt.Y("source_a:N", title=None, sort=source_order, axis=alt.Axis(labelLimit=180)),
        color=alt.Color(
            "agreement_rate:Q",
            title="agreement rate",
            scale=alt.Scale(scheme="blues", domain=[0, 1]),
        ),
        tooltip=[
            alt.Tooltip("source_a:N", title="source"),
            alt.Tooltip("source_b:N", title="compared with"),
            alt.Tooltip("agreements:Q", title="exact agreements"),
            alt.Tooltip("overlap:Q", title="shared rows"),
            alt.Tooltip("agreement_rate:Q", title="agreement rate", format=".1%"),
        ],
    ).properties(
        width=380,
        height=320,
        title={
            "text": "November 2024 Pairwise Agreement Heatmap",
            "subtitle": "Off-diagonal cells show exact agreements / rows where both sources produced output",
        },
    )
    labels = alt.Chart(matrix).mark_text(fontSize=12, color="#111827").encode(
        x=alt.X("source_b:N", sort=source_order),
        y=alt.Y("source_a:N", sort=source_order),
        text="label:N",
    )
    return heatmap + labels

source_count_rows = []
for prefix in historic_sources:
    stage = "December 13, 2022" if prefix == "historic_2022" else "November 2024"
    frame = historic_outputs[prefix]["reference_count_summary"].copy()
    frame["stage"] = stage
    source_count_rows.append(frame)
historic_source_count = pd.concat(source_count_rows, ignore_index=True)
historic_source_count_long = historic_source_count.melt(
    id_vars=["stage", "n_sources_present"],
    value_vars=["rows", "sources_agree"],
    var_name="measure",
    value_name="count",
)

source_count_chart = alt.Chart(historic_source_count_long).mark_bar().encode(
    x=alt.X("n_sources_present:O", title="Number of historic sources present"),
    y=alt.Y("count:Q", title="Rows"),
    color=alt.Color(
        "measure:N",
        title=None,
        scale=alt.Scale(domain=["rows", "sources_agree"], range=["#90caf9", "#2e7d32"]),
    ),
    tooltip=["stage:N", "n_sources_present:O", "measure:N", "count:Q"],
).properties(width=320, height=230).facet(
    column=alt.Column("stage:N", title=None),
    title={
        "text": "Agreement by Number of Available Historic Sources",
        "subtitle": "Green counts rows where all available outputs agree exactly",
    },
)

article_figures_dir = FIGURE_DIR

november_2024_heatmap = _heatmap_for_2024()
coverage_png = article_figures_dir / "november_2024_source_coverage.png"
coverage_publication_png = finalized_figures_dir / "figure01.png"
heatmap_png = article_figures_dir / "november_2024_pairwise_agreement_heatmap.png"
heatmap_publication_png = finalized_figures_dir / "figure02.png"
save_chart(historic_2024_coverage_chart, str(coverage_png), scale_factor=2)
save_chart(historic_2024_coverage_chart.properties(title=None), str(coverage_publication_png), scale_factor=2)
save_chart(november_2024_heatmap, str(heatmap_png), scale_factor=2)
save_chart(november_2024_heatmap.properties(title=None), str(heatmap_publication_png), scale_factor=2)
print("Saved article chart.")
print("Saved article chart.")

display(historic_2024_coverage_chart)
display(pairwise_agreement_chart)
display(november_2024_heatmap)
display(source_count_chart)

historic_2022_detail = historic_outputs["historic_2022"]["reference_convergence_detail"]
historic_2024_detail = historic_outputs["historic_2024"]["reference_convergence_detail"]

historic_2022_divergence_examples = (
    historic_2022_detail[
        (historic_2022_detail["n_sources_present"] >= 2)
        & (~historic_2022_detail["sources_agree"].astype(bool))
    ][[
        "language_code",
        "language_name",
        "wikipedia_or_preexisting_term",
        "google_translate_historic_output",
        "difference_summary",
    ]]
    .head(10)
)
historic_2024_divergence_examples = (
    historic_2024_detail[
        (historic_2024_detail["n_sources_present"] >= 2)
        & (~historic_2024_detail["sources_agree"].astype(bool))
    ][[
        "language_code",
        "language_name",
        "wikipedia_historic_output",
        "google_translate_historic_output",
        "easynmt_historic_output",
        "openai_historic_output",
        "difference_summary",
    ]]
    .head(10)
)
print("Examples where the December 13, 2022 pre-existing/Wikipedia term and GT output diverged:")
display(historic_2022_divergence_examples)
print("Examples where the November 2024 Wikipedia, GT, EasyNMT, and OpenAI outputs diverged:")
display(historic_2024_divergence_examples)


Article-facing historic stage summary


,historic_stage,rows,rows_with_2plus_sources,multi_source_coverage_rate,rows_where_available_sources_agree,rows_where_available_sources_diverge,all_source_overlap,all_source_exact_agreement,all_source_agreement_rate
0,November 2022 flattened artifact,185,19,10.3%,13,6,19,13,68.4%
1,November 2024 prompt-reference artifact,185,118,63.8%,34,84,22,8,36.4%


Detailed source coverage and pairwise agreement for prose drafting


,historic_stage,rows,source_coverage,rows_with_2plus_sources,rows_where_available_sources_agree,rows_where_available_sources_diverge,all_source_overlap,all_source_exact_agreement,all_source_agreement_rate,pairwise_agreement
0,November 2022 flattened artifact,185,Wikipedia/pre-existing term: 19/185 (10.3%); Historic Google Translate output: 123/185 (66.5%),19,13,6,19,13,68.4%,Wikipedia/pre-existing term vs Historic Google Translate output: 13/19 (68.4%)
1,November 2024 prompt-reference artifact,185,Wikipedia: 34/185 (18.4%); Google Translate: 123/185 (66.5%); EasyNMT: 58/185 (31.4%); OpenAI: 169/185 (91.4%),118,34,84,22,8,36.4%,Wikipedia vs Google Translate: 20/34 (58.8%); Wikipedia vs EasyNMT: 8/22 (36.4%); Wikipedia vs OpenAI: 30/34 (88.2%); Google Translate vs EasyNMT: 12/58 (20.7%); Google Translate vs OpenAI: 50/112 (44.6%); EasyNMT vs OpenAI: 14/52 (26.9%)


Saved article chart.
Saved article chart.


...

alt.FacetChart(...)

Examples where the November 2022 pre-existing/Wikipedia term and GT output diverged:


,language_code,language_name,wikipedia_or_preexisting_term,google_translate_historic_output,difference_summary
2,la,Latin,Humanitates digitales,Digital Humanities,content
3,ta,Tamil,எண்ணிம மனிதவியல்,டிஜிட்டல் மனிதநேயம்,content
6,de,German,Digital Humanities,Digitale Geisteswissenschaften,content
8,it,Italian,Informatica umanistica,Scienze umanistiche digitali,content
11,ar,Arabic,إنسانيات رقمية,العلوم الإنسانية الرقمية,content
14,fi,Finnish,Digitaaliset ihmistieteet,Digitaaliset humanistiset tieteet,content


Examples where the November 2024 Wikipedia, GT, EasyNMT, and OpenAI outputs diverged:


,language_code,language_name,wikipedia_historic_output,google_translate_historic_output,easynmt_historic_output,openai_historic_output,difference_summary
3,af,Afrikaans,,Digitale Geesteswetenskappe,Digitale Menslikhede,Digitale Geesteswetenskappe,content
4,ak,Akan,,Digitals Nnipa Ho Adesua,,Dijital Nnipa Ho Adesua,content
5,am,Amharic,,ዲጂታል ሰብአዊነት,,ዲጂታል ሰውነት,content
7,as,Assamese,,ডিজিটেল মানৱীয় বিজ্ঞান,,ডিজিটেল মানৱবিদ্যা,content
9,ay,Aymara,,Humanidades Digitales uka tuqit yatxataña,,Jach’a tekhnoloqiya ukat jaqi yatiña,content
10,az,Azerbaijani,,Rəqəmsal humanitar elmlər,İnsaniyyətləri,,content
12,be,Belarusian,,Лічбавая гуманітарная навука,,Лічбавыя гуманітарныя навукі,content
13,bg,Bulgarian,,Дигитални хуманитарни науки,(Средредредредредредредредредредредредредредре...,Дигитални хуманитарни науки,content
14,bh,Bihari,,डिजिटल मानविकी के बारे में बतावल गइल बा,,डिजिटल मानविकी,content
16,bm,Bambara,,Hadamadenw ka dɔnniya nimɔrɔma,,Hadamadenw ka dɔ́nmɔgɔlibugɔ,content


The two historic artifacts make different parts of the early workflow visible. In the December 13, 2022 file, only nineteen `Digital Humanities` rows contain both a pre-existing/Wikipedia-derived term and a historic Google Translate output, and thirteen of those agree exactly. By November 2024, the initial prompt-reference file makes the OpenAI comparison possible across a fuller 185-row historic set: Wikipedia supplied 34 rows, GT produced outputs for 123 rows, EasyNMT for 58, and OpenAI for 169. The key result is not simply coverage, but uneven convergence. All four sources overlap in twenty-two rows and exactly agree in eight.




### Coverage by Language Family

Next I group coverage by the reviewed language-family labels from `get_language_family()`. This view shows whether missingness is evenly distributed or concentrated in particular language families. A family-level pattern is not automatically a linguistic explanation, but it helps identify where infrastructure, script, geography, and data availability may be interacting.


In [6]:
HEATMAP_SERVICES = {
    "Wikipedia":        "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT":          "enmt_translated_term",
    "Lingvanex":        "lingvanex_translated_term",
    "OpenAI":           "openai_translated_term",
    "Claude":           "claude_translated_term",
    "Gemini":           "gemini_translated_term",
    "DeepSeek":         "deepseek_translated_term",
    "Llama":            "llama_translated_term",
    "Gemma":            "gemma_translated_term",
    "Qwen":             "qwen_translated_term",
    "Mistral":          "mistral_translated_term",
}
service_order = list(HEATMAP_SERVICES.keys())

term = TARGET_TERMS[0]
all_variants = all_dfs[term].copy()
all_variants["language_family"] = all_variants["language_code"].apply(get_language_family)

# Max coverage across all prompt variants: a language counts as covered by a
# service if any variant produced a valid translation for it.
avail_cols = {svc: col for svc, col in HEATMAP_SERVICES.items() if col in all_variants.columns}
coverage_by_lang = (
    all_variants
    .assign(**{col: all_variants[col].notna() for col in avail_cols.values()})
    .groupby("language_code")
    .agg({col: "max" for col in avail_cols.values()} | {"language_family": "first"})
    .reset_index()
)

other_langs = coverage_by_lang[coverage_by_lang["language_family"] == "Other"]["language_code"].tolist()
if other_langs:
    print(f"⚠ Ungrouped languages: {other_langs}")
else:
    print("✓ All languages assigned to a family")

family_counts = coverage_by_lang["language_family"].value_counts().to_dict()

def format_family_axis_label(family):
    """Shorten genealogical family names for display without changing reviewed metadata."""
    special_labels = {
        "Artificial languages": "Artificial languages",
        "Sign languages": "Sign languages",
        "Language isolate": "Language isolates",
        "Exclude from family aggregation": "Excluded from family aggregation",
    }
    if family in special_labels:
        return special_labels[family]
    return family.removesuffix(" languages")


rows = []
for family, n_fam in sorted(family_counts.items(), key=lambda x: -x[1]):
    fam_df = coverage_by_lang[coverage_by_lang["language_family"] == family]
    label = f"{format_family_axis_label(family)} ({n_fam})"
    for service, col in HEATMAP_SERVICES.items():
        n_translated = int(fam_df[col].sum()) if col in fam_df.columns else 0
        pct = round(n_translated / n_fam * 100, 1) if n_fam else 0.0
        rows.append({"Family": label, "Service": service, "Coverage": pct, "N": n_translated, "Family_size": n_fam})
coverage_long = pd.DataFrame(rows)

family_order = [
    f"{format_family_axis_label(f)} ({family_counts[f]})"
    for f in sorted(family_counts.keys(), key=lambda k: -family_counts[k])
]
chart = alt.Chart(coverage_long).mark_rect().encode(
    x=alt.X("Service:N", sort=service_order, title="Translation Service"),
    y=alt.Y("Family:N", sort=family_order, title="Language Family"),
    color=alt.Color("Coverage:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, 100]),
        legend=alt.Legend(title="% coverage"),
    ),
    tooltip=[
        "Family", "Service",
        alt.Tooltip("Coverage:Q", format=".1f", title="% coverage"),
        alt.Tooltip("N:Q", title="languages translated"),
        alt.Tooltip("Family_size:Q", title="family size"),
    ],
).properties(
    title=alt.Title(
        "Languages Translated by Service and Language Family",
        subtitle="LLM services: any prompt variant = covered. Wikipedia community translation and MT baselines run once.",
    ),
    width=520, height=720,
)
text = chart.mark_text(baseline="middle", fontSize=8).encode(
    text=alt.Text("Coverage:Q", format=".0f"),
    color=alt.condition(alt.datum.Coverage > 60, alt.value("white"), alt.value("black")),
)
languages_by_service_family_chart = chart + text

article_figures_dir = FIGURE_DIR
language_family_coverage_png = article_figures_dir / "languages_translated_by_service_and_language_family.png"
save_chart(languages_by_service_family_chart, str(language_family_coverage_png), scale_factor=2)
print("Saved article chart.")

languages_by_service_family_chart

✓ All languages assigned to a family
Saved article chart.


alt.LayerChart(...)

### Coverage Depth by Language

Coverage depth counts how many services produced at least one translation for each language. I use this as a quick way to distinguish languages with rich multi-service evidence from languages where the pipeline is relying on one or two outputs. Low coverage does not automatically mean a translation is wrong, but it does mean later claims have less corroborating infrastructure behind them.


In [7]:
term = TARGET_TERMS[0]
df_all = all_dfs[term].copy()
df_all["language_family"] = df_all["language_code"].apply(get_language_family)

all_svc_cols = {**PROMPT_INVARIANT_SERVICES, **LLM_SERVICES}
svc_cols = [c for c in all_svc_cols.values() if c in df_all.columns]

# A service "covers" a language if it succeeded in at least one prompt variant
coverage = df_all.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any()).reset_index()
lang_meta = df_all[["language_code", "language_name", "language_family"]].drop_duplicates("language_code")
coverage_depth = coverage.merge(lang_meta, on="language_code")
coverage_depth["n_services"] = coverage_depth[svc_cols].sum(axis=1)

TIERS = {"0 — none": (0,0), "1–2 — sparse": (1,2), "3–5 — partial": (3,5), "6–9 — rich": (6,9)}
def tier(n):
    for label, (lo, hi) in TIERS.items():
        if lo <= n <= hi: return label
    return "other"
coverage_depth["tier"] = coverage_depth["n_services"].apply(tier)

tier_order = list(TIERS.keys())
hist = alt.Chart(coverage_depth).mark_bar().encode(
    x=alt.X("n_services:O", title="Services that translated this language (any prompt)", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("count():Q", title="Number of languages"),
    color=alt.Color("tier:N", sort=tier_order,
        scale=alt.Scale(domain=tier_order, range=["#d32f2f","#ff9800","#1976d2","#388e3c"]),
        title="Coverage tier"),
    tooltip=["n_services:O", "count():Q", "tier:N"],
).properties(width=360, height=240, title="Service coverage depth per language (best across all prompts)")

low_cov = coverage_depth[coverage_depth["n_services"] <= 3][
    ["language_code","language_name","language_family","n_services","tier"]
].sort_values(["n_services","language_family"]).reset_index(drop=True)

print(f"Zero coverage: {(coverage_depth['n_services']==0).sum()} | Sparse (1-3): {((coverage_depth['n_services']>=1)&(coverage_depth['n_services']<=3)).sum()}. Only sparse languages: {low_cov['language_name'].tolist()}")

hist

Zero coverage: 0 | Sparse (1-3): 0. Only sparse languages: []


alt.Chart(...)

### Error Log Analysis

The service coverage charts count parsed translations. The error logs tell me what failed before a translation could be parsed or accepted. I group logged failures by service, status, and language so that API errors, parse failures, empty outputs, refusal language, and repetition loops do not get collapsed into a single "missing" category. The service-category matrix displays only categories observed in the current logs. `missing_error_detail` identifies parsing-error records whose logger saved no explanatory message; `other` retains non-empty messages that do not yet match a more specific rule.


In [8]:
SERVICE_NAMES = {
    "claude":    "Claude",   "deepseek": "DeepSeek", "enmt":  "EasyNMT",
    "gemini":    "Gemini",   "gemma":    "Gemma",    "gt":    "Google Translate",
    "lingvanex": "Lingvanex","llama":    "Llama",    "mistral": "Mistral",
    "openai":    "OpenAI",   "qwen":     "Qwen",     "wikipedia": "Wikipedia",
}

STATUS_NAMES = {
    "400": "400 Bad Request",
    "401": "401 Unauthorized",
    "403": "403 Forbidden",
    "404": "404 Not Found",
    "408": "408 Timeout",
    "429": "429 Rate Limited",
    "500": "500 Server Error",
    "502": "502 Bad Gateway",
    "503": "503 Unavailable",
    "504": "504 Gateway Timeout",
    "unknown": "Unknown",
}

error_dir = os.path.join(DATA_DIR, "error_logs")
error_rows = []
target_term = TARGET_TERMS[0]

if os.path.exists(error_dir):
    for fname in sorted(os.listdir(error_dir)):
        if not fname.endswith(".csv"):
            continue
        key = fname.removesuffix("_translation_errors.csv").removesuffix("translation_errors.csv").strip("_-")
        name = SERVICE_NAMES.get(key, key.replace("_", " ").title())
        try:
            edf = read_csv_file(os.path.join(error_dir, fname))
            if "term_source" in edf.columns:
                edf = edf[edf["term_source"].astype(str).str.contains(target_term, regex=False, na=False)].copy()
            if edf.empty:
                continue
            if "language_code" not in edf.columns:
                edf["language_code"] = "__unknown_language__"
            edf["status"] = (
                edf["status_code"].fillna("unknown").astype(str).map(STATUS_NAMES).fillna(
                    edf["status_code"].fillna("unknown").astype(str)
                ) if "status_code" in edf.columns else "Unknown"
            )
            status_summary = (
                edf.groupby("status", as_index=False)
                .agg(
                    count=("language_code", "nunique"),
                    raw_error_rows=("language_code", "size"),
                )
            )
            for _, row in status_summary.iterrows():
                error_rows.append({
                    "service": name,
                    "status": row["status"],
                    "count": int(row["count"]),
                    "raw_error_rows": int(row["raw_error_rows"]),
                })
        except Exception as e:
            print(f"Could not read {fname}: {e}")

error_df = pd.DataFrame(error_rows)

if error_df.empty:
    print(f"No {target_term} error rows found — skipping error chart.")
else:
    totals = (
        error_df.groupby("service", as_index=False)
        .agg(total_languages=("count", "sum"), total_raw_error_rows=("raw_error_rows", "sum"))
        .sort_values("total_languages", ascending=False)
    )
    totals["total_label"] = totals.apply(
        lambda row: str(int(row["total_languages"]))
        if int(row["total_languages"]) == int(row["total_raw_error_rows"])
        else f'{int(row["total_languages"])} ({int(row["total_raw_error_rows"])})',
        axis=1,
    )
    x_domain_max = int(totals["total_languages"].max() * 1.18)
    svc_order = totals["service"].tolist()
    error_df = error_df.merge(totals, on="service", how="left")

    status_order = (
        error_df.groupby("status")["count"]
        .sum()
        .sort_values(ascending=False)
        .index.tolist()
    )

    error_chart = alt.Chart(error_df).mark_bar().encode(
        y=alt.Y(
            "service:N",
            sort=svc_order,
            title=None,
            axis=alt.Axis(labelLimit=180),
        ),
        x=alt.X(
            "count:Q",
            title="unique language codes with logged errors",
            scale=alt.Scale(domain=[0, x_domain_max]),
        ),
        color=alt.Color(
            "status:N",
            title="Status code",
            sort=status_order,
            scale=alt.Scale(scheme="set2"),
            legend=alt.Legend(
                orient="bottom",
                direction="horizontal",
                columns=3,
                titleLimit=160,
                labelLimit=180,
            ),
        ),
        order=alt.Order("status:N", sort="ascending"),
    ).properties(
        width=640,
        height=max(220, 28 * len(svc_order)),
        title=alt.Title(
            "Digital Humanities Error Logs by Service and Status Code",
            subtitle="Labels show unique language codes; parentheses appear only when raw logged rows are higher.",
        ),
    )

    total_text = alt.Chart(totals).mark_text(
        align="left",
        baseline="middle",
        dx=4,
        fontSize=10,
        color="black",
    ).encode(
        y=alt.Y("service:N", sort=svc_order, title=None),
        x=alt.X("total_languages:Q"),
        text=alt.Text("total_label:N"),
    )

    error_logs_by_service_status_chart = (error_chart + total_text).configure_view(stroke=None)

    article_figures_dir = FIGURE_DIR
    error_logs_png = article_figures_dir / "digital_humanities_error_logs_by_service_status.png"
    save_chart(error_logs_by_service_status_chart, str(error_logs_png), scale_factor=2)
    print("Saved article chart.")

    display(error_logs_by_service_status_chart)


Saved article chart.


alt.LayerChart(...)

In [9]:
PARSE_PREFIX = "Could not parse translation response: "
LLM_ERROR_FILES = {
    "Claude":   "claude_translation_errors.csv",
    "OpenAI":   "openai_translation_errors.csv",
    "Gemini":   "gemini_translation_errors.csv",
    "DeepSeek": "deepseek_translation_errors.csv",
    "Llama":    "llama_translation_errors.csv",
    "Gemma":    "gemma_translation_errors.csv",
    "Qwen":     "qwen_translation_errors.csv",
    "Mistral":  "mistral_translation_errors.csv",
}

CATEGORIES = [
    ("max_tokens",        ["MAX_TOKENS", "token limit", "hit token limit", "truncated"]),
    ("api_error",         ["INTERNAL", "An internal error has occurred",
                           "read operation timed out", "timeout", "timed out"]),
    ("empty_translation", ["Model returned empty translated_term"]),
    ("ollama_timeout",    ["'total_duration'"]),
    ("extinct_ancient",   ["extinct", "ancient", "undeciphered", "no native speakers",
                           "historical language", "limited corpus", "Minoan", "funerary",
                           "no longer spoken"]),
    ("knowledge_gap",     ["limited resources", "limited documentation", "limited data",
                           "limited information", "limited available",
                           "not have the ability", "not able to provide",
                           "don't have the capability", "do not have the capacity",
                           "not equipped", "knowledge cutoff", "training data",
                           "not within my capabilities", "cannot provide",
                           "unable to provide", "not sufficiently documented",
                           "not thoroughly documented", "not widely documented",
                           "under-documented", "lesser-known", "lesser-documented",
                           "less commonly", "less widely", "no comprehensive",
                           "no data on", "not have data", "do not have specific",
                           "don't have specific",
                           "does not have a direct equivalent",
                           "does not directly translate",
                           "no available data", "no available translation",
                           "no known translation",
                           "couldn't find", "could not find"]),
    ("generic_refusal",   ["I'm sorry, I can't", "I'm sorry, I cannot",
                           "I'm sorry, but",
                           "I must respectfully decline",
                           "I can't assist", "I can't comply", "I can't do that",
                           "I cannot perform", "No rationale provided",
                           "cannot fulfill", "can't fulfill", "unable to fulfill",
                           "unable to translate", "not_available"]),
]

ERROR_LAYER_MAP = {
    "api_error": "request_failure",
    "max_tokens": "request_failure",
    "ollama_timeout": "request_failure",
    "parse_format": "parse_failure",
    "empty_translation": "empty_output",
    "generic_refusal": "model_refusal",
    "knowledge_gap": "model_uncertainty",
    "extinct_ancient": "model_uncertainty",
    "repetition_loop": "generation_artifact",
    "missing_error_detail": "parse_failure",
    "other": "unclassified",
}

import re as _re

def _is_repetition_loop(raw_msg: str) -> bool:
    """True if the error wraps a translated_term that is a hallucination repetition loop."""
    text = raw_msg.replace(PARSE_PREFIX, "").strip()
    text = _re.sub(r"^```json\s*", "", text).strip()
    m = _re.search(r'"translated_term"\s*:\s*"(.{15,})', text)
    if not m:
        return False
    term = m.group(1).rstrip('"} \n\r\t')
    if len(term) < 20:
        return False
    if len(set(term)) / len(term) < 0.15:
        return True
    for start in range(min(5, len(term))):
        for chunk_len in range(2, min(len(term) // 4 + 1, 12)):
            chunk = term[start : start + chunk_len]
            if chunk and term.count(chunk) >= 5:
                return True
    return False

def classify_error(msg):
    if not isinstance(msg, str) or not msg.strip():
        return "missing_error_detail"
    text = msg.replace(PARSE_PREFIX, "").strip()
    # Repetition-loop check before parse_format: the JSON may be structurally valid
    # but the translated_term is a hallucination loop.
    if _is_repetition_loop(msg):
        return "repetition_loop"
    if text.startswith("{") or '"translated_term"' in text:
        return "parse_format"
    for cat, keywords in CATEGORIES:
        if any(kw.lower() in text.lower() for kw in keywords):
            return cat
    return "other"

def message_signature(msg):
    """Normalize repeated boilerplate without hiding raw error counts."""
    if not isinstance(msg, str) or not msg.strip():
        return "<empty>"
    text = msg.replace(PARSE_PREFIX, "").strip()
    text = _re.sub(r"```json|```", "", text, flags=_re.I)
    text = _re.sub(r'"translated_term"\s*:\s*"[^"]*"', '"translated_term": "<term>"', text, flags=_re.S)
    text = _re.sub(r'"translation_rationale"\s*:\s*"[^"]*"', '"translation_rationale": "<rationale>"', text, flags=_re.S)
    text = _re.sub(r"Digital Humanities\s*→\s*[^.\n]+", "Digital Humanities → <language>", text)
    text = _re.sub(r"\b[0-9]{3,}\b", "<num>", text)
    text = _re.sub(r"(.)\1{5,}", r"\1<repeat>", text)
    text = _re.sub(r"\s+", " ", text).strip()
    return text[:180]

lang_names = (
    all_dfs[TARGET_TERMS[0]][["language_code", "language_name"]]
    .drop_duplicates("language_code")
    .set_index("language_code")["language_name"]
    .to_dict()
)

llm_frames = []
for service, fname in LLM_ERROR_FILES.items():
    path = os.path.join(error_dir, fname)
    if not os.path.exists(path):
        continue
    edf = read_csv_file(path)
    if edf.empty:
        continue
    edf["service"] = service
    edf["variant"] = edf.get("variant", "").fillna("").astype(str).str.strip()
    non_current = edf[~edf["variant"].isin(VARIANTS)]
    if not non_current.empty:
        print(f"Warning: {service} has {len(non_current)} non-current error rows still present; excluding them from Notebook 02.")
    edf = edf[edf["variant"].isin(VARIANTS)].copy()
    if edf.empty:
        continue
    edf["category"] = edf["error_message"].apply(classify_error)
    edf["failure_layer"] = edf["category"].map(ERROR_LAYER_MAP).fillna("unclassified")
    edf["clean_msg"] = edf["error_message"].str.replace(PARSE_PREFIX, "", regex=False).str.strip()
    edf["message_signature"] = edf["error_message"].apply(message_signature)
    edf["language_name"] = edf["language_code"].map(lang_names).fillna(edf["language_code"])
    llm_frames.append(edf)

llm_err = pd.concat(llm_frames, ignore_index=True) if llm_frames else pd.DataFrame()

if llm_err.empty:
    print("No current-variant LLM errors found.")
else:
    print(f"Current-variant LLM errors: {len(llm_err)} rows across {llm_err['language_code'].nunique()} unique languages")
    print(f"Unique raw error messages: {llm_err['error_message'].nunique()}")
    print(f"Unique normalized message signatures: {llm_err['message_signature'].nunique()}\n")
    print("Most repeated message signatures:")
    display(
        llm_err["message_signature"].value_counts()
        .head(10)
        .reset_index()
        .rename(columns={"message_signature": "signature", "count": "n_error_rows"})
    )


Current-variant LLM errors: 614 rows across 302 unique languages
Unique raw error messages: 531
Unique normalized message signatures: 529

Most repeated message signatures:


,signature,n_error_rows
0,"<num> INTERNAL. {'error': {'code': <num>, 'mes...",24
1,Model returned empty translated_term,23
2,<empty>,16
3,'NoneType' object has no attribute 'strip',7
4,I don't have the capability to translate langu...,5
5,"I'm sorry, but I can't assist with that request.",3
6,I don't have the capability to translate langu...,3
7,I can't fulfill that request.,3
8,I don't have the capability to translate terms...,3
9,"{""translated_term"": ""གཟའ་རིགས་རིག་གནས་རིག་གནས་...",2


In [10]:
if llm_err.empty:
    print("No LLM error categories to plot.")
else:
    configured_cat_order = [c for c, _ in CATEGORIES] + [
        "repetition_loop", "parse_format", "missing_error_detail", "other"
    ]
    # Keep the matrix focused on observed failures. Categories remain available
    # to the classifier and will appear automatically when current rows use them.
    observed_categories = set(llm_err["category"].dropna())
    cat_order = [c for c in configured_cat_order if c in observed_categories]
    layer_order = [
        "request_failure", "parse_failure", "empty_output", "model_refusal",
        "model_uncertainty", "generation_artifact", "unclassified",
    ]

    error_summary = (
        llm_err.groupby(["failure_layer", "category"], as_index=False)
        .agg(
            n_error_rows=("error_message", "size"),
            n_languages=("language_code", "nunique"),
            n_services=("service", "nunique"),
            n_message_signatures=("message_signature", "nunique"),
        )
        .sort_values("n_error_rows", ascending=False)
    )
    print("Failure layer × category summary:")
    display(error_summary)

    layer_counts = (
        llm_err.groupby("failure_layer", as_index=False)
        .agg(n_error_rows=("error_message", "size"), n_languages=("language_code", "nunique"))
    )
    layer_chart = alt.Chart(layer_counts).mark_bar().encode(
        y=alt.Y("failure_layer:N", sort=layer_order, title=None),
        x=alt.X("n_error_rows:Q", title="logged error rows"),
        color=alt.Color("failure_layer:N", sort=layer_order, legend=None, scale=alt.Scale(scheme="tableau10")),
        tooltip=["failure_layer:N", "n_error_rows:Q", "n_languages:Q"],
    ).properties(width=360, height=180, title="LLM errors by failure layer")
    layer_text = layer_chart.mark_text(align="left", dx=4, fontSize=10).encode(
        text="n_error_rows:Q", color=alt.value("black")
    )

    service_order = [s for s in LLM_SERVICES if s in llm_err["service"].unique()]
    cat_grid = pd.MultiIndex.from_product(
        [service_order, cat_order], names=["service", "category"]
    ).to_frame(index=False)
    cat_counts = (
        llm_err.groupby(["service", "category"])
        .size()
        .reset_index(name="n")
    )
    cat_counts = (
        cat_grid.merge(cat_counts, on=["service", "category"], how="left")
        .fillna({"n": 0})
    )
    cat_counts["n"] = cat_counts["n"].astype(int)

    service_category_heat = alt.Chart(cat_counts).mark_rect().encode(
        x=alt.X("category:N", sort=cat_order, title="Error category", axis=alt.Axis(labelAngle=-35)),
        y=alt.Y("service:N", sort=service_order, title="Service"),
        color=alt.Color(
            "n:Q",
            scale=alt.Scale(scheme="oranges"),
            legend=alt.Legend(title="Logged error rows"),
        ),
        tooltip=["service:N", "category:N", alt.Tooltip("n:Q", title="logged error rows")],
    ).properties(width=560, height=220, title="LLM error category matrix by service")

    service_category_text = alt.Chart(cat_counts[cat_counts["n"] > 0]).mark_text(fontSize=10).encode(
        x=alt.X("category:N", sort=cat_order),
        y=alt.Y("service:N", sort=service_order),
        text="n:Q",
        color=alt.condition(alt.datum.n > cat_counts["n"].max() * 0.45, alt.value("white"), alt.value("black")),
    )

    llm_error_category_matrix_chart = service_category_heat + service_category_text

    article_figures_dir = FIGURE_DIR
    llm_error_matrix_png = article_figures_dir / "llm_error_category_matrix_by_service.png"
    save_chart(llm_error_category_matrix_chart, str(llm_error_matrix_png), scale_factor=2)
    print("Saved article chart.")

    display((layer_chart + layer_text) | llm_error_category_matrix_chart)

    # Signature audit: category-level view of repetition vs diversity.
    sig_audit = (
        llm_err.groupby(["failure_layer", "category", "message_signature"], as_index=False)
        .agg(
            n_error_rows=("error_message", "size"),
            n_languages=("language_code", "nunique"),
            services=("service", lambda x: "; ".join(sorted(x.unique()))),
        )
        .sort_values(["category", "n_error_rows"], ascending=[True, False])
    )
    focus_categories = ["other", "parse_format", "generic_refusal", "knowledge_gap", "repetition_loop"]
    print("Top normalized message signatures for audit categories:")
    display(
        sig_audit[sig_audit["category"].isin(focus_categories)]
        .groupby("category", group_keys=False)
        .head(5)
        .reset_index(drop=True)
    )

    # Drill-down: languages in model uncertainty and generation-artifact categories.
    for cat in ("knowledge_gap", "extinct_ancient", "repetition_loop"):
        sub = llm_err[llm_err["category"] == cat][
            ["service", "language_code", "language_name", "message_signature", "clean_msg"]
        ].copy()
        if sub.empty:
            continue
        print(f"\n-- {cat} ({len(sub)} errors, {sub['language_code'].nunique()} languages) --")

        multi_svc = (
            sub.groupby(["language_code", "language_name"])
            .agg(
                n_services=("service", "nunique"),
                services=("service", lambda x: sorted(x.unique())),
            )
            .reset_index()
            .query("n_services >= 2")
            .sort_values("n_services", ascending=False)
        )
        if not multi_svc.empty:
            print(f"  Flagged by >=2 services ({len(multi_svc)} languages):")
            for _, r in multi_svc.iterrows():
                print(f"    {r['language_code']} ({r['language_name']}): {r['services']}")

        print("\n  Sample signatures:")
        for svc, grp in sub.groupby("service"):
            print(f"  [{svc}]")
            for _, row in grp.drop_duplicates("message_signature").head(2).iterrows():
                print(f"    {row['language_code']} ({row['language_name']}): {row['message_signature'][:140]}")


Failure layer × category summary:


,failure_layer,category,n_error_rows,n_languages,n_services,n_message_signatures
4,model_uncertainty,knowledge_gap,179,129,4,170
2,model_refusal,generic_refusal,125,108,3,121
9,unclassified,other,104,89,4,98
1,generation_artifact,repetition_loop,61,56,4,60
8,request_failure,max_tokens,39,34,3,34
3,model_uncertainty,extinct_ancient,37,22,5,37
7,request_failure,api_error,25,18,1,2
0,empty_output,empty_translation,23,23,3,1
5,parse_failure,missing_error_detail,16,15,1,1
6,parse_failure,parse_format,5,5,3,5


Saved article chart.


alt.HConcatChart(...)

Top normalized message signatures for audit categories:


,failure_layer,category,message_signature,n_error_rows,n_languages,services
0,model_refusal,generic_refusal,I can't fulfill that request.,3,3,Llama
1,model_refusal,generic_refusal,"I'm sorry, but I can't assist with that request.",3,3,OpenAI
2,model_refusal,generic_refusal,I can't fulfill that request. I can help you w...,1,1,Llama
3,model_refusal,generic_refusal,I can't fulfill that request. I can’t provide ...,1,1,Llama
4,model_refusal,generic_refusal,I can't fulfill that request. I can’t provide ...,1,1,Llama
5,model_uncertainty,knowledge_gap,I don't have the capability to translate langu...,5,5,Llama
6,model_uncertainty,knowledge_gap,I don't have the capability to translate langu...,3,3,Llama
7,model_uncertainty,knowledge_gap,I don't have the capability to translate terms...,3,3,Llama
8,model_uncertainty,knowledge_gap,I don't have the capability to translate langu...,2,2,Llama
9,model_uncertainty,knowledge_gap,Classical Mandaic does not have a direct equiv...,1,1,Llama



-- knowledge_gap (179 errors, 129 languages) --
  Flagged by >=2 services (28 languages):
    mdt (Mbere): ['Gemini', 'Llama', 'OpenAI']
    ain (Ainu): ['Llama', 'OpenAI']
    nxq (Naxi): ['Llama', 'OpenAI']
    yrk (Nenets): ['Llama', 'OpenAI']
    xmr (Meroitic): ['Llama', 'OpenAI']
    xlc (Lycian): ['Gemini', 'OpenAI']
    xcr (Carian): ['Llama', 'OpenAI']
    was (Washo): ['Llama', 'OpenAI']
    unx (Munda): ['Llama', 'OpenAI']
    tht (Tahltan): ['Llama', 'OpenAI']
    tgx (Tagish): ['Llama', 'OpenAI']
    saz (Saurashtra): ['Llama', 'Mistral']
    sad (Sandawe): ['Llama', 'OpenAI']
    oka (Okanagan): ['Llama', 'Mistral']
    mvy (Indus Kohistani): ['Llama', 'OpenAI']
    aro (Araona): ['Gemini', 'OpenAI']
    mde (Maba): ['Llama', 'OpenAI']
    kvx (Parkari Koli): ['Llama', 'Mistral']
    kut (Kutenai): ['Llama', 'OpenAI']
    hnn (Hanunoo): ['Llama', 'OpenAI']
    gld (Nanai): ['Llama', 'OpenAI']
    gjk (Kachi Koli): ['Llama', 'OpenAI']
    fud (East Futuna): ['Mistral', 'O

#### Error Recurrence Across Languages, Models, and Prompts

The service-level totals do not show whether failures are concentrated in the same languages. Here I collapse repeated log entries to one error per language, service, and prompt variant. I then ask three related questions: how many service/prompt cells fail for each language, which services repeatedly fail on the same languages, and whether those patterns cluster by reviewed language family.

I treat a language as having a cross-model, cross-prompt recurrence when it has logged errors from at least two LLM services and at least two prompt variants. This is a prioritization signal rather than a claim that the language itself caused the errors. Family rates use all 32 possible LLM cells per language (eight services × four prompts), which prevents large families from ranking highly simply because they contain more languages. Families with fewer than five languages remain in the table but are omitted from the family chart because their rates are unstable.


In [11]:
if llm_err.empty:
    print("No current-variant LLM errors available for recurrence analysis.")
else:
    # One failure per attempted language × service × prompt cell. Raw logs can
    # contain repeated entries from reruns, which should not inflate recurrence.
    error_cells = (
        llm_err.drop_duplicates(["language_code", "service", "variant"])
        [["language_code", "service", "variant", "category", "failure_layer"]]
        .copy()
    )

    language_meta = (
        all_dfs[TARGET_TERMS[0]][["language_code", "language_name"]]
        .drop_duplicates("language_code")
        .copy()
    )
    language_meta["language_family"] = language_meta["language_code"].apply(get_language_family)

    language_error_stats = (
        error_cells.groupby("language_code")
        .agg(
            n_error_cells=("variant", "size"),
            n_services_with_errors=("service", "nunique"),
            n_variants_with_errors=("variant", "nunique"),
            n_error_categories=("category", "nunique"),
            error_categories=("category", lambda x: "; ".join(sorted(set(x)))),
        )
        .reset_index()
    )
    language_error_stats = language_meta.merge(language_error_stats, on="language_code", how="left")
    count_cols = ["n_error_cells", "n_services_with_errors", "n_variants_with_errors", "n_error_categories"]
    language_error_stats[count_cols] = language_error_stats[count_cols].fillna(0).astype(int)
    language_error_stats["error_categories"] = language_error_stats["error_categories"].fillna("")
    n_possible_llm_cells = len(LLM_SERVICES) * len(VARIANTS)
    language_error_stats["error_cell_rate"] = language_error_stats["n_error_cells"] / n_possible_llm_cells
    language_error_stats["cross_model_prompt_recurrence"] = (
        language_error_stats["n_services_with_errors"].ge(2)
        & language_error_stats["n_variants_with_errors"].ge(2)
    )

    n_any_error = int(language_error_stats["n_error_cells"].gt(0).sum())
    n_recurrent = int(language_error_stats["cross_model_prompt_recurrence"].sum())
    print(f"Languages with at least one logged LLM error: {n_any_error:,} / {len(language_error_stats):,}")
    print(f"Cross-model, cross-prompt recurrence: {n_recurrent:,} languages")

    top_error_languages = language_error_stats.sort_values(
        ["n_error_cells", "n_services_with_errors", "n_variants_with_errors"],
        ascending=False,
    ).head(30).copy()
    display(top_error_languages[[
        "language_code", "language_name", "language_family", "n_error_cells",
        "n_services_with_errors", "n_variants_with_errors", "error_categories",
    ]])

    # Distribution includes languages with no logged errors.
    recurrence_dist = (
        language_error_stats["n_error_cells"].value_counts()
        .rename_axis("n_error_cells")
        .reset_index(name="n_languages")
        .sort_values("n_error_cells")
    )
    recurrence_dist["label"] = recurrence_dist["n_languages"].astype(str)
    recurrence_bars = alt.Chart(recurrence_dist).mark_bar(color="#4c78a8").encode(
        x=alt.X("n_error_cells:O", title="failed LLM service × prompt cells per language", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("n_languages:Q", title="languages"),
        tooltip=["n_error_cells:O", "n_languages:Q"],
    ).properties(
        width=460, height=230,
        title=alt.Title("How concentrated are logged LLM errors?", subtitle=f"Denominator: {len(language_error_stats):,} languages; maximum {n_possible_llm_cells} possible LLM cells each"),
    )
    recurrence_labels = recurrence_bars.mark_text(dy=-6, fontSize=10).encode(
        text="label:N", color=alt.value("black")
    )
    error_recurrence_distribution_chart = recurrence_bars + recurrence_labels

    # For the most recurrent languages, each heatmap cell records how many of
    # the four prompts failed for that service.
    top_codes = top_error_languages["language_code"].tolist()
    top_order = (top_error_languages["language_name"] + " (" + top_error_languages["language_code"] + ")").tolist()
    heat_counts = (
        error_cells[error_cells["language_code"].isin(top_codes)]
        .groupby(["language_code", "service"])["variant"]
        .nunique().reset_index(name="n_failed_variants")
    )
    heat_grid = pd.MultiIndex.from_product(
        [top_codes, [s for s in LLM_SERVICES if s in llm_err["service"].unique()]],
        names=["language_code", "service"],
    ).to_frame(index=False)
    heat_grid = heat_grid.merge(heat_counts, on=["language_code", "service"], how="left")
    heat_grid["n_failed_variants"] = heat_grid["n_failed_variants"].fillna(0).astype(int)
    heat_grid = heat_grid.merge(language_meta, on="language_code", how="left")
    heat_grid["language_label"] = heat_grid["language_name"] + " (" + heat_grid["language_code"] + ")"

    recurrence_heat = alt.Chart(heat_grid).mark_rect().encode(
        x=alt.X("service:N", sort=list(LLM_SERVICES), title="LLM service", axis=alt.Axis(labelAngle=-35)),
        y=alt.Y("language_label:N", sort=top_order, title=None, axis=alt.Axis(labelLimit=220)),
        color=alt.Color("n_failed_variants:Q", scale=alt.Scale(domain=[0, len(VARIANTS)], scheme="orangered"), title="Prompts with errors"),
        tooltip=["language_label:N", "language_family:N", "service:N", alt.Tooltip("n_failed_variants:Q", title="prompt variants with errors")],
    ).properties(
        width=560, height=24 * len(top_order),
        title=alt.Title("Languages with recurrent errors across LLMs and prompts", subtitle="Top 30 by distinct failed service × prompt cells; values are failed prompts out of four"),
    )
    recurrence_heat_text = alt.Chart(heat_grid[heat_grid["n_failed_variants"] > 0]).mark_text(fontSize=9).encode(
        x=alt.X("service:N", sort=list(LLM_SERVICES)),
        y=alt.Y("language_label:N", sort=top_order),
        text="n_failed_variants:Q",
        color=alt.condition(alt.datum.n_failed_variants >= 3, alt.value("white"), alt.value("black")),
    )
    language_error_recurrence_heatmap = recurrence_heat + recurrence_heat_text

    family_error_stats = (
        language_error_stats.groupby("language_family")
        .agg(
            n_languages=("language_code", "nunique"),
            languages_with_errors=("n_error_cells", lambda x: x.gt(0).sum()),
            recurrent_languages=("cross_model_prompt_recurrence", "sum"),
            total_error_cells=("n_error_cells", "sum"),
            mean_error_cells=("n_error_cells", "mean"),
        )
        .reset_index()
    )
    family_error_stats["error_cell_rate"] = family_error_stats["total_error_cells"] / (family_error_stats["n_languages"] * n_possible_llm_cells)
    family_error_stats["languages_with_error_rate"] = family_error_stats["languages_with_errors"] / family_error_stats["n_languages"]
    family_error_stats["recurrence_rate"] = family_error_stats["recurrent_languages"] / family_error_stats["n_languages"]
    family_error_stats = family_error_stats.sort_values("error_cell_rate", ascending=False)
    display(family_error_stats.style.format({
        "error_cell_rate": "{:.1%}", "languages_with_error_rate": "{:.1%}",
        "recurrence_rate": "{:.1%}", "mean_error_cells": "{:.2f}",
    }))

    family_plot_df = family_error_stats[family_error_stats["n_languages"] >= 5].copy()
    family_order = family_plot_df.sort_values("error_cell_rate", ascending=False)["language_family"].tolist()
    family_error_chart = alt.Chart(family_plot_df).mark_bar().encode(
        y=alt.Y("language_family:N", sort=family_order, title=None, axis=alt.Axis(labelLimit=240)),
        x=alt.X("error_cell_rate:Q", axis=alt.Axis(format=".1%"), title="logged errors as share of 32 possible LLM cells per language"),
        color=alt.Color("recurrence_rate:Q", scale=alt.Scale(scheme="redyellowblue", reverse=True, domain=[0, 1]), legend=alt.Legend(title="Cross-model + prompt recurrence", format=".0%")),
        tooltip=[
            "language_family:N", "n_languages:Q", "languages_with_errors:Q", "recurrent_languages:Q",
            alt.Tooltip("error_cell_rate:Q", format=".1%", title="error-cell rate"),
            alt.Tooltip("recurrence_rate:Q", format=".1%", title="recurrent-language share"),
        ],
    ).properties(
        width=520, height=max(300, 21 * len(family_plot_df)),
        title=alt.Title("Logged LLM error recurrence by language family", subtitle="Normalized by family size and 32 possible service × prompt cells; families with n ≥ 5"),
    )

    article_figures_dir = FIGURE_DIR
    recurrence_dist_png = article_figures_dir / "llm_error_recurrence_distribution_by_language.png"
    recurrence_heat_png = article_figures_dir / "llm_error_recurrence_by_language_and_service.png"
    family_error_png = article_figures_dir / "llm_error_recurrence_by_language_family.png"
    save_chart(error_recurrence_distribution_chart, str(recurrence_dist_png), scale_factor=2)
    save_chart(language_error_recurrence_heatmap, str(recurrence_heat_png), scale_factor=2)
    save_chart(family_error_chart, str(family_error_png), scale_factor=2)
    print("Saved article charts.")

    display(error_recurrence_distribution_chart)
    display(language_error_recurrence_heatmap)
    display(family_error_chart)


Languages with at least one logged LLM error: 302 / 881
Cross-model, cross-prompt recurrence: 112 languages


,language_code,language_name,language_family,n_error_cells,n_services_with_errors,n_variants_with_errors,error_categories
319,xcr,Carian,Indo-European languages,10,5,4,extinct_ancient; knowledge_gap; max_tokens; ot...
3,ecy,Eteocypriot,Indo-European languages,6,4,3,api_error; extinct_ancient; knowledge_gap
321,xlc,Lycian,Indo-European languages,6,4,3,empty_translation; extinct_ancient; knowledge_...
39,oka,Okanagan,Salishan,5,4,3,api_error; generic_refusal; knowledge_gap
292,zbl,Blissymbols,Artificial languages,5,4,3,empty_translation; other; repetition_loop
357,sog,Sogdian,Indo-European languages,5,4,3,api_error; extinct_ancient; other; repetition_...
732,lil,Lillooet,Salishan,5,4,2,generic_refusal; knowledge_gap; max_tokens; re...
233,unx,Munda,Austro-Asiatic languages,5,3,3,generic_refusal; knowledge_gap; other
402,tht,Tahltan,Athabaskan-Eyak-Tlingit,5,3,3,knowledge_gap; repetition_loop
422,str,Straits Salish,Salishan,5,3,3,generic_refusal; knowledge_gap; max_tokens; other


,language_family,n_languages,languages_with_errors,recurrent_languages,total_error_cells,mean_error_cells,error_cell_rate,languages_with_error_rate,recurrence_rate
49,Pano-Tacanan,1,1,1,5,5.00,15.6%,100.0%,100.0%
24,Exclude from family aggregation,1,1,1,4,4.00,12.5%,100.0%,100.0%
52,Salishan,6,5,4,19,3.17,9.9%,83.3%,66.7%
15,Cariban,1,1,1,3,3.00,9.4%,100.0%,100.0%
37,Misumalpan,1,1,0,3,3.00,9.4%,100.0%,0.0%
25,Furan,1,1,1,3,3.00,9.4%,100.0%,100.0%
18,Chibchan,1,1,1,3,3.00,9.4%,100.0%,100.0%
14,Caddoan,1,1,1,3,3.00,9.4%,100.0%,100.0%
33,Language isolate,14,13,10,39,2.79,8.7%,92.9%,71.4%
19,Chukotko-Kamchatkan languages,2,2,1,5,2.50,7.8%,100.0%,50.0%


Saved article charts.


alt.LayerChart(...)

alt.LayerChart(...)

alt.Chart(...)

**Current pattern.** Of the 881 languages, 579 have no logged LLM error in the current four-prompt run, 302 have at least one, and 112 meet the cross-model, cross-prompt recurrence rule. Most affected languages have only one or two failed cells; the long tail is much smaller, with Carian (`xcr`) reaching ten distinct failed cells across five services and all four prompts. The heatmap also shows that recurrence is distributed unevenly across services, so a language's position reflects both language-resource conditions and each model's failure behavior.

After normalization, Salishan, language isolates, and Athabaskan-Eyak-Tlingit have the highest logged error-cell rates among families with at least five rows. Many of the repeatedly affected rows are ancient, historically attested, Indigenous, or otherwise low-resource languages. I therefore read this as evidence about uneven model and infrastructure coverage, not as evidence that membership in a language family intrinsically causes errors. The family chart is most useful for identifying clusters that deserve closer review; the language-service heatmap shows which individual rows and models produce each cluster.


#### Error Recurrence Through Khanna and Li's Vitality–Digitality Framework

Family and script do not adequately explain which languages fail repeatedly. Khanna and Li offer a more useful distinction between **language vitality** and **digitality**. A language can have many active speakers while remaining weakly represented in Common Crawl, Wikipedia, language archives, and machine-learning datasets; a language with fewer contemporary speakers can retain a comparatively substantial digital record. Their Strongholds, Digital Echoes, Fading Voices, and Invisible Giants describe these different relationships rather than treating speaker population and computational resources as interchangeable.

I make a deliberately narrower adaptation for this error analysis. A reviewed population layer combines CLDR estimates with Wikidata fallback observations, using Glottolog identifiers to prevent code collisions. These estimates stand in for vitality, while presence in Wikimedia's language-project inventory provides one upstream indicator of digital community infrastructure. Unlike the GitHub measure introduced in Notebook 09, Wikimedia presence does not depend on the translations generated by this pipeline. The categories remain project-specific proxies, not a replication of Khanna and Li's composite measures. Population sources, measurement types, dates, and disagreements remain available in `language_population_evidence.csv`; this analysis uses the conservative resolved value from `language_population_resolved.csv`.

The assignment rule is explicit: **Stronghold** means above-median population plus Wikimedia presence; **Digital Echo** means at-or-below-median population plus Wikimedia presence; **Fading Voice** means at-or-below-median population without Wikimedia presence; and **Invisible Giant** means above-median population without Wikimedia presence. Only population values marked applicable to a current language population enter these quadrants.

In [12]:
import numpy as np

language_metadata_path = Path(DATA_DIR) / 'metadata_files' / 'language_codes_comprehensive_family_reviewed.csv'
population_metadata_path = Path(DATA_DIR) / 'metadata_files' / 'language_population_resolved.csv'
error_visibility_metadata = pd.read_csv(
    language_metadata_path, dtype=str, keep_default_na=False,
)
population_metadata = pd.read_csv(
    population_metadata_path, converters={'language_code': str},
)
population_metadata['speaker_population'] = pd.to_numeric(
    population_metadata['speaker_population_selected'], errors='coerce',
)
error_visibility_metadata = error_visibility_metadata.merge(
    population_metadata[[
        'language_code', 'speaker_population', 'population_source_selected',
        'population_measure_selected', 'population_reference_year_selected',
        'population_resolution', 'population_status', 'population_applicable',
    ]],
    on='language_code', how='left', validate='one_to_one',
)
error_visibility_metadata['has_wikimedia_presence'] = (
    error_visibility_metadata['in_wikimedia'].str.strip().str.lower().isin(['true', '1', 'yes'])
)

error_visibility = error_visibility_metadata.merge(
    language_error_stats[[
        'language_code', 'n_error_cells', 'n_services_with_errors',
        'n_variants_with_errors', 'cross_model_prompt_recurrence',
    ]],
    on='language_code', how='left',
)
error_visibility['has_any_llm_error'] = error_visibility['n_error_cells'].fillna(0).gt(0)
error_visibility['cross_model_prompt_recurrence'] = (
    error_visibility['cross_model_prompt_recurrence'].fillna(False).astype(bool)
)
error_visibility['has_usable_population'] = (
    error_visibility['speaker_population'].notna()
    & error_visibility['speaker_population'].ge(0)
    & error_visibility['population_applicable'].fillna('').eq('yes')
)
population_unavailable = error_visibility[~error_visibility['has_usable_population']].copy()
population_unavailable['population_analysis_category'] = np.where(
    population_unavailable['population_applicable'].eq('yes'),
    'Vitality unmeasured', 'Population not comparable',
)
error_visibility = error_visibility[error_visibility['has_usable_population']].copy()
population_median = error_visibility['speaker_population'].median()
error_visibility['above_median_population'] = (
    error_visibility['speaker_population'].gt(population_median)
)
error_visibility['adapted_category'] = np.select(
    [
        error_visibility['above_median_population'] & error_visibility['has_wikimedia_presence'],
        ~error_visibility['above_median_population'] & error_visibility['has_wikimedia_presence'],
        ~error_visibility['above_median_population'] & ~error_visibility['has_wikimedia_presence'],
        error_visibility['above_median_population'] & ~error_visibility['has_wikimedia_presence'],
    ],
    ['Stronghold', 'Digital Echo', 'Fading Voice', 'Invisible Giant'],
    default='Unclassified',
)
visibility_category_order = ['Stronghold', 'Digital Echo', 'Fading Voice', 'Invisible Giant']
error_visibility_summary = (
    error_visibility.groupby('adapted_category')
    .agg(
        languages=('language_code', 'nunique'),
        languages_with_any_error=('has_any_llm_error', 'sum'),
        languages_with_recurrent_errors=('cross_model_prompt_recurrence', 'sum'),
        total_failed_service_prompt_cells=('n_error_cells', 'sum'),
    )
    .reindex(visibility_category_order)
    .reset_index()
)
error_visibility_summary['any_error_rate'] = (
    error_visibility_summary['languages_with_any_error'] / error_visibility_summary['languages']
)
error_visibility_summary['recurrent_error_rate'] = (
    error_visibility_summary['languages_with_recurrent_errors'] / error_visibility_summary['languages']
)
print(f'Languages with resolved CLDR/Wikidata population estimates: {len(error_visibility):,}')
print(f'Median estimated speaker population: {population_median:,.0f}')
display(error_visibility_summary.style.format({
    'any_error_rate': '{:.1%}', 'recurrent_error_rate': '{:.1%}',
}))

error_visibility_plot = error_visibility_summary.melt(
    id_vars=['adapted_category'],
    value_vars=['any_error_rate', 'recurrent_error_rate'],
    var_name='error_measure', value_name='language_rate',
)
error_visibility_plot['error_measure'] = error_visibility_plot['error_measure'].map({
    'any_error_rate': 'At least one logged LLM error',
    'recurrent_error_rate': 'Recurrent LLM errors',
})
error_measure_order = ['At least one logged LLM error', 'Recurrent LLM errors']
error_visibility_bars = alt.Chart(error_visibility_plot).mark_bar().encode(
    x=alt.X('adapted_category:N', sort=visibility_category_order, title=None),
    xOffset=alt.XOffset('error_measure:N', sort=error_measure_order),
    y=alt.Y('language_rate:Q', title='Share of languages', axis=alt.Axis(format='%')),
    color=alt.Color(
        'error_measure:N', sort=error_measure_order,
        legend=alt.Legend(title=None, orient='bottom', direction='horizontal'),
        scale=alt.Scale(domain=error_measure_order, range=['#4C78A8', '#E45756']),
    ),
    tooltip=[
        'adapted_category:N', 'error_measure:N',
        alt.Tooltip('language_rate:Q', format='.1%', title='share of languages'),
    ],
).properties(
    width=650, height=320,
    title=alt.TitleParams(
        'LLM Translation Failures by Population and Wikimedia Presence',
        subtitle=f'Project-specific adaptation of Khanna and Li; {len(error_visibility):,} languages with population estimates.',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)
error_visibility_labels = error_visibility_bars.mark_text(dy=-7, fontSize=10).encode(
    text=alt.Text('language_rate:Q', format='.1%'), color=alt.value('black'),
)
error_visibility_main_chart = error_visibility_bars + error_visibility_labels
residual_note_stats = (
    population_unavailable.groupby('population_analysis_category')
    .agg(
        languages=('language_code', 'nunique'),
        languages_with_any_error=('has_any_llm_error', 'sum'),
        languages_with_recurrent_errors=('cross_model_prompt_recurrence', 'sum'),
    )
)
vitality_unmeasured_note = residual_note_stats.loc['Vitality unmeasured']
population_not_comparable_note = residual_note_stats.loc['Population not comparable']
population_unavailable_with_wikimedia = int(population_unavailable['has_wikimedia_presence'].sum())
population_unavailable_without_wikimedia = int((~population_unavailable['has_wikimedia_presence']).sum())
population_unavailable_any_error = int(population_unavailable['has_any_llm_error'].sum())
population_unavailable_recurrent_error = int(population_unavailable['cross_model_prompt_recurrence'].sum())
population_unavailable_status_counts = population_unavailable['population_status'].value_counts()
residual_footnote = (
    f"Note: {len(population_unavailable)} languages could not be placed on the current-speaker population axis and therefore fall outside the four quadrants.\n"
    f"Of these, {population_unavailable_with_wikimedia} had Wikimedia presence and {population_unavailable_without_wikimedia} did not; "
    f"{population_unavailable_any_error} ({population_unavailable_any_error / len(population_unavailable):.1%}) had at least one logged error and "
    f"{population_unavailable_recurrent_error} ({population_unavailable_recurrent_error / len(population_unavailable):.1%}) had recurrent errors.\n"
    f"{int(vitality_unmeasured_note['languages'])} contemporary languages lacked a usable estimate; the remaining {int(population_not_comparable_note['languages'])} comprised "
    f"{int(population_unavailable_status_counts.get('historical_or_non_modern', 0))} historical or non-modern languages, "
    f"{int(population_unavailable_status_counts.get('collective_or_residual', 0))} collective language-group code, and "
    f"{int(population_unavailable_status_counts.get('bookkeeping', 0))} Glottolog bookkeeping records."
)
error_visibility_footnote = alt.Chart(
    pd.DataFrame({'note': [residual_footnote]})
).mark_text(
    align='left', baseline='top', fontSize=9.2, lineBreak='\n', lineHeight=14, color='#555',
).encode(
    x=alt.value(0), y=alt.value(2), text='note:N',
).properties(width=650, height=54)
error_visibility_chart = alt.vconcat(
    error_visibility_main_chart, error_visibility_footnote, spacing=4,
).configure_view(stroke=None)
display(error_visibility_chart)
save_chart(
    error_visibility_chart,
    str(article_figures_dir / 'wikimedia_population_llm_error_rates.png'),
    scale_factor=2,
)

# Missing population data is not random and should not disappear from the analysis.
population_unavailable['infrastructure_group'] = (
    population_unavailable['population_analysis_category'] + ' / ' +
    np.where(
        population_unavailable['has_wikimedia_presence'],
        'Wikimedia present', 'No Wikimedia presence',
    )
)
population_unavailable_order = [
    'Vitality unmeasured / Wikimedia present',
    'Vitality unmeasured / No Wikimedia presence',
    'Population not comparable / Wikimedia present',
    'Population not comparable / No Wikimedia presence',
]
population_unavailable_summary = (
    population_unavailable.groupby('infrastructure_group')
    .agg(
        languages=('language_code', 'nunique'),
        languages_with_any_error=('has_any_llm_error', 'sum'),
        languages_with_recurrent_errors=('cross_model_prompt_recurrence', 'sum'),
        total_failed_service_prompt_cells=('n_error_cells', 'sum'),
    )
    .reindex(population_unavailable_order)
    .reset_index()
)
population_unavailable_summary['any_error_rate'] = (
    population_unavailable_summary['languages_with_any_error']
    / population_unavailable_summary['languages']
)
population_unavailable_summary['recurrent_error_rate'] = (
    population_unavailable_summary['languages_with_recurrent_errors']
    / population_unavailable_summary['languages']
)
print(f'Languages without a usable population estimate: {len(population_unavailable):,}')
display(population_unavailable_summary.style.format({
    'any_error_rate': '{:.1%}', 'recurrent_error_rate': '{:.1%}',
}))

population_unavailable_plot = population_unavailable_summary.melt(
    id_vars=['infrastructure_group'],
    value_vars=['any_error_rate', 'recurrent_error_rate'],
    var_name='error_measure', value_name='language_rate',
)
population_unavailable_plot['error_measure'] = population_unavailable_plot['error_measure'].map({
    'any_error_rate': 'At least one logged LLM error',
    'recurrent_error_rate': 'Recurrent LLM errors',
})
population_unavailable_bars = alt.Chart(population_unavailable_plot).mark_bar().encode(
    y=alt.Y(
        'infrastructure_group:N', sort=population_unavailable_order, title=None,
        axis=alt.Axis(labelLimit=430),
    ),
    yOffset=alt.YOffset('error_measure:N', sort=error_measure_order),
    x=alt.X('language_rate:Q', title='Share of languages', axis=alt.Axis(format='%')),
    color=alt.Color(
        'error_measure:N', sort=error_measure_order,
        legend=alt.Legend(title=None, orient='bottom', direction='horizontal'),
        scale=alt.Scale(domain=error_measure_order, range=['#4C78A8', '#E45756']),
    ),
    tooltip=[
        'infrastructure_group:N', 'error_measure:N',
        alt.Tooltip('language_rate:Q', format='.1%', title='share of languages'),
    ],
).properties(
    width=560, height=240,
    title=alt.TitleParams(
        'LLM Translation Failures Outside the Four Population Quadrants',
        subtitle=f'The {len(population_unavailable):,} residual rows retain explicit evidence categories and Wikimedia status.',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)
population_unavailable_labels = population_unavailable_bars.mark_text(
    align='left', dx=5, fontSize=10,
).encode(
    text=alt.Text('language_rate:Q', format='.1%'), color=alt.value('black'),
)
population_unavailable_chart = population_unavailable_bars + population_unavailable_labels
display(population_unavailable_chart)
save_chart(
    population_unavailable_chart,
    str(article_figures_dir / 'population_unavailable_llm_error_rates.png'),
    scale_factor=2,
)

Languages with resolved CLDR/Wikidata population estimates: 787
Median estimated speaker population: 575,000


,adapted_category,languages,languages_with_any_error,languages_with_recurrent_errors,total_failed_service_prompt_cells,any_error_rate,recurrent_error_rate
0,Stronghold,178,11,3,14,6.2%,1.7%
1,Digital Echo,84,6,2,10,7.1%,2.4%
2,Fading Voice,310,156,60,326,50.3%,19.4%
3,Invisible Giant,215,64,9,90,29.8%,4.2%


alt.VConcatChart(...)

Languages without a usable population estimate: 94


,infrastructure_group,languages,languages_with_any_error,languages_with_recurrent_errors,total_failed_service_prompt_cells,any_error_rate,recurrent_error_rate
0,Vitality unmeasured / Wikimedia present,6,0,0,0,0.0%,0.0%
1,Vitality unmeasured / No Wikimedia presence,60,42,23,95,70.0%,38.3%
2,Population not comparable / Wikimedia present,2,1,0,1,50.0%,0.0%
3,Population not comparable / No Wikimedia presence,26,22,15,76,84.6%,57.7%


alt.LayerChart(...)

In [13]:
error_visibility_chart

alt.VConcatChart(...)

**Current pattern.** The chart recalculates the four categories from the resolved CLDR/Wikidata population layer. Wikidata recovers estimates for many rows absent from CLDR, while the unresolved languages remain in a separate panel rather than disappearing from the denominator. Exact category counts and rates are generated above so that the interpretation follows the current evidence artifact rather than a copied result.

The rows outside the four quadrants are analytically important rather than disposable missing data. I classify contemporary rows lacking a worldwide estimate as **Vitality unmeasured** and historical, collective, or bookkeeping rows as **Population not comparable**. The second panel retains Wikimedia presence within both groups. Every project row therefore receives an evidence category, even when its position above or below the population median cannot be assigned responsibly.

This does not show that Wikimedia presence causes better model performance. Wikimedia is one indicator of the wider ecosystem through which a language acquires standardized identifiers, corpora, software support, and public digital text. The result nevertheless fits Khanna and Li's larger argument: computational scarcity is not an inherent feature of a language, demographic vitality alone does not determine whether digital systems can represent it reliably, and missing infrastructure data can itself be patterned evidence.

#### Directionality, Script, and Identifier-Recognition Errors

The family pattern raises two narrower possibilities. First, models might fail more often on right-to-left scripts or on writing systems that appear less frequently in their training data. Second, a model might fail before translation because it does not recognize a valid language identifier. I test directionality and primary script using the reviewed language metadata, again normalizing by 32 possible LLM service/prompt cells per language.

For identifier recognition, I use a deliberately narrow text pattern. It captures messages claiming that a code is invalid, fictional, absent from ISO, or not associated with a known language; it does not count every refusal that merely mentions an ISO code. I then retain only codes confirmed in the cached SIL ISO 639-3 table. This isolates false code rejection from the much larger category of resource-based uncertainty.


In [14]:
if llm_err.empty:
    print("No current-variant LLM errors available for script and identifier analysis.")
else:
    metadata_dir = Path(DATA_DIR) / "metadata_files"
    reviewed_metadata_path = metadata_dir / "language_codes_comprehensive_family_reviewed.csv"
    generated_metadata_path = metadata_dir / "language_codes_comprehensive.csv"
    language_metadata_path = reviewed_metadata_path if reviewed_metadata_path.exists() else generated_metadata_path
    script_meta = pd.read_csv(
        language_metadata_path, converters={"language_code": str}, keep_default_na=False
    )[["language_code", "language_name", "directionality", "primary_script"]].copy()

    script_error_stats = script_meta.merge(
        language_error_stats[[
            "language_code", "n_error_cells", "n_services_with_errors",
            "n_variants_with_errors", "cross_model_prompt_recurrence",
        ]],
        on="language_code", how="left",
    )
    script_error_stats[["n_error_cells", "n_services_with_errors", "n_variants_with_errors"]] = (
        script_error_stats[["n_error_cells", "n_services_with_errors", "n_variants_with_errors"]]
        .fillna(0).astype(int)
    )
    script_error_stats["cross_model_prompt_recurrence"] = script_error_stats["cross_model_prompt_recurrence"].fillna(False).astype(bool)

    direction_summary = (
        script_error_stats.groupby("directionality")
        .agg(
            n_languages=("language_code", "nunique"),
            languages_with_errors=("n_error_cells", lambda x: x.gt(0).sum()),
            total_error_cells=("n_error_cells", "sum"),
            recurrent_languages=("cross_model_prompt_recurrence", "sum"),
        )
        .reset_index()
    )
    direction_summary["error_cell_rate"] = direction_summary["total_error_cells"] / (direction_summary["n_languages"] * n_possible_llm_cells)
    direction_summary["language_error_rate"] = direction_summary["languages_with_errors"] / direction_summary["n_languages"]
    direction_summary["recurrence_rate"] = direction_summary["recurrent_languages"] / direction_summary["n_languages"]
    direction_summary["direction"] = direction_summary["directionality"].str.upper()
    display(direction_summary.style.format({
        "error_cell_rate": "{:.2%}", "language_error_rate": "{:.1%}", "recurrence_rate": "{:.1%}",
    }))

    direction_long = direction_summary.melt(
        id_vars=["direction", "n_languages"],
        value_vars=["error_cell_rate", "recurrence_rate"],
        var_name="metric", value_name="rate",
    )
    direction_long["metric"] = direction_long["metric"].map({
        "error_cell_rate": "Failed service × prompt cells",
        "recurrence_rate": "Languages recurrent across models + prompts",
    })
    directionality_error_chart = alt.Chart(direction_long).mark_bar().encode(
        x=alt.X("direction:N", sort=["LTR", "RTL"], title="Script directionality"),
        xOffset=alt.XOffset("metric:N"),
        y=alt.Y("rate:Q", axis=alt.Axis(format=".1%"), title="normalized rate"),
        color=alt.Color(
            "metric:N", title=None, scale=alt.Scale(range=["#4c78a8", "#e45756"]),
            legend=alt.Legend(orient="bottom", direction="horizontal", labelLimit=320),
        ),
        tooltip=["direction:N", "metric:N", "n_languages:Q", alt.Tooltip("rate:Q", format=".2%")],
    ).properties(
        width=360, height=230,
        title=alt.Title("Logged LLM errors by script directionality", subtitle="Cell failures and cross-model/prompt recurrence use different denominators"),
    )

    script_summary = (
        script_error_stats.groupby("primary_script")
        .agg(
            n_languages=("language_code", "nunique"),
            languages_with_errors=("n_error_cells", lambda x: x.gt(0).sum()),
            total_error_cells=("n_error_cells", "sum"),
            recurrent_languages=("cross_model_prompt_recurrence", "sum"),
        )
        .reset_index()
    )
    script_summary["error_cell_rate"] = script_summary["total_error_cells"] / (script_summary["n_languages"] * n_possible_llm_cells)
    script_summary["recurrence_rate"] = script_summary["recurrent_languages"] / script_summary["n_languages"]
    script_names = {
        "": "No primary script recorded", "Arab": "Arabic", "Beng": "Bengali",
        "Cans": "Canadian Aboriginal Syllabics", "Cyrl": "Cyrillic",
        "Deva": "Devanagari", "Ethi": "Ethiopic", "Grek": "Greek",
        "Hans": "Han (Simplified)", "Hebr": "Hebrew", "Latn": "Latin", "Thai": "Thai",
    }
    script_summary["script_label"] = script_summary["primary_script"].map(script_names).fillna(script_summary["primary_script"])
    script_plot_df = script_summary[script_summary["n_languages"] >= 5].sort_values("error_cell_rate", ascending=False).copy()
    script_order = script_plot_df["script_label"].tolist()
    primary_script_error_chart = alt.Chart(script_plot_df).mark_bar().encode(
        y=alt.Y("script_label:N", sort=script_order, title=None, axis=alt.Axis(labelLimit=220)),
        x=alt.X("error_cell_rate:Q", axis=alt.Axis(format=".1%"), title="logged errors as share of possible LLM cells"),
        color=alt.Color("recurrence_rate:Q", scale=alt.Scale(scheme="redyellowblue", reverse=True, domain=[0, 1]), legend=alt.Legend(title="Recurrent-language share", format=".0%")),
        tooltip=[
            "script_label:N", "n_languages:Q", "languages_with_errors:Q", "recurrent_languages:Q",
            alt.Tooltip("error_cell_rate:Q", format=".2%"), alt.Tooltip("recurrence_rate:Q", format=".1%"),
        ],
    ).properties(
        width=470, height=max(260, 24 * len(script_plot_df)),
        title=alt.Title("Logged LLM error recurrence by primary script", subtitle="Normalized by script-group size; groups with n ≥ 5 languages"),
    )

    # Identify explicit false claims that valid ISO 639-3 codes are invalid,
    # rather than counting every resource-related refusal that mentions ISO.
    false_code_pattern = (
        r"(?i)(?:not (?:a )?(?:recognized|valid|real) (?:language )?code"
        r"|does not correspond to any known language"
        r"|no ISO standard known as ISO"
        r"|fictional or (?:non-existent|made-up)"
        r"|doesn.t appear to be a recognized standard"
        r"|does not have (?:an?|a standardized|a widely accepted) ISO"
        r"|no widely accepted ISO"
        r"|does not include a code"
        r"|there is no ISO 639-3 code"
        r"|not a recognized language code in the ISO)"
    )
    sil_path = metadata_dir / "cldr-cache" / "iso-639-3.tab"
    sil_codes = set()
    if sil_path.exists():
        sil_codes = set(pd.read_csv(sil_path, sep="\t", dtype=str, na_filter=False)["Id"].str.strip())

    false_code_errors = llm_err[
        llm_err["error_message"].astype(str).str.contains(false_code_pattern, regex=True, na=False)
        & llm_err["language_code"].isin(sil_codes)
    ].drop_duplicates(["language_code", "service", "variant"]).copy()
    false_code_errors = false_code_errors.merge(
        script_meta[["language_code", "language_name", "primary_script", "directionality"]],
        on="language_code", how="left", suffixes=("", "_metadata"),
    )
    false_code_errors["language_name"] = false_code_errors["language_name_metadata"].fillna(false_code_errors["language_name"])
    false_code_errors["language_label"] = false_code_errors["language_name"] + " (" + false_code_errors["language_code"] + ")"
    print(f"False ISO 639-3 code rejections: {len(false_code_errors)} service/prompt cells across {false_code_errors['language_code'].nunique()} valid codes")
    display(false_code_errors[[
        "language_code", "language_name", "service", "variant", "primary_script", "directionality", "clean_msg",
    ]].sort_values(["language_code", "service", "variant"]))

    false_code_counts = (
        false_code_errors.groupby(["language_label", "service"])["variant"]
        .nunique().reset_index(name="n_prompt_cells")
    )
    false_code_order = (
        false_code_counts.groupby("language_label")["n_prompt_cells"].sum()
        .sort_values(ascending=False).index.tolist()
    )
    false_iso_code_chart = alt.Chart(false_code_counts).mark_bar().encode(
        y=alt.Y("language_label:N", sort=false_code_order, title=None, axis=alt.Axis(labelLimit=220)),
        x=alt.X("n_prompt_cells:Q", title="prompt cells falsely rejecting the valid code", axis=alt.Axis(tickMinStep=1)),
        color=alt.Color("service:N", sort=list(LLM_SERVICES), title="LLM service", scale=alt.Scale(scheme="tableau10")),
        tooltip=["language_label:N", "service:N", "n_prompt_cells:Q"],
    ).properties(
        width=470, height=max(220, 28 * false_code_errors["language_code"].nunique()),
        title=alt.Title("Valid ISO 639-3 codes falsely rejected by LLMs", subtitle="Narrow text match, verified against the cached SIL ISO 639-3 table"),
    )

    article_figures_dir = FIGURE_DIR
    directionality_png = article_figures_dir / "llm_error_recurrence_by_script_directionality.png"
    primary_script_png = article_figures_dir / "llm_error_recurrence_by_primary_script.png"
    false_iso_png = article_figures_dir / "valid_iso639_3_codes_falsely_rejected_by_llms.png"
    save_chart(directionality_error_chart, str(directionality_png), scale_factor=2)
    save_chart(primary_script_error_chart, str(primary_script_png), scale_factor=2)
    save_chart(false_iso_code_chart, str(false_iso_png), scale_factor=2)
    print("Saved article charts.")

    display(directionality_error_chart)
    display(primary_script_error_chart)
    display(false_iso_code_chart)


,directionality,n_languages,languages_with_errors,total_error_cells,recurrent_languages,error_cell_rate,language_error_rate,recurrence_rate,direction
0,ltr,802,277,555,100,2.16%,34.5%,12.5%,LTR
1,rtl,79,25,57,12,2.25%,31.6%,15.2%,RTL


False ISO 639-3 code rejections: 12 service/prompt cells across 10 valid codes


,language_code,language_name,service,variant,primary_script,directionality,clean_msg
7,bbj,Ghomala,Llama,github_searcher,Latn,ltr,Note: Ghomala (bbj) is not a real language cod...
4,dcc,Deccan,OpenAI,minimal,Arab,rtl,"I'm sorry, but as of my last update, there is ..."
0,ecy,Eteocypriot,OpenAI,minimal,Cprt,rtl,"I'm sorry, but I cannot provide a translation ..."
11,kvx,Parkari Koli,Mistral,minimal,Arab,rtl,"I'm sorry for any inconvenience, but the Parka..."
6,mzb,Tumzabt,Llama,minimal,Arab,rtl,I'm not familiar with the Tumzabt language or ...
8,saz,Sourashtra,Llama,github_searcher,Saur,ltr,I couldn't find any information on the Sourash...
10,xcr,Carian,Mistral,minimal,Cari,ltr,"I'm sorry for any inconvenience, but as of now..."
1,xcr,Carian,OpenAI,minimal,Cari,ltr,"I'm sorry, but I cannot provide a translation ..."
3,xlc,Lycian,OpenAI,minimal,Lyci,ltr,"I'm sorry, but I cannot provide a translation ..."
2,xld,Lydian,OpenAI,minimal,Lydi,rtl,"I'm sorry, but I am unable to provide a transl..."


Saved article charts.


alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

**Current pattern.** Directionality alone explains very little: LTR languages have a 2.16% error-cell rate and RTL languages 2.25%. Cross-model, cross-prompt recurrence affects 12.5% of LTR languages and 15.2% of RTL languages. The difference is small enough that I would not describe RTL directionality as a general failure condition. At the primary-script level, Canadian Aboriginal Syllabics stands out more clearly: 6 of its 10 languages have recurrent errors, producing a 6.25% error-cell rate. This likely reflects the digital-resource position of the languages using that script rather than a mechanical inability to generate right-to-left or non-Latin text. Arabic-script languages, for example, have a lower error-cell rate than the Latin-script group in this run.

The identifier audit finds twelve service/prompt cells in which a model falsely rejects ten valid ISO 639-3 codes. The affected codes are `bbj`, `dcc`, `ecy`, `kvx`, `mzb`, `saz`, `xcr`, `xlc`, `xld`, and `xmn`. These are not all ancient languages or members of one family: the set mixes ancient corpus languages with living but digitally underrepresented languages. The common factor is that the identifier falls outside the shorter, more familiar code inventory that models appear to recognize reliably. Carian is rejected twice, but it is part of a broader identifier-recognition problem rather than an isolated anomaly.


#### Service × Prompt × Outcome Matrix

This matrix is the main accounting view for prompt-based LLM output. For each service and prompt variant, I compare the number of parsed translations with logged failures and missing cells. This matters because "the service translated all languages" can mean very different things depending on whether the success came from every prompt or from at least one prompt rescuing failures in another.


In [15]:
# Service × prompt × outcome matrix.
# This quantifies the main multiple-comparison question in Notebook 02:
# which service/prompt combinations have missing output or logged failures?
term = TARGET_TERMS[0]
df = all_dfs[term].copy()

service_order = list(LLM_SERVICES.keys())
variant_order = VARIANTS

# Error logs have been pruned to the current prompt variants, so this matrix
# can use llm_err directly.
current_llm_err = llm_err.copy()

outcome_rows = []
for variant in variant_order:
    vdf = df[df["prompt_variant"] == variant]
    possible = int(vdf["language_code"].nunique())
    for service, col in LLM_SERVICES.items():
        if col in vdf.columns:
            has_translation = vdf[col].notna() & ~vdf[col].astype(str).str.strip().isin(["", "nan"])
            parsed_translations = int(has_translation.sum())
        else:
            parsed_translations = 0
        svc_var_err = current_llm_err[
            (current_llm_err["service"] == service) &
            (current_llm_err["variant"] == variant)
        ]
        error_languages = int(svc_var_err["language_code"].nunique()) if not svc_var_err.empty else 0
        logged_errors = int(len(svc_var_err))
        missing_cells = max(possible - parsed_translations, 0)
        outcome_rows.append({
            "service": service,
            "variant": variant,
            "possible_cells": possible,
            "parsed_translations": parsed_translations,
            "missing_translation_cells": missing_cells,
            "logged_error_rows": logged_errors,
            "logged_error_languages": error_languages,
            "parsed_translation_rate": parsed_translations / possible if possible else 0,
            "missing_translation_rate": missing_cells / possible if possible else 0,
            "logged_error_language_rate": error_languages / possible if possible else 0,
        })

service_variant_outcomes = pd.DataFrame(outcome_rows)

# Error-category table with both raw rows and unique-language counts.
err_grid = pd.MultiIndex.from_product(
    [service_order, variant_order, cat_order],
    names=["service", "variant", "category"],
).to_frame(index=False)

if current_llm_err.empty:
    service_variant_errors = err_grid.assign(n_error_rows=0, n_languages=0)
else:
    err_row_counts = (
        current_llm_err.groupby(["service", "variant", "category"])
        .size()
        .reset_index(name="n_error_rows")
    )
    err_lang_counts = (
        current_llm_err.groupby(["service", "variant", "category"])["language_code"]
        .nunique()
        .reset_index(name="n_languages")
    )
    service_variant_errors = (
        err_grid.merge(err_row_counts, on=["service", "variant", "category"], how="left")
        .merge(err_lang_counts, on=["service", "variant", "category"], how="left")
        .fillna({"n_error_rows": 0, "n_languages": 0})
    )
    service_variant_errors[["n_error_rows", "n_languages"]] = service_variant_errors[["n_error_rows", "n_languages"]].astype(int)

service_variant_errors = service_variant_errors.merge(
    service_variant_outcomes[["service", "variant", "possible_cells"]],
    on=["service", "variant"],
    how="left",
)
service_variant_errors["error_rows_per_attempt"] = (
    service_variant_errors["n_error_rows"] / service_variant_errors["possible_cells"]
).fillna(0)
service_variant_errors["error_languages_per_attempt"] = (
    service_variant_errors["n_languages"] / service_variant_errors["possible_cells"]
).fillna(0)

# Standardized residuals within each error category.
residual_frames = []
for category, sub in service_variant_errors.groupby("category"):
    total_n = sub["n_error_rows"].sum()
    if total_n == 0:
        tmp = sub.copy()
        tmp["expected_error_rows"] = 0.0
        tmp["std_residual"] = 0.0
        residual_frames.append(tmp)
        continue
    service_totals = sub.groupby("service")["n_error_rows"].transform("sum")
    variant_totals = sub.groupby("variant")["n_error_rows"].transform("sum")
    tmp = sub.copy()
    tmp["expected_error_rows"] = service_totals * variant_totals / total_n
    tmp["std_residual"] = tmp.apply(
        lambda r: (r["n_error_rows"] - r["expected_error_rows"]) / (r["expected_error_rows"] ** 0.5)
        if r["expected_error_rows"] > 0 else 0,
        axis=1,
    )
    residual_frames.append(tmp)

service_variant_error_residuals = pd.concat(residual_frames, ignore_index=True)

# Save reusable summaries for later notebooks.
eval_dir = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "evaluation")
os.makedirs(eval_dir, exist_ok=True)
outcome_path = os.path.join(eval_dir, "service_variant_outcomes.csv")
error_path = os.path.join(eval_dir, "service_variant_error_summary.csv")
resid_path = os.path.join(eval_dir, "service_variant_error_residuals.csv")
service_variant_outcomes.to_csv(outcome_path, index=False)
service_variant_errors.to_csv(error_path, index=False)
service_variant_error_residuals.to_csv(resid_path, index=False)

print("Saved evaluation table.")
print("Saved evaluation table.")
print("Saved evaluation table.")
print()
print("Service × prompt outcome summary:")
display(
    service_variant_outcomes
    .sort_values(["missing_translation_rate", "logged_error_language_rate"], ascending=False)
    .style.format({
        "parsed_translation_rate": "{:.1%}",
        "missing_translation_rate": "{:.1%}",
        "logged_error_language_rate": "{:.1%}",
    })
)

# Heatmap 1: missing output by service and prompt variant.
missing_heat = alt.Chart(service_variant_outcomes).mark_rect().encode(
    x=alt.X("variant:N", sort=variant_order, title="Prompt variant"),
    y=alt.Y("service:N", sort=service_order, title="Service"),
    color=alt.Color(
        "missing_translation_rate:Q",
        scale=alt.Scale(scheme="oranges"),
        title="Missing translation rate",
        legend=alt.Legend(format="%"),
    ),
    tooltip=[
        "service:N", "variant:N",
        alt.Tooltip("possible_cells:Q", title="expected cells"),
        alt.Tooltip("parsed_translations:Q", title="parsed translations"),
        alt.Tooltip("missing_translation_cells:Q", title="missing cells"),
        alt.Tooltip("logged_error_languages:Q", title="languages with logged errors"),
        alt.Tooltip("missing_translation_rate:Q", format=".1%", title="missing rate"),
    ],
).properties(width=360, height=230, title="Missing translation cells by service × prompt")

missing_text = alt.Chart(service_variant_outcomes).mark_text(fontSize=10).encode(
    x=alt.X("variant:N", sort=variant_order),
    y=alt.Y("service:N", sort=service_order),
    text=alt.Text("missing_translation_rate:Q", format=".0%"),
    color=alt.condition(alt.datum.missing_translation_rate > 0.25, alt.value("white"), alt.value("black")),
)

display(missing_heat + missing_text)

# Heatmap 2: standardized residuals for the most common logged error categories.
top_error_categories = (
    service_variant_errors.groupby("category")["n_error_rows"].sum()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0]
    .head(6)
    .index.tolist()
)
print(f"Top error categories for residual plotting: {top_error_categories}")

if top_error_categories:
    resid_plot_df = service_variant_error_residuals[
        service_variant_error_residuals["category"].isin(top_error_categories)
    ].copy()
    resid_chart = alt.Chart(resid_plot_df).mark_rect().encode(
        x=alt.X("variant:N", sort=variant_order, title="Prompt variant"),
        y=alt.Y("service:N", sort=service_order, title="Service"),
        color=alt.Color(
            "std_residual:Q",
            scale=alt.Scale(scheme="redblue", domainMid=0),
            title="Std. residual",
        ),
        tooltip=[
            "category:N", "service:N", "variant:N",
            alt.Tooltip("n_error_rows:Q", title="observed rows"),
            alt.Tooltip("expected_error_rows:Q", format=".2f", title="expected rows"),
            alt.Tooltip("std_residual:Q", format=".2f", title="std. residual"),
        ],
    ).properties(width=180, height=190).facet(
        facet=alt.Facet("category:N", sort=top_error_categories, title="Error category"),
        columns=3,
        title="Over/under-represented service × prompt combinations by error type",
    )
    article_figures_dir = FIGURE_DIR
    residual_png = article_figures_dir / "service_prompt_error_type_residuals.png"
    save_chart(resid_chart, str(residual_png), scale_factor=2)
    print("Saved article chart.")
    display(resid_chart)

    count_plot_df = service_variant_errors[
        service_variant_errors["category"].isin(top_error_categories)
    ].copy()
    max_count = max(int(count_plot_df["n_error_rows"].max()), 1)

    # Layer first, attach data at the top level, then facet.
    service_prompt_error_count_chart = alt.layer(
        alt.Chart().mark_rect().encode(
            x=alt.X("variant:N", sort=variant_order, title="Prompt variant"),
            y=alt.Y("service:N", sort=service_order, title="Service"),
            color=alt.Color(
                "n_error_rows:Q",
                scale=alt.Scale(scheme="oranges"),
                title="Logged error rows",
            ),
        ),
        alt.Chart().mark_text(fontSize=10).encode(
            x=alt.X("variant:N", sort=variant_order),
            y=alt.Y("service:N", sort=service_order),
            text=alt.condition(
                alt.datum.n_error_rows > 0,
                alt.Text("n_error_rows:Q"),
                alt.value(""),
            ),
            color=alt.condition(
                alt.datum.n_error_rows > max_count * 0.45,
                alt.value("white"),
                alt.value("black"),
            ),
        ),
        data=count_plot_df,
    ).properties(width=180, height=190).facet(
        facet=alt.Facet("category:N", sort=top_error_categories, title="Error category"),
        columns=3,
        title="Logged error counts by service × prompt combination and error type",
    )

    counts_png = article_figures_dir / "service_prompt_error_type_counts.png"
    save_chart(service_prompt_error_count_chart, str(counts_png), scale_factor=2)
    print("Saved article chart.")
    display(service_prompt_error_count_chart)
else:
    print("No current-variant LLM errors available for residual/count plotting.")


Saved evaluation table.
Saved evaluation table.
Saved evaluation table.

Service × prompt outcome summary:


,service,variant,possible_cells,parsed_translations,missing_translation_cells,logged_error_rows,logged_error_languages,parsed_translation_rate,missing_translation_rate,logged_error_language_rate
8,OpenAI,fluent_speaker,881,631,250,131,131,71.6%,28.4%,14.9%
20,Llama,github_searcher,881,786,95,87,87,89.2%,10.8%,9.9%
4,Llama,minimal,881,787,94,94,94,89.3%,10.7%,10.7%
0,OpenAI,minimal,881,797,84,90,90,90.5%,9.5%,10.2%
7,Mistral,minimal,881,829,52,52,52,94.1%,5.9%,5.9%
2,Gemini,minimal,881,846,35,40,40,96.0%,4.0%,4.5%
18,Gemini,github_searcher,881,848,33,23,23,96.3%,3.7%,2.6%
5,Gemma,minimal,881,865,16,16,16,98.2%,1.8%,1.8%
10,Gemini,fluent_speaker,881,865,16,10,10,98.2%,1.8%,1.1%
3,DeepSeek,minimal,881,868,13,17,17,98.5%,1.5%,1.9%


alt.LayerChart(...)

Top error categories for residual plotting: ['knowledge_gap', 'generic_refusal', 'other', 'repetition_loop', 'max_tokens', 'extinct_ancient']
Saved article chart.


alt.FacetChart(...)

Saved article chart.


alt.FacetChart(...)

The service × prompt matrix above stays at the level of pipeline accounting: which cells are present, missing, or logged as failures. Notebook 04 handles manual review of individual outputs. Here I only need to know which prompt/service combinations are structurally complete enough to support later comparison.


#### Repetition-Loop Language Profile

Repetition loops are a distinctive LLM failure mode: the model produces a fluent-looking string, but the output repeats or spirals rather than giving a usable translation. I summarize them by language and service because repetition loops can otherwise look like ordinary non-empty translations in coverage charts.


In [16]:
# Repetition-loop language profile
rep = llm_err[llm_err["category"] == "repetition_loop"].copy()

print(f"Repetition-loop errors: {len(rep)} across {rep['language_code'].nunique()} languages")

# Multi-service languages (hardest cases)
multi = (
	rep.groupby(["language_code", "language_name"])
	.agg(n_services=("service", "nunique"), services=("service", lambda x: sorted(x.unique())))
	.reset_index()
	.query("n_services >= 2")
	.sort_values("n_services", ascending=False)
)
print(f"Languages flagged by ≥2 services ({len(multi)}):")
for _, r in multi.iterrows():
	print(f"  {r['language_code']} ({r['language_name']}): {r['services']}")

# Cross-check: do any also have an honest refusal from a *different* service?
print("Honest-refusal cross-service overlap (loop + honest refusal from different service):")
for lang, grp in rep.groupby("language_code"):
	rep_svcs = set(grp["service"])
	honest = llm_err[
		(llm_err["language_code"] == lang) &
		(llm_err["category"].isin(["knowledge_gap", "extinct_ancient"])) &
		(~llm_err["service"].isin(rep_svcs))
	]
	if not honest.empty:
		lang_name = grp["language_name"].iloc[0]
		print(f"  {lang} ({lang_name}): loops in {sorted(rep_svcs)}, "
			  f"honest refusal ({sorted(honest['category'].unique())}) from {sorted(honest['service'].unique())}")

print("Per-service repetition-loop count:")
print(rep["service"].value_counts().to_string())

# Family breakdown (unique languages only)
# language_family is not a column in all_dfs — derive it via get_language_family
fam_map = {
	code: get_language_family(code)
	for code in all_dfs[TARGET_TERMS[0]]["language_code"].unique()
}
rep_unique = rep.drop_duplicates("language_code").copy()
rep_unique["language_family"] = rep_unique["language_code"].map(fam_map)
print("Family breakdown (unique languages):")
print(rep_unique["language_family"].value_counts().to_string())

Repetition-loop errors: 61 across 56 languages
Languages flagged by ≥2 services (3):
  xcr (Carian): ['DeepSeek', 'OpenAI']
  xna (Ancient North Arabian): ['DeepSeek', 'OpenAI']
  zbl (Blissymbols): ['DeepSeek', 'OpenAI']
Honest-refusal cross-service overlap (loop + honest refusal from different service):
  bqv (Koro Wachi): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['OpenAI']
  bsc (bsc): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['Llama', 'OpenAI']
  eka (Ekajuk): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['OpenAI']
  gld (Nanai): loops in ['Gemma'], honest refusal (['knowledge_gap']) from ['Llama', 'OpenAI']
  hit (Hittite): loops in ['OpenAI'], honest refusal (['extinct_ancient', 'knowledge_gap']) from ['Llama']
  kro (kro): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['OpenAI']
  lab (Linear A): loops in ['Gemma'], honest refusal (['extinct_ancient']) from ['Gemini', 'OpenAI']
  lcp (Western Lawa): l

#### Refusal Grammar

Refusals are not just absences. They are model-generated explanations of inability, uncertainty, or safety boundaries. I inspect the wording of refusal and knowledge-gap messages because different services expose their limits differently: some fail through API errors, while others produce polished statements that still amount to no translation.


In [17]:
# OpenAI refusal grammar analysis
oa = llm_err[llm_err["service"] == "OpenAI"]
refusals = oa[oa["category"].isin(["generic_refusal", "knowledge_gap"])].copy()

print(f"OpenAI refusal errors: {len(refusals)} ({len(refusals[refusals['category']=='generic_refusal'])} generic + {len(refusals[refusals['category']=='knowledge_gap'])} knowledge_gap)")
print()

openings = refusals["clean_msg"].str.split().str[:8].str.join(" ")
top = openings.value_counts()
print("Top opening phrases (first 8 words):")
cumulative = 0
for i, (phrase, n) in enumerate(top.head(8).items()):
    pct = n / len(refusals) * 100
    cumulative += n
    cum_pct = cumulative / len(refusals) * 100
    print(f"  {n:3d} ({pct:5.1f}% | cum {cum_pct:5.1f}%) '{phrase}'")

top5_n = top.head(5).sum()
print(f"\nTop 5 openings cover {top5_n}/{len(refusals)} = {top5_n/len(refusals)*100:.1f}% of OpenAI refusals")

# Compare with other services' refusal patterns
print("\nComparison: refusal openings by service")
for svc in sorted(llm_err["service"].unique()):
    if svc == "OpenAI":
        continue
    svc_ref = llm_err[(llm_err["service"] == svc) & (llm_err["category"].isin(["generic_refusal", "knowledge_gap"]))]
    if len(svc_ref) == 0:
        continue
    print(f"  {svc} ({len(svc_ref)} refusals):")
    svc_openings = svc_ref["clean_msg"].str.split().str[:8].str.join(" ").value_counts()
    for phrase, n in svc_openings.head(3).items():
        print(f"    {n:3d}  '{phrase}'")


OpenAI refusal errors: 201 (115 generic + 86 knowledge_gap)

Top opening phrases (first 8 words):
  111 ( 55.2% | cum  55.2%) 'I'm sorry, but I can't provide a translation'
   66 ( 32.8% | cum  88.1%) 'I'm sorry, but I currently do not have'
    9 (  4.5% | cum  92.5%) 'I'm sorry, but I cannot provide a translation'
    5 (  2.5% | cum  95.0%) 'I'm sorry, but I don't have the capability'
    3 (  1.5% | cum  96.5%) 'I'm sorry, but as of my last update,'
    3 (  1.5% | cum  98.0%) 'I'm sorry, but I can't assist with that'
    2 (  1.0% | cum  99.0%) 'I'm sorry, but I currently don't have the'
    1 (  0.5% | cum  99.5%) 'Kutenai (ISO kut) is a language spoken by'

Top 5 openings cover 194/201 = 96.5% of OpenAI refusals

Comparison: refusal openings by service
  Gemini (11 refusals):
      2  'I cannot provide a translation of "Digital Humanities"'
      2  'I am unable to provide a translation of'
      1  'I cannot provide a translation into "xlc" because'
  Llama (87 refusals):
     

## 2.2 Data Quality and Curation

After coverage, I turn to output quality. The goal here is not to decide every disputed translation by hand. It is to identify structural problems that can be detected consistently enough to become review signals: missing rationales, script mismatch, source-term leakage, repetition, placeholders, and suspiciously short or long outputs.


### Missing Rationales

For LLM services, a translation and rationale are supposed to travel together. A translation without a rationale is harder to interpret, and a rationale without a translation usually means parsing or generation failed. This section checks those pairings across services and prompt variants, including raw prompt-service files where the loaded dataframe has already enforced translation/rationale pairing.


In [18]:
LLM_RAT_COLS = {
    "Claude":   "claude_translation_rationale",
    "OpenAI":   "openai_translation_rationale",
    "Gemini":   "gemini_translation_rationale",
    "DeepSeek": "deepseek_translation_rationale",
    "Llama":    "llama_translation_rationale",
    "Gemma":    "gemma_translation_rationale",
    "Qwen":     "qwen_translation_rationale",
    "Mistral":  "mistral_translation_rationale",
}

mismatch_rows = []
df = all_dfs[TARGET_TERMS[0]]

for variant in VARIANTS:
    vdf = df[df["prompt_variant"] == variant]
    for service, trans_col in LLM_SERVICES.items():
        rat_col = LLM_RAT_COLS.get(service)
        if trans_col not in vdf.columns or rat_col not in vdf.columns:
            continue
        has_trans = vdf[trans_col].notna() & ~vdf[trans_col].astype(str).str.strip().isin(["", "nan"])
        has_rat   = vdf[rat_col].notna()   & ~vdf[rat_col].astype(str).str.strip().isin(["", "nan"])
        t_no_r = int((has_trans & ~has_rat).sum())
        r_no_t = int((~has_trans & has_rat).sum())
        mismatch_rows.append({
            "service":             service,
            "variant":             variant,
            "n_translation":       int(has_trans.sum()),
            "n_rationale":         int(has_rat.sum()),
            "trans_no_rationale":  t_no_r,
            "rationale_no_trans":  r_no_t,
        })

mismatch_df = pd.DataFrame(mismatch_rows)

def highlight_nonzero(val):
    return "background-color: #ffe0e0" if isinstance(val, int) and val > 0 else ""

display(
    mismatch_df.style
    .map(highlight_nonzero, subset=["trans_no_rationale", "rationale_no_trans"])
    .format({
        "n_translation":      "{:,}",
        "n_rationale":        "{:,}",
        "trans_no_rationale": "{:,}",
        "rationale_no_trans": "{:,}",
    })
    .set_caption("LLM service translation ↔ rationale mismatches (red = non-zero; these rows are excluded from explore_disagreements.py)")
)

,service,variant,n_translation,n_rationale,trans_no_rationale,rationale_no_trans
0,OpenAI,minimal,797,797,0,0
1,Claude,minimal,881,881,0,0
2,Gemini,minimal,846,848,0,2
3,DeepSeek,minimal,868,870,0,2
4,Llama,minimal,787,787,0,0
5,Gemma,minimal,865,865,0,0
6,Qwen,minimal,875,877,0,2
7,Mistral,minimal,829,829,1,1
8,OpenAI,fluent_speaker,631,750,0,119
9,Claude,fluent_speaker,881,881,0,0


In [19]:
mismatch_plot = mismatch_df.melt(
    id_vars=["service", "variant"],
    value_vars=["trans_no_rationale", "rationale_no_trans"],
    var_name="mismatch_type",
    value_name="n",
)
mismatch_plot["mismatch_type"] = mismatch_plot["mismatch_type"].map({
    "trans_no_rationale": "Translation without rationale",
    "rationale_no_trans": "Rationale without translation",
})

service_order = list(LLM_SERVICES.keys())

def mismatch_panel(label, color_title):
    sub = mismatch_plot[mismatch_plot["mismatch_type"] == label].copy()
    max_n = max(int(sub["n"].max()), 1)
    heat = alt.Chart(sub).mark_rect().encode(
        x=alt.X("variant:N", sort=VARIANTS, title="Prompt variant"),
        y=alt.Y("service:N", sort=service_order, title="Service"),
        color=alt.Color(
            "n:Q",
            scale=alt.Scale(scheme="oranges", domain=[0, max_n]),
            title=color_title,
        ),
    ).properties(width=250, height=230, title=label)
    text = alt.Chart(sub[sub["n"] > 0]).mark_text(fontSize=10).encode(
        x=alt.X("variant:N", sort=VARIANTS),
        y=alt.Y("service:N", sort=service_order),
        text=alt.Text("n:Q"),
        color=alt.condition(
            alt.datum.n > max_n * 0.45,
            alt.value("white"),
            alt.value("black"),
        ),
    )
    return heat + text

translation_without_rationale_chart = mismatch_panel(
    "Translation without rationale",
    "Rows",
)
rationale_without_translation_chart = mismatch_panel(
    "Rationale without translation",
    "Rows",
)

missing_translation_rationale_chart = alt.hconcat(
    translation_without_rationale_chart,
    rationale_without_translation_chart,
    spacing=35,
).properties(
    title=alt.Title(
        "Missing Translation/Rationale Pairings by Service and Prompt",
        subtitle="Counts show rows where a translation and rationale did not travel together.",
    )
).resolve_scale(color="independent")

article_figures_dir = FIGURE_DIR
missing_pair_png = article_figures_dir / "missing_translation_rationale_pairings_by_service_prompt.png"
save_chart(missing_translation_rationale_chart, str(missing_pair_png), scale_factor=2)
print("Saved article chart.")

display(missing_translation_rationale_chart)


Saved article chart.


alt.HConcatChart(...)

### Script and Directionality

Script and directionality are basic plausibility checks. They do not prove a translation is correct, but they help catch outputs that are structurally unlikely for the target language: Latin-script stand-ins for non-Latin languages, mixed-script artifacts, copied English source terms, and right-to-left languages rendered in unexpected ways.


In [20]:
# Build script-group sets from the comprehensive language metadata
_lang_meta = read_csv_file(os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv"))
_lang_meta = _lang_meta.dropna(subset=["language_code"])

# RTL: CLDR-derived directionality column (78 codes); already has FORCE_LTR overrides applied
# — more complete than any hardcoded list and correctly excludes diq/ha/ku/uz/uz_AF
RTL_CODES = set(_lang_meta.loc[_lang_meta["directionality"] == "rtl", "language_code"])

# CJK / SE Asian: derive from primary_script
CJK_SCRIPTS = {
    "Bopomofo", "Simplified", "Traditional", "Japanese", "Korean", "Katakana",
    "Tibetan", "Myanmar", "Khmer", "Lao", "Thai", "Han",
}
cjk_from_csv = set(_lang_meta.loc[_lang_meta["primary_script"].isin(CJK_SCRIPTS), "language_code"].dropna())
# Chinese variant codes present in pipeline data but missing primary_script in the metadata CSV
CJK_EXTRA = {"zh-tw", "zh-classical", "zh-min-nan", "zh-yue", "cdo"}
CJK_CODES = cjk_from_csv | CJK_EXTRA

print(f"RTL codes: {len(RTL_CODES)}  |  CJK/SE Asian codes: {len(CJK_CODES)}")

def script_group(code):
    if code in RTL_CODES:
        return "RTL"
    if code in CJK_CODES:
        return "CJK / SE Asian"
    return "LTR (Latin/other)"

term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()
svc_cols = {s: c for s, c in HEATMAP_SERVICES.items() if c in baseline.columns}
baseline["n_services"] = baseline[[c for c in svc_cols.values()]].notna().sum(axis=1)
baseline["script"] = baseline["language_code"].apply(script_group)

script_rows = []
for service, col in svc_cols.items():
    for script, grp in baseline.groupby("script"):
        n = int(grp[col].notna().sum())
        total = len(grp)
        script_rows.append({"service": service, "script": script, "n": n, "total": total})
script_df = pd.DataFrame(script_rows)

script_order = ["LTR (Latin/other)", "RTL", "CJK / SE Asian"]
svc_order = list(HEATMAP_SERVICES.keys())
max_n_script = script_df["n"].max()

heatmap = alt.Chart(script_df).mark_rect().encode(
    x=alt.X("service:N", sort=svc_order, title=None),
    y=alt.Y("script:N", sort=script_order, title=None),
    color=alt.Color("n:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, max_n_script]),
        title="languages translated"),
    tooltip=["service:N","script:N",
             alt.Tooltip("n:Q",title="translated"),
             alt.Tooltip("total:Q",title="total in group")],
)
hmap_text = heatmap.mark_text(fontSize=10).encode(
    text="n:Q",
    color=alt.condition(alt.datum.n > max_n_script * 0.6, alt.value("white"), alt.value("black")),
)

box = alt.Chart(baseline).mark_boxplot(extent="min-max").encode(
    x=alt.X("script:N", sort=script_order, title=None),
    y=alt.Y("n_services:Q", title="services that translated it", scale=alt.Scale(domain=[0,9])),
    color=alt.Color("script:N", sort=script_order, legend=None),
    tooltip=["script:N"],
).properties(width=260, height=220, title="Coverage depth by script group")

(heatmap + hmap_text).properties(
    width=480, height=100,
    title="Languages translated per service by script group",
) & box

RTL codes: 79  |  CJK/SE Asian codes: 12


alt.VConcatChart(...)

#### Coverage by Script Group

The previous heatmap gives the compact count view. I also use the panel chart below to make the coverage pattern easier to read across the three script/directionality groups. This chart follows the same coverage logic as the language-family heatmap above: LLM services count as covering a language if any prompt variant produced a translation, while Wikipedia and the machine-translation baselines are counted from their prompt-invariant outputs.


In [21]:
term = TARGET_TERMS[0]
all_variants = all_dfs[term].copy()

avail_cols = {svc: col for svc, col in HEATMAP_SERVICES.items() if col in all_variants.columns}
script_coverage_by_lang = (
    all_variants
    .assign(**{col: all_variants[col].notna() for col in avail_cols.values()})
    .groupby("language_code")
    .agg({col: "max" for col in avail_cols.values()})
    .reset_index()
)
script_coverage_by_lang["script"] = script_coverage_by_lang["language_code"].apply(script_group)

script_group_totals = script_coverage_by_lang["script"].value_counts().to_dict()
script_panel_rows = []
for script in script_order:
    grp = script_coverage_by_lang[script_coverage_by_lang["script"] == script]
    total = int(script_group_totals.get(script, 0))
    script_label = f"{script} ({total})"
    for service, col in HEATMAP_SERVICES.items():
        n_translated = int(grp[col].sum()) if col in grp.columns else 0
        pct = round(n_translated / total * 100, 1) if total else 0.0
        script_panel_rows.append({
            "Script group": script_label,
            "script_sort": script_order.index(script),
            "Service": service,
            "Coverage": pct,
            "N": n_translated,
            "Total": total,
            "Label": f"{n_translated}/{total}",
        })
script_panel_df = pd.DataFrame(script_panel_rows)
script_panel_order = [f"{script} ({script_group_totals.get(script, 0)})" for script in script_order]

base = alt.Chart(script_panel_df).encode(
    y=alt.Y("Service:N", sort=service_order, title=None),
    x=alt.X("Coverage:Q", title="% of languages translated", scale=alt.Scale(domain=[0, 108])),
    color=alt.Color(
        "Coverage:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, 100]),
        legend=alt.Legend(title="% coverage"),
    ),
    tooltip=[
        "Script group:N", "Service:N",
        alt.Tooltip("Coverage:Q", format=".1f", title="% coverage"),
        alt.Tooltip("N:Q", title="languages translated"),
        alt.Tooltip("Total:Q", title="languages in group"),
    ],
)

bars = base.mark_bar()
labels = base.mark_text(
    align="left",
    dx=4,
    fontSize=9,
).encode(
    text="Label:N",
    color=alt.value("black"),
)

script_panel_chart = (bars + labels).properties(
    width=245,
    height=300,
).facet(
    column=alt.Column(
        "Script group:N",
        sort=script_panel_order,
        title=None,
        header=alt.Header(labelOrient="top", labelFontSize=12, labelFontWeight="bold"),
    )
).properties(
    title=alt.Title(
        "Languages Translated by Service and Script Group",
        subtitle="LLM services: any prompt variant = covered. Wikipedia community translation and MT baselines run once.",
    ),
).resolve_scale(y="shared", color="shared")

article_figures_dir = FIGURE_DIR
script_group_panels_png = article_figures_dir / "languages_translated_by_service_and_script_group_panels.png"
save_chart(script_panel_chart, str(script_group_panels_png), scale_factor=2)
print("Saved article chart.")

script_panel_chart


Saved article chart.


alt.FacetChart(...)

#### Script Agreement Across LLM Services

Here I ask whether LLM services agree on the dominant script of their outputs for the same language and prompt. Agreement does not guarantee correctness, but disagreement is useful evidence. If several services use one script and one service uses another, that outlier may be a transliteration, a fallback, or a mistake worth reviewing.

I keep this script-agreement check limited to LLM services because it asks about model behavior under comparable prompts. Wikipedia and machine-translation systems remain important reference points, but they are not prompt-responsive in the same way; including them here would turn the section into a broader source-support comparison rather than a test of LLM script stability.


In [22]:
from collections import Counter
from scripts.review.analyze_scripts import detect_dominant_script

LLM_SCRIPT_COLS = {
    'Claude':   'claude_translated_term',
    'OpenAI':   'openai_translated_term',
    'Gemini':   'gemini_translated_term',
    'DeepSeek': 'deepseek_translated_term',
    'Llama':    'llama_translated_term',
    'Gemma':    'gemma_translated_term',
    'Qwen':     'qwen_translated_term',
    'Mistral':  'mistral_translated_term',
}

df = all_dfs[TARGET_TERMS[0]]

# ── Part 1: general script agreement across all 8 LLMs ───────────────────────
agreement_rows = []
outlier_rows = []

for variant in VARIANTS:
    vdf = df[df['prompt_variant'] == variant]
    for _, row in vdf.iterrows():
        lang = row['language_code']
        fam  = get_language_family(lang)
        scripts = {}
        for svc, col in LLM_SCRIPT_COLS.items():
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ('', 'nan'):
                s = detect_dominant_script(str(val))
                if s != 'Unknown':
                    scripts[svc] = s
        if len(scripts) < 2:
            continue

        n_distinct = len(set(scripts.values()))
        agreement_rows.append({
            'language_code':     lang,
            'language_name':     row.get('language_name', lang),
            'language_family':   fam,
            'variant':           variant,
            'n_services':        len(scripts),
            'n_distinct_scripts': n_distinct,
        })

        if n_distinct > 1:
            majority_script = Counter(scripts.values()).most_common(1)[0][0]
            for svc, script in scripts.items():
                if script != majority_script:
                    outlier_rows.append({
                        'language_code':   lang,
                        'language_family': fam,
                        'variant':         variant,
                        'service':         svc,
                        'majority_script': majority_script,
                        'outlier_script':  script,
                    })

agree_df   = pd.DataFrame(agreement_rows)
outlier_df = pd.DataFrame(outlier_rows)

n_rows     = len(agree_df)
n_disagree = (agree_df['n_distinct_scripts'] > 1).sum()
print(f"Rows with ≥2 LLM translations: {n_rows:,}")
print(f"Any script disagreement: {n_disagree:,} ({n_disagree/n_rows*100:.1f}%)")
print(f"Full agreement (all same script): {(agree_df['n_distinct_scripts']==1).sum():,} "
      f"({(agree_df['n_distinct_scripts']==1).mean()*100:.1f}%)")

# Chart 1: distribution of n_distinct_scripts per row
label_map = {1: '1 — all agree', 2: '2 scripts', 3: '3 scripts', 4: '4 scripts',
             5: '5 scripts', 6: '6 scripts', 7: '7 scripts', 8: '8 scripts'}
dist = (
    agree_df['n_distinct_scripts'].value_counts().reset_index()
    .rename(columns={'n_distinct_scripts': 'n_distinct', 'count': 'count'})
)
dist['pct']   = (dist['count'] / n_rows * 100).round(1)
dist['label'] = dist['n_distinct'].map(label_map)
label_order   = [label_map[i] for i in sorted(label_map) if i in dist['n_distinct'].values]

dist_bar = alt.Chart(dist).mark_bar().encode(
    x=alt.X('label:N', sort=label_order, title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('count:Q', title='rows (language × variant)'),
    color=alt.condition(
        alt.datum.n_distinct == 1, alt.value('#388e3c'), alt.value('#d32f2f')),
    tooltip=['label:N', 'count:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
).properties(width=300, height=220, title='Script agreement across 8 LLM services')
dist_text = dist_bar.mark_text(dy=-6, fontSize=11).encode(
    text=alt.Text('pct:Q', format='.1f'), color=alt.value('black'))

# Chart 2: which service is most often the outlier?
svc_outlier = (
    outlier_df.groupby('service').size().reset_index(name='n')
    .sort_values('n', ascending=False)
)
svc_outlier['pct'] = (svc_outlier['n'] / len(outlier_df) * 100).round(1)

outlier_bar = alt.Chart(svc_outlier).mark_bar().encode(
    y=alt.Y('service:N', sort=alt.EncodingSortField('n', order='descending'), title=None),
    x=alt.X('n:Q', title='times as script outlier'),
    color=alt.Color('service:N', legend=None, scale=alt.Scale(scheme='tableau10')),
    tooltip=['service:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='% of all outlier events')],
).properties(width=300, height=200, title='Which service is most often the script outlier?')
outlier_text = outlier_bar.mark_text(align='left', dx=4, fontSize=9).encode(
    text='n:Q', color=alt.value('black'))

# Chart 3: families with most script disagreement
fam_disagree = (
    agree_df[agree_df['n_distinct_scripts'] > 1]
    .groupby('language_family')['language_code']
    .nunique().reset_index(name='n_langs')
    .sort_values('n_langs', ascending=False).head(12)
)
fam_dis_bar = alt.Chart(fam_disagree).mark_bar(color='#1976d2').encode(
    y=alt.Y('language_family:N',
            sort=alt.EncodingSortField('n_langs', order='descending'), title=None),
    x=alt.X('n_langs:Q', title='unique languages with any script disagreement'),
    tooltip=['language_family:N', 'n_langs:Q'],
).properties(width=300, height=260, title='Families with most script disagreement')
fam_dis_text = fam_dis_bar.mark_text(align='left', dx=4, fontSize=9).encode(
    text='n_langs:Q', color=alt.value('black'))

script_agreement_chart = dist_bar + dist_text
script_outlier_service_chart = outlier_bar + outlier_text
script_disagreement_family_chart = fam_dis_bar + fam_dis_text

article_figures_dir = FIGURE_DIR
script_agreement_png = article_figures_dir / "script_agreement_across_8_llm_services.png"
script_outlier_png = article_figures_dir / "script_outlier_service_frequency.png"
save_chart(script_agreement_chart, str(script_agreement_png), scale_factor=2)
save_chart(script_outlier_service_chart, str(script_outlier_png), scale_factor=2)
print("Saved article chart.")
print("Saved article chart.")

script_agreement_chart.display()
(script_outlier_service_chart | script_disagreement_family_chart).display()

# ── Part 2: Local vs API model script outlier analysis ───────────────────────
LOCAL_MODELS = ['Llama', 'Gemma', 'Qwen', 'Mistral']
API_MODELS   = ['Claude', 'OpenAI', 'Gemini', 'DeepSeek']

local_script_rows = []

for variant in VARIANTS:
    vdf = df[df['prompt_variant'] == variant]
    for _, row in vdf.iterrows():
        scripts = {}
        for svc, col in LLM_SCRIPT_COLS.items():
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ('', 'nan'):
                s = detect_dominant_script(str(val))
                if s != 'Unknown':
                    scripts[svc] = s

        api_scripts = {s: scripts[s] for s in API_MODELS if s in scripts}
        if len(api_scripts) < 2:
            continue
        majority_script = Counter(api_scripts.values()).most_common(1)[0][0]
        majority_count  = list(api_scripts.values()).count(majority_script)
        if majority_count < 2:
            continue

        for local_svc in LOCAL_MODELS:
            if local_svc not in scripts:
                continue
            if scripts[local_svc] != majority_script:
                local_script_rows.append({
                    'variant':         variant,
                    'language_code':   row['language_code'],
                    'language_name':   row.get('language_name', row['language_code']),
                    'language_family': get_language_family(row['language_code']),
                    'local_service':   local_svc,
                    'local_script':    scripts[local_svc],
                    'majority_script': majority_script,
                    'api_agree_n':     majority_count,
                })

local_sd = pd.DataFrame(local_script_rows)

print(f"\nLocal model script outliers (API majority ≥2 agree on different script): "
      f"{len(local_sd)} rows, {local_sd['language_code'].nunique() if len(local_sd) else 0} unique languages")

if len(local_sd) > 0:
    # Chart 4: which local model is most often the script outlier?
    svc_counts = (
        local_sd.groupby('local_service').size().reset_index(name='n')
        .sort_values('n', ascending=False)
    )
    svc_counts['pct'] = (svc_counts['n'] / len(local_sd) * 100).round(1)

    svc_bar = alt.Chart(svc_counts).mark_bar().encode(
        y=alt.Y('local_service:N', sort=alt.EncodingSortField('n', order='descending'), title=None),
        x=alt.X('n:Q', title='times as script outlier vs API majority'),
        color=alt.Color('local_service:N', legend=None, scale=alt.Scale(scheme='tableau10')),
        tooltip=['local_service:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='% of local outlier events')],
    ).properties(width=300, height=160, title='Which local model deviates most from API script consensus?')
    svc_text = svc_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n:Q', color=alt.value('black'))

    # Chart 5: script-pair breakdown (API majority → local model's script)
    pair_counts = (
        local_sd.groupby(['majority_script', 'local_script'])
        .size().reset_index(name='n')
        .sort_values('n', ascending=False).head(12)
    )
    pair_counts['pair'] = pair_counts['majority_script'] + ' → ' + pair_counts['local_script']

    pair_bar = alt.Chart(pair_counts).mark_bar().encode(
        y=alt.Y('pair:N', sort='-x', title=None),
        x=alt.X('n:Q', title='occurrences (across all variants)'),
        color=alt.Color('local_script:N', title="local model's script",
                        scale=alt.Scale(scheme='tableau10')),
        tooltip=['pair:N', 'n:Q', 'majority_script:N', 'local_script:N'],
    ).properties(width=380, height=280,
                 title="Local model script mismatches: API majority script → local model's script")
    pair_text = pair_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n:Q', color=alt.value('black'))

    # Chart 6: top families in local model outlier cases
    fam_local = (
        local_sd.groupby('language_family')['language_code']
        .nunique().reset_index(name='n_langs')
        .sort_values('n_langs', ascending=False).head(10)
    )
    fam_loc_bar = alt.Chart(fam_local).mark_bar(color='#8b00d4').encode(
        y=alt.Y('language_family:N',
                sort=alt.EncodingSortField('n_langs', order='descending'), title=None),
        x=alt.X('n_langs:Q', title='unique languages'),
        tooltip=['language_family:N', 'n_langs:Q'],
    ).properties(width=280, height=240, title='Families most affected by local model script switching')
    fam_loc_text = fam_loc_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n_langs:Q', color=alt.value('black'))

    local_model_script_outlier_chart = svc_bar + svc_text
    local_model_script_mismatch_chart = pair_bar + pair_text
    local_model_script_family_chart = fam_loc_bar + fam_loc_text

    local_script_mismatch_png = article_figures_dir / "local_model_script_mismatches_api_majority_to_local_script.png"
    save_chart(local_model_script_mismatch_chart, str(local_script_mismatch_png), scale_factor=2)
    print("Saved article chart.")

    local_model_script_outlier_chart.display()
    local_model_script_mismatch_chart.display()
    local_model_script_family_chart.display()
else:
    print("No local model script outlier cases found.")


Rows with ≥2 LLM translations: 3,524
Any script disagreement: 1,053 (29.9%)
Full agreement (all same script): 2,471 (70.1%)
Saved article chart.
Saved article chart.


alt.LayerChart(...)

alt.HConcatChart(...)


Local model script outliers (API majority ≥2 agree on different script): 1851 rows, 366 unique languages
Saved article chart.


alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

#### Script Disagreement by Language Family

I summarize script disagreement by family to see whether the problem is concentrated in particular parts of the dataset. Families with many registered scripts, romanization conventions, or uneven digital representation may naturally produce more disagreement. I treat this as a diagnostic view, not as a claim that family membership causes the errors.


In [23]:
# Script disagreement by language family
# agree_df and outlier_df are built in the cell above (Script Agreement Across LLM Services)
# _lang_meta is built in the Script and Directionality setup cell

scripts_per_lang = (
    _lang_meta[["language_code", "n_scripts"]]
    .dropna(subset=["language_code"])
    .copy()
)

# Summarize at language level: disagreement = any variant had n_distinct_scripts > 1
lang_disagr = (
    agree_df.groupby(["language_code", "language_family"])
    .agg(has_script_disagr=("n_distinct_scripts", lambda x: (x > 1).any()))
    .reset_index()
)
lang_disagr = lang_disagr.merge(scripts_per_lang, on="language_code", how="left")

fam_stats = (
    lang_disagr.groupby("language_family")
    .agg(
        n_langs=("language_code", "nunique"),
        n_disagr=("has_script_disagr", "sum"),
        avg_n_scripts=("n_scripts", "mean"),
    )
    .reset_index()
)
fam_stats["rate"] = fam_stats["n_disagr"] / fam_stats["n_langs"]
fam_stats = fam_stats.sort_values("rate", ascending=False)

chart = alt.Chart(fam_stats[fam_stats["n_langs"] >= 5]).mark_bar().encode(
    y=alt.Y("language_family:N", sort="-x", title=None),
    x=alt.X("rate:Q", axis=alt.Axis(format="%"), title="script disagreement rate"),
    color=alt.Color("avg_n_scripts:Q",
                    scale=alt.Scale(scheme="blues", domainMin=1),
                    title="avg registered scripts"),
    tooltip=[
        "language_family:N",
        alt.Tooltip("rate:Q", format=".1%"),
        alt.Tooltip("n_langs:Q", title="languages"),
        alt.Tooltip("n_disagr:Q", title="disagreements"),
        alt.Tooltip("avg_n_scripts:Q", format=".2f", title="avg scripts"),
    ],
).properties(width=420, height=280,
             title="Script disagreement rate by family (n ≥ 5 languages)")
display(chart)

# Drill into top 3 high-rate / low avg_n_scripts families (model-injected noise)
high_rate = fam_stats[
    (fam_stats["n_langs"] >= 5) &
    (fam_stats["rate"] > 0.5) &
    (fam_stats["avg_n_scripts"] < 1.3)
].head(3)

print("High script-disagreement rate, low expected script diversity (model-injected noise):")
for _, frow in high_rate.iterrows():
    fam = frow["language_family"]
    print(f"\n  {fam}  rate={frow['rate']:.0%}, {int(frow['n_langs'])} langs, "
          f"avg {frow['avg_n_scripts']:.2f} scripts/lang")
    # Outlier services for this family
    fam_out = (
        outlier_df[outlier_df["language_family"] == fam]
        .groupby("service")["language_code"]
        .nunique()
        .sort_values(ascending=False)
    )
    print(f"  Outlier services: {dict(fam_out)}")
    # Example languages
    ex_langs = (
        lang_disagr[
            (lang_disagr["language_family"] == fam) &
            (lang_disagr["has_script_disagr"])
        ][["language_code"]]
        .drop_duplicates()
        .merge(
            all_dfs[TARGET_TERMS[0]][["language_code", "language_name"]].drop_duplicates("language_code"),
            on="language_code", how="left"
        )
        .head(5)
    )
    for _, r in ex_langs.iterrows():
        print(f"    {r['language_code']} ({r['language_name']})")

# Languages with n_scripts > 1 but no disagreement (model correctly converges)
multi_nodisr = lang_disagr[
    (lang_disagr["n_scripts"] > 1) & (~lang_disagr["has_script_disagr"])
]
print(f"\nLanguages with ≥2 registered scripts but no cross-service disagreement: {len(multi_nodisr)}")
print("(Models converge on dominant digital-text script despite multiple registered orthographies)")


alt.Chart(...)

High script-disagreement rate, low expected script diversity (model-injected noise):

  Tai-Kadai languages  rate=100%, 11 langs, avg 1.09 scripts/lang
  Outlier services: {'Llama': np.int64(9), 'Gemma': np.int64(8), 'Gemini': np.int64(5), 'Mistral': np.int64(5), 'Claude': np.int64(4), 'OpenAI': np.int64(4), 'DeepSeek': np.int64(3), 'Qwen': np.int64(3)}
    blt (Tai Dam)
    khb (Lü)
    kht (Khamti)
    lo (Lao)
    nod (Northern Thai)

  Sino-Tibetan languages  rate=92%, 48 langs, avg 1.19 scripts/lang
  Outlier services: {'Gemma': np.int64(38), 'Llama': np.int64(35), 'Mistral': np.int64(22), 'Qwen': np.int64(22), 'Gemini': np.int64(12), 'DeepSeek': np.int64(11), 'OpenAI': np.int64(11), 'Claude': np.int64(8)}
    bap (Bantawa)
    bft (Balti)
    bo (Tibetan)
    brx (Boro)
    cdo (Min Dong Chinese)

  Eskimo-Aleut languages  rate=67%, 6 langs, avg 1.00 scripts/lang
  Outlier services: {'Mistral': np.int64(4), 'Gemini': np.int64(3), 'Gemma': np.int64(3), 'Qwen': np.int64(3), 'OpenAI

#### Mixed-Script Translations

Mixed-script output can be legitimate, especially for languages that normally combine scripts or borrow technical terms. But it can also signal source leakage, transliteration, explanatory text, or a model stitching together incompatible fragments. This section identifies outputs where a secondary script accounts for enough characters to require review.


In [24]:
import unicodedata as ud
from scripts.review.analyze_scripts import char_script

MIXED_THRESHOLD = 0.10  # secondary script must be ≥10% of script chars to count

def script_profile(text):
    """Return {script: char_count}, ignoring punctuation/digits/spaces."""
    if not isinstance(text, str) or not text.strip():
        return {}
    counts = Counter()
    for ch in text:
        cp = ord(ch)
        cat = ud.category(ch)
        if cat[0] in ('Z', 'P', 'S', 'C') or cat == 'Nd':
            continue
        s = char_script(cp)
        if s == 'Other':
            continue
        # CJK kanji + Japanese kana co-occur normally in Japanese — merge them
        s = 'CJK/Japanese' if s in ('CJK', 'Japanese') else s
        counts[s] += 1
    return dict(counts)

def _is_mixed(profile, threshold=MIXED_THRESHOLD):
    if len(profile) < 2:
        return False
    total = sum(profile.values())
    dominant = max(profile.values())
    return total > 0 and (total - dominant) / total >= threshold

# ── Build mixed-script rows across all services and variants ─────────────────
ALL_SERVICES = {**BASELINE_SERVICES, **LLM_SERVICES}
term = TARGET_TERMS[0]
df_all = all_dfs[term].copy()
df_all["language_family"] = df_all["language_code"].apply(get_language_family)

# For prompt-invariant services use minimal variant only (they do not vary by prompt)
prompt_invariant_rows_df = df_all[df_all["prompt_variant"] == "minimal"].copy()

mixed_rows = []
for service, col in ALL_SERVICES.items():
    if col not in df_all.columns:
        continue
    src = prompt_invariant_rows_df if service in PROMPT_INVARIANT_SERVICES else df_all
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        profile = script_profile(val)
        if not profile:
            continue
        total_chars = sum(profile.values())
        scripts_present = sorted(profile, key=lambda s: -profile[s])
        dominant = scripts_present[0]
        minority_frac = (total_chars - profile[dominant]) / total_chars
        mixed_rows.append({
            "service":         service,
            "language_code":   row["language_code"],
            "language_name":   row.get("language_name", row["language_code"]),
            "language_family": row["language_family"],
            "variant":         row["prompt_variant"],
            "translation":     val,
            "dominant_script": dominant,
            "n_scripts":       len(profile),
            "minority_frac":   round(minority_frac, 3),
            "is_mixed":        _is_mixed(profile),
            "scripts_str":     " + ".join(scripts_present),
        })

mixed_df = pd.DataFrame(mixed_rows)

print(f"Total translations checked: {len(mixed_df):,}")
print(f"Mixed-script ({MIXED_THRESHOLD*100:.0f}% threshold): {mixed_df['is_mixed'].sum():,} "
      f"({mixed_df['is_mixed'].mean()*100:.1f}%)")
print()
print(mixed_df[mixed_df['is_mixed']].groupby('service')['is_mixed'].sum().sort_values(ascending=False).to_string())

# ── Chart 1: raw mixed-script counts by service ──────────────────────────────
svc_mix = (
    mixed_df.groupby("service")
    .agg(n_total=("is_mixed", "count"), n_mixed=("is_mixed", "sum"))
    .assign(pct_mixed=lambda d: (d["n_mixed"] / d["n_total"] * 100).round(1))
    .reset_index()
    .sort_values("n_mixed", ascending=False)
)
svc_bar = alt.Chart(svc_mix).mark_bar().encode(
    y=alt.Y("service:N", sort=alt.EncodingSortField("n_mixed", order="descending"), title=None),
    x=alt.X("n_mixed:Q", title="mixed-script translations"),
    color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["service:N",
             alt.Tooltip("pct_mixed:Q", format=".1f", title="% mixed"),
             alt.Tooltip("n_mixed:Q", title="mixed count"),
             alt.Tooltip("n_total:Q", title="total")],
).properties(width=340, height=220, title="Mixed-script translations by service")
svc_text = svc_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text="n_mixed:Q", color=alt.value("black"))

# ── Chart 2: dominant script pairs (what gets mixed with what) ───────────────
only_mixed = mixed_df[mixed_df["is_mixed"]].copy()
pair_counts = (
    only_mixed.groupby(["scripts_str", "service"])
    .size().reset_index(name="n")
    .sort_values("n", ascending=False)
)
top_pairs = pair_counts.groupby("scripts_str")["n"].sum().nlargest(10).index
pair_top = pair_counts[pair_counts["scripts_str"].isin(top_pairs)]

pair_bar = alt.Chart(pair_top).mark_bar().encode(
    y=alt.Y("scripts_str:N", sort=alt.EncodingSortField("n", order="descending"), title=None),
    x=alt.X("n:Q", title="occurrences"),
    color=alt.Color("service:N", scale=alt.Scale(scheme="tableau10"), title="Service"),
    tooltip=["scripts_str:N", "service:N", "n:Q"],
).properties(width=380, height=240, title="Most common script mixtures (top 10 pairs)")

# ── Chart 3: affected language families ──────────────────────────────────────
fam_mix = (
    only_mixed.groupby("language_family")["language_code"]
    .nunique().reset_index(name="n_langs")
    .sort_values("n_langs", ascending=False).head(12)
)
fam_bar = alt.Chart(fam_mix).mark_bar(color="#6a1b9a").encode(
    y=alt.Y("language_family:N", sort=alt.EncodingSortField("n_langs", order="descending"), title=None),
    x=alt.X("n_langs:Q", title="unique languages with mixed-script output"),
    tooltip=["language_family:N", "n_langs:Q"],
).properties(width=300, height=240, title="Language families with most mixed-script languages")
fam_text = fam_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text="n_langs:Q", color=alt.value("black"))

mixed_script_by_service_chart = svc_bar + svc_text
common_script_mixtures_chart = pair_bar
mixed_script_family_chart = fam_bar + fam_text

article_figures_dir = FIGURE_DIR
mixed_script_by_service_png = article_figures_dir / "mixed_script_translations_by_service_counts.png"
common_script_mixtures_png = article_figures_dir / "most_common_script_mixtures_top_10_pairs.png"
save_chart(mixed_script_by_service_chart, str(mixed_script_by_service_png), scale_factor=2)
save_chart(common_script_mixtures_chart, str(common_script_mixtures_png), scale_factor=2)
print("Saved article chart.")
print("Saved article chart.")

mixed_script_by_service_chart.display()
(common_script_mixtures_chart | mixed_script_family_chart).display()

# ── Sample mixed-script translations ─────────────────────────────────────────
print("\nSample mixed-script translations (≥25% minority script):")
samples = (
    only_mixed[only_mixed["minority_frac"] >= 0.25]
    [["service","language_name","dominant_script","scripts_str","minority_frac","translation"]]
    .sort_values(["service","minority_frac"], ascending=[True, False])
    .groupby("service").head(3)
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", 60)
display(samples)

Total translations checked: 27,364
Mixed-script (10% threshold): 374 (1.4%)

service
Gemma               153
Qwen                 98
Llama                56
Mistral              37
Claude               14
Gemini               13
DeepSeek              2
Google Translate      1
Saved article chart.
Saved article chart.


alt.LayerChart(...)

alt.HConcatChart(...)


Sample mixed-script translations (≥25% minority script):


,service,language_name,dominant_script,scripts_str,minority_frac,translation
0,Claude,mey,Latin,Latin + Arabic,0.449,al-ʿulūm al-insāniyya ar-raqmiyya / العلوم الإنسانية الر...
1,Claude,Ancient North Arabian,Latin,Latin + Arabic,0.286,ḥkmت ṣnʿت
2,Claude,Sogdian,Latin,Latin + Greek,0.250,δβ'nyk 'nš'n-δ'nšn'
3,DeepSeek,Sogdian,Latin,Latin + Greek,0.250,δβ'nyk 'nš'n-δ'nšn'
4,Gemini,Cherokee,Cherokee,Cherokee + Korean,0.400,Ꮧ지털 ᏴᏫ
5,Gemini,Lisu,Myanmar,Myanmar + Lisu,0.333,နီꓫလီꓸ မူꓲဖီꓲ လီꓲယိꓲ
6,Gemini,Cherokee,Cherokee,Cherokee + Korean,0.286,Ꮧ지털 ᎭᎹᏂᏘ
7,Gemma,Pāli,Thai,Thai + Devanagari + Malayalam,0.562,ดิจิทัล മാനवीय कला
8,Gemma,Kachin,Latin,Latin + Thai + Myanmar + Khmer + Sinhala,0.515,ඩිជីថល လူမှုศาสตร์ (Di-ji-thal lu-mu-saat)
9,Gemma,Tuvinian,Cyrillic,Cyrillic + Latin,0.500,Дигитал хүннүгэ (Digital hunnuge)


#### Curating Mixed-Script Translations

The curation pass applies conservative cleanup rules from `translation_classifier.py`. Some outputs can be stripped down to a plausible translation, while others are nulled or marked as placeholders. I use this pass to make the automated review layer reproducible before moving to manual decisions.


In [25]:
from scripts.review.translation_classifier import curate_translation, curate_df

term = TARGET_TERMS[0]
raw_df = all_dfs[term].copy()

# Apply curation to all *_translated_term columns
curated_df, summary = curate_df(raw_df)

print("Curation summary per service:")
print(summary.to_string(index=False))

# ── Chart: stripped / nulled / placeholder counts by service ──────────────────
summary_long = summary.melt(
    id_vars="service", value_vars=["stripped", "nulled", "placeholder"],
    var_name="action", value_name="count"
)
summary_long = summary_long[summary_long["count"] > 0]

if summary_long.empty:
    print("\nNo mixed-script or placeholder translations found at current threshold.")
else:
    action_bar = alt.Chart(summary_long).mark_bar().encode(
        y=alt.Y("service:N", sort=alt.EncodingSortField("count", order="descending"), title=None),
        x=alt.X("count:Q", title="translations affected"),
        color=alt.Color("action:N",
            scale=alt.Scale(
                domain=["stripped", "nulled", "placeholder"],
                range=["#1976d2", "#d32f2f", "#e67e00"],
            ),
            title="Action"),
        row=alt.Row("action:N", title=None),
        tooltip=["service:N", "action:N", "count:Q"],
    ).properties(width=380, height=120)
    action_bar.display()

# ── Sample before → after table ───────────────────────────────────────────────
ALL_TERM_COLS = {**PROMPT_INVARIANT_SERVICES, **LLM_SERVICES}
sample_rows = []
for service, col in ALL_TERM_COLS.items():
    if col not in raw_df.columns:
        continue
    for (_, raw_row), (_, clean_row) in zip(raw_df.iterrows(), curated_df.iterrows()):
        before = raw_row[col]
        after  = clean_row[col]
        import pandas as _pd
        if not isinstance(before, str):
            continue
        # Only show rows where something changed
        if before == after or (_pd.isna(after) and not isinstance(before, str)):
            continue
        if isinstance(before, str) and (after is None or _pd.isna(after) or before != after):
            _, action = curate_translation(before)
            if action == "unchanged":
                continue
            after_display = str(after) if after is not None and not _pd.isna(after) else f"— {action} —"
            sample_rows.append({
                "service":  service,
                "language": raw_row.get("language_name", raw_row["language_code"]),
                "action":   action,
                "before":   before,
                "after":    after_display,
            })

sample_df = (
    pd.DataFrame(sample_rows)
    .sort_values(["action", "service"])
    .groupby(["action", "service"]).head(2)
    .reset_index(drop=True)
)

if not sample_df.empty:
    print("\nSample transformations:")
    pd.set_option("display.max_colwidth", 70)
    display(sample_df[["service", "language", "action", "before", "after"]])

# ── Coverage impact: how many languages gain/lose coverage after curation ─────
svc_cols = [c for c in ALL_TERM_COLS.values() if c in raw_df.columns]
before_cov = raw_df.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any())
after_cov  = curated_df.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any())

impact_rows = []
for col, service in {v: k for k, v in ALL_TERM_COLS.items() if v in raw_df.columns}.items():
    lost = int((before_cov[col] & ~after_cov[col]).sum())
    impact_rows.append({"service": service, "languages_losing_coverage": lost})

impact_df = pd.DataFrame(impact_rows).sort_values("languages_losing_coverage", ascending=False)
print("\nLanguages losing coverage after nulling/placeholder curation:")
print(impact_df[impact_df["languages_losing_coverage"] > 0].to_string(index=False))


Curation summary per service:
  service  unchanged  stripped  nulled  placeholder
     enmt       3524         0       0            0
wikipedia       3524         0       0            0
       gt       3524         0       0            0
lingvanex       3524         0       0            0
   claude       3518         1       0            5
    llama       3474        19      31            0
    gemma       3395        76      53            0
     qwen       3460        13      51            0
  mistral       3485        27       7            5
   openai       3512         0       0           12
 deepseek       3523         0       0            1
   gemini       3503         3       3           15


alt.Chart(...)


Sample transformations:


,service,language,action,before,after
0,Gemini,Tachelhit,nulled,تودرت ن ⵓⵎⴰⵏ ⴷ ⵓⵎⴰⵏ ⴰⴷⵉⵊⵉⵜⴰⵍ,— nulled —
1,Gemini,Lisu,nulled,နီꓫလီꓸ မူꓲဖီꓲ လီꓲယိꓲ,— nulled —
2,Gemma,Maba,nulled,ඩිජිටල් මානවwissenschaft,— nulled —
3,Gemma,Mongolian,nulled,Дижитал humanistууд,— nulled —
4,Llama,Urdu,nulled,ڈیجیٹل مानवیات,— nulled —
5,Llama,Saraiki,nulled,ڈیجیٹل مानवیات,— nulled —
6,Mistral,Tae',nulled,디지털 umanis,— nulled —
7,Mistral,Abkhaz,nulled,Информационные гumanitарные науки,— nulled —
8,Qwen,Punjabi (Western),nulled,デジタル ਮਨੇਜਮਨ ਵਿਗਿਆਨ,— nulled —
9,Qwen,Russian,nulled,Дigitalnye гуманитарные науки,— nulled —



Languages losing coverage after nulling/placeholder curation:
service  languages_losing_coverage
  Llama                          1


### Automated Review Signals (`automated_review_signals.csv`)

This section writes the language-level review signal file used downstream. The file does not make final exclusion decisions. It records evidence: whether a language has repetition loops, mixed-script outputs, placeholder terms, unicode escapes, extreme lengths, source-term leakage, refusal rationales, transliteration rationales, and related issues.

I keep this as a separate CSV because later notebooks need a stable audit trail. Automated signals can filter obvious failures, but they also guide manual review rather than replacing it.


In [26]:
from collections import defaultdict
from scripts.review.translation_classifier import (
    curate_translation, has_source_leakage,
    is_repetition_loop, has_extreme_term_length, has_short_translation, has_unicode_escape,
    is_refusal_rationale, is_transliteration_rationale, has_placeholder_claim,
    is_language_name_term, has_unexpected_rationale_language, script_mix_detail,
)

term = TARGET_TERMS[0]
df_raw = all_dfs[term].copy()

# ── 1. Missing rationale ─────────────────────────────────────────────────────
# all_dfs uses load_variant_df, which applies enforce_translation_rationale_pairing
# before returning — so mismatches are already erased there. Read the raw
# prompt_services CSVs directly to catch the original pairings.
PLACEHOLDER_RATS = {
    "no rationale provided", "no rationale", "n/a", "none",
    "not applicable", "no explanation provided", "no reason provided",
}

def _has_rat(val):
    if not isinstance(val, str) or not val.strip():
        return False
    return val.strip().rstrip(".").lower() not in PLACEHOLDER_RATS

SVC_FILE_KEY = {
    "Claude": "claude", "OpenAI": "openai", "Gemini": "gemini", "DeepSeek": "deepseek",
    "Llama": "llama", "Gemma": "gemma", "Qwen": "qwen", "Mistral": "mistral",
}
prompt_dir   = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "prompt_services")

missing_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        fpath = os.path.join(prompt_dir, f"{file_key}_{variant}_translations.csv")
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if trans_col not in raw_vdf.columns or rat_col not in raw_vdf.columns:
            continue
        has_trans = raw_vdf[trans_col].notna() & ~raw_vdf[trans_col].astype(str).str.strip().isin(["", "nan"])
        has_rat   = raw_vdf[rat_col].apply(_has_rat)
        mismatch  = (has_trans & ~has_rat) | (~has_trans & has_rat)
        for lc in raw_vdf[mismatch]["language_code"].unique():
            missing_rat_by_lang[lc].add(service)

# ── 1c. Transliteration rationale (model describes a phonetic mapping) ──────
# English-only — fluent_speaker rationales are in the target language and skipped.
transliteration_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        if variant == 'fluent_speaker':
            continue
        fpath = os.path.join(prompt_dir, f'{file_key}_{variant}_translations.csv')
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if rat_col not in raw_vdf.columns:
            continue
        for _, row in raw_vdf.iterrows():
            rat = row.get(rat_col)
            if isinstance(rat, str) and is_transliteration_rationale(rat):
                transliteration_rat_by_lang[str(row['language_code'])].add(service)

# ── 1b. Refusal rationale (model admits no translation in rationale text) ────
# English-only — fluent_speaker rationales are in the target language and skipped.
refusal_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        if variant == 'fluent_speaker':
            continue
        fpath = os.path.join(prompt_dir, f'{file_key}_{variant}_translations.csv')
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if rat_col not in raw_vdf.columns:
            continue
        for _, row in raw_vdf.iterrows():
            rat = row.get(rat_col)
            if isinstance(rat, str) and is_refusal_rationale(rat):
                refusal_rat_by_lang[str(row['language_code'])].add(service)

# ── 1d. Placeholder rationale (model admits using a placeholder/stand-in) ───
# English-only — fluent_speaker rationales are in the target language and skipped.
placeholder_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        if variant == 'fluent_speaker':
            continue
        fpath = os.path.join(prompt_dir, f'{file_key}_{variant}_translations.csv')
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if rat_col not in raw_vdf.columns:
            continue
        for _, row in raw_vdf.iterrows():
            rat = row.get(rat_col)
            if isinstance(rat, str) and has_placeholder_claim(rat):
                placeholder_rat_by_lang[str(row['language_code'])].add(service)

# ── 1e. Unexpected rationale language (rationale majority non-Latin for English variants) ─
# Skips fluent_speaker (target-language rationales are expected to be non-Latin).
unexp_rat_lang_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        if variant == 'fluent_speaker':
            continue
        fpath = os.path.join(prompt_dir, f'{file_key}_{variant}_translations.csv')
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if rat_col not in raw_vdf.columns:
            continue
        for _, row in raw_vdf.iterrows():
            rat = row.get(rat_col)
            if isinstance(rat, str) and has_unexpected_rationale_language(rat):
                unexp_rat_lang_by_lang[str(row['language_code'])].add(service)

# ── 2. Mixed script, romanization, and placeholder refusals ──────────────────
ALL_SVC_COLS = {**BASELINE_SERVICES, **LLM_SERVICES}
prompt_invariant_df = df_raw[df_raw["prompt_variant"] == "minimal"].copy()

mixed_by_lang       = defaultdict(set)
roman_by_lang       = defaultdict(set)
placeholder_by_lang = defaultdict(set)  # model explicitly refused to translate

for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        _, action = curate_translation(val)
        lc = row["language_code"]
        if action == "nulled":
            mixed_by_lang[lc].add(service)
        elif action == "stripped":
            roman_by_lang[lc].add(service)
        elif action == "placeholder":
            placeholder_by_lang[lc].add(service)

# ── 3. Script disagreement (from §1.8.1 outlier_df) ──────────────────────────
disagr_by_lang = defaultdict(set)
if not outlier_df.empty:
    for _, row in outlier_df.iterrows():
        disagr_by_lang[row["language_code"]].add(row["service"])

# ── 4. Source term in translation (untranslated leakage) ─────────────────────
# Flags any non-English translation that still contains the English source term
# ("Digital Humanities") or its initials abbreviation ("DH") as a standalone token.
source_term_by_lang = defaultdict(set)
for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        lc = row["language_code"]
        if lc == "en":
            continue
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        if has_source_leakage(val, term):
            source_term_by_lang[lc].add(service)

# ── 5. Repetition loops, extreme term length, unicode escapes ────────────────
repeat_loop_by_lang = defaultdict(set)
extreme_len_by_lang = defaultdict(set)
unicode_esc_by_lang = defaultdict(set)
short_by_lang       = defaultdict(set)

for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        lc = row["language_code"]
        if is_repetition_loop(val):
            repeat_loop_by_lang[lc].add(service)
        if has_extreme_term_length(val):
            extreme_len_by_lang[lc].add(service)
        if has_unicode_escape(val):
            unicode_esc_by_lang[lc].add(service)
        if has_short_translation(val):
            short_by_lang[lc].add(service)

# ── 5b. Language-name pass-through (term contains target language name) ──────
# Catches cases where the model returns the language name itself as the term,
# e.g. 'Mbere mbere mbere' for Mbere or 'Tasawaq baaɗaɗe' for Tasawaq. Applies
# to all variants and all services (LLM and direct).
lang_name_lookup = (
    df_raw[['language_code', 'language_name']]
    .drop_duplicates('language_code')
    .dropna(subset=['language_code'])
    .set_index('language_code')['language_name']
    .to_dict()
)
language_name_term_by_lang = defaultdict(set)
for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src_df = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src_df.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        lc = row['language_code']
        lname = lang_name_lookup.get(lc, '')
        if lname and is_language_name_term(val, lname):
            language_name_term_by_lang[lc].add(service)

# ── 6. Any script mixing (including below exclusion threshold) ──────────────
# any_mixing fires whenever secondary-script chars are present at all.
# This is a superset of has_mixed_script (nulled) + has_romanization (stripped):
# it also captures sub-threshold cases that curate_translation leaves unchanged.
any_mixing_by_lang = defaultdict(set)

for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        detail = script_mix_detail(val)
        if detail.get("any_mixing"):
            any_mixing_by_lang[row["language_code"]].add(service)

# ── Build DataFrame ───────────────────────────────────────────────────────────
lang_meta = (
    df_raw[["language_code", "language_name"]]
    .drop_duplicates("language_code")
    .dropna(subset=["language_code"])
)
lang_meta = lang_meta.copy()
lang_meta["language_family"] = lang_meta["language_code"].apply(get_language_family)

# Fix any language_name values that fell back to the language code
ref_path = os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv")
if os.path.exists(ref_path):
    _ref = pd.read_csv(ref_path, dtype=str).set_index("language_code")["language_name"]
    broken = lang_meta["language_name"] == lang_meta["language_code"]
    if broken.any():
        lang_meta.loc[broken, "language_name"] = (
            lang_meta.loc[broken, "language_code"].map(_ref)
            .fillna(lang_meta.loc[broken, "language_code"])
        )

flag_rows = []
for _, meta in lang_meta.iterrows():
    lc = meta["language_code"]
    m_svcs = sorted(missing_rat_by_lang.get(lc, set()))
    x_svcs = sorted(mixed_by_lang.get(lc, set()))
    r_svcs = sorted(roman_by_lang.get(lc, set()))
    d_svcs = sorted(disagr_by_lang.get(lc, set()))
    l_svcs = sorted(source_term_by_lang.get(lc, set()))
    p_svcs = sorted(placeholder_by_lang.get(lc, set()))
    rl_svcs = sorted(repeat_loop_by_lang.get(lc, set()))
    el_svcs = sorted(extreme_len_by_lang.get(lc, set()))
    ue_svcs = sorted(unicode_esc_by_lang.get(lc, set()))
    sh_svcs = sorted(short_by_lang.get(lc, set()))
    am_svcs = sorted(any_mixing_by_lang.get(lc, set()))
    rf_svcs = sorted(refusal_rat_by_lang.get(lc, set()))
    tr_svcs = sorted(transliteration_rat_by_lang.get(lc, set()))
    pr_svcs = sorted(placeholder_rat_by_lang.get(lc, set()))
    ln_svcs = sorted(language_name_term_by_lang.get(lc, set()))
    ur_svcs = sorted(unexp_rat_lang_by_lang.get(lc, set()))
    active = (
        (["missing_rationale"]   if m_svcs  else []) +
        (["mixed_script"]        if x_svcs  else []) +
        (["romanization"]        if r_svcs  else []) +
        (["script_disagreement"] if d_svcs  else []) +
        (["source_term"]         if l_svcs  else []) +
        (["placeholder_term"]    if p_svcs  else []) +
        (["repetition_loop"]     if rl_svcs else []) +
        (["extreme_term_length"] if el_svcs else []) +
        (["unicode_escape"]      if ue_svcs else []) +
        (["short_translation"]   if sh_svcs else []) +
        (["any_mixing"]          if am_svcs else []) +
        (["refusal_rationale"]   if rf_svcs else []) +
        (["transliteration_rationale"] if tr_svcs else []) +
        (["placeholder_rationale"] if pr_svcs else []) +
        (["language_name_term"] if ln_svcs else []) +
        (["unexpected_rationale_language"] if ur_svcs else [])
    )
    flag_rows.append({
        "language_code":               lc,
        "language_name":               meta["language_name"],
        "language_family":             meta["language_family"],
        "has_missing_rationale":       bool(m_svcs),
        "missing_rationale_services":  ";".join(m_svcs),
        "has_mixed_script":            bool(x_svcs),
        "mixed_script_services":       ";".join(x_svcs),
        "has_romanization":            bool(r_svcs),
        "romanization_services":       ";".join(r_svcs),
        "has_script_disagreement":     bool(d_svcs),
        "script_disagr_services":      ";".join(d_svcs),
        "has_source_term":             bool(l_svcs),
        "source_term_services":        ";".join(l_svcs),
        "has_placeholder_term":        bool(p_svcs),
        "placeholder_term_services":   ";".join(p_svcs),
        "has_repetition_loop":         bool(rl_svcs),
        "repetition_loop_services":    ";".join(rl_svcs),
        "has_extreme_term_length":     bool(el_svcs),
        "extreme_term_length_services":";".join(el_svcs),
        "has_unicode_escape":          bool(ue_svcs),
        "unicode_escape_services":     ";".join(ue_svcs),
        "has_short_translation":        bool(sh_svcs),
        "short_translation_services":  ";".join(sh_svcs),
        "has_any_mixing":               bool(am_svcs),
        "any_mixing_services":          ";".join(am_svcs),
        "has_refusal_rationale":              bool(rf_svcs),
        "refusal_rationale_services":         ";".join(rf_svcs),
        "has_transliteration_rationale":      bool(tr_svcs),
        "transliteration_rationale_services": ";".join(tr_svcs),
        "has_placeholder_rationale":          bool(pr_svcs),
        "placeholder_rationale_services":     ";".join(pr_svcs),
        "has_language_name_term":             bool(ln_svcs),
        "language_name_term_services":        ";".join(ln_svcs),
        "has_unexpected_rationale_language":  bool(ur_svcs),
        "unexpected_rationale_language_services": ";".join(ur_svcs),
        "automated_review_signals":   ";".join(active),
        # Backward-compatible alias for older notebooks/scripts.
        "quality_flags":               ";".join(active),
        "flag_count":                  len(active),
    })

quality_flags_df = (
    pd.DataFrame(flag_rows)
    .sort_values(["flag_count", "language_name"], ascending=[False, True])
    .reset_index(drop=True)
)

# ── Save ──────────────────────────────────────────────────────────────────────
flags_dir = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "evaluation")
os.makedirs(flags_dir, exist_ok=True)
signals_path = os.path.join(flags_dir, "automated_review_signals.csv")
quality_flags_df.to_csv(signals_path, index=False)

flagged = quality_flags_df[quality_flags_df["flag_count"] > 0]
print("Saved evaluation table.")
print(f"  Total languages  : {len(quality_flags_df)}")
print(f"  Flagged          : {len(flagged)} ({len(flagged)/len(quality_flags_df)*100:.1f}%)")
print()
for col, label in [
    ("has_missing_rationale", "Missing rationale "),
    ("has_mixed_script", "Mixed script above threshold"),
    ("has_romanization", "Romanization"),
    ("has_script_disagreement", "Script disagreement"),
    ("has_source_term", "Source term"),
    ("has_placeholder_term", "Placeholder term"),
    ("has_repetition_loop", "Repetition loop"),
    ("has_extreme_term_length", "Extreme length "),
    ("has_unicode_escape", "Unicode escape"),
    ("has_short_translation", "Short translation (<4 codepoints)"),
    ("has_any_mixing",   "Any mixing of scripts (including sub-threshold)"),
    ("has_refusal_rationale", "Refusal rationale (model admits no translation in rationale text)"),
    ("has_transliteration_rationale", "Transliteration rationale (model describes phonetic mapping)"),
    ("has_placeholder_rationale", "Placeholder rationale (model admits using a placeholder/stand-in)"),
    ("has_language_name_term", "Language-name pass-through (term contains target language name)"),
    ("has_unexpected_rationale_language", "Unexpected rationale language (English variant produced non-Latin rationale)"),
]:
    n   = int(quality_flags_df[col].sum())
    pct = n / len(quality_flags_df) * 100
    print(f"  {label}: {n:>3d}  ({pct:.1f}%)")


SIGNAL_GROUPS = {
    "pipeline_integrity": [
        "has_missing_rationale",
        "has_unexpected_rationale_language",
    ],
    "output_failure": [
        "has_mixed_script",
        "has_placeholder_term",
        "has_repetition_loop",
        "has_extreme_term_length",
        "has_unicode_escape",
        "has_refusal_rationale",
        "has_placeholder_rationale",
    ],
    "curation_search_complication": [
        "has_romanization",
        "has_source_term",
        "has_short_translation",
        "has_language_name_term",
        "has_transliteration_rationale",
    ],
    "analytical_signal": [
        "has_script_disagreement",
        "has_any_mixing",
    ],
}

signal_group_rows = []
for group, cols in SIGNAL_GROUPS.items():
    present = [c for c in cols if c in quality_flags_df.columns]
    if not present:
        continue
    n = int(quality_flags_df[present].any(axis=1).sum())
    signal_group_rows.append({
        "signal_group": group,
        "n_languages": n,
        "pct_languages": round(n / len(quality_flags_df) * 100, 1),
        "signals": "; ".join(c.replace("has_", "") for c in present),
    })

print("\nSignal groups:")
display(pd.DataFrame(signal_group_rows))

flag_count_dist = (
    quality_flags_df["flag_count"]
    .value_counts()
    .rename_axis("n_signals")
    .reset_index(name="n_languages")
    .sort_values("n_signals")
)
print("\nLanguages by number of automated review signals:")
display(flag_count_dist)

print("\nTop 10 flagged languages:")
print(quality_flags_df.head(10)[["language_name", "automated_review_signals", "flag_count"]].to_string(index=False))


Saved evaluation table.
  Total languages  : 881
  Flagged          : 848 (96.3%)

  Missing rationale : 407  (46.2%)
  Mixed script above threshold:  94  (10.7%)
  Romanization:  89  (10.1%)
  Script disagreement: 376  (42.7%)
  Source term: 420  (47.7%)
  Placeholder term:  28  (3.2%)
  Repetition loop:  14  (1.6%)
  Extreme length :  21  (2.4%)
  Unicode escape:  10  (1.1%)
  Short translation (<4 codepoints):  24  (2.7%)
  Any mixing of scripts (including sub-threshold): 236  (26.8%)
  Refusal rationale (model admits no translation in rationale text): 687  (78.0%)
  Transliteration rationale (model describes phonetic mapping): 735  (83.4%)
  Placeholder rationale (model admits using a placeholder/stand-in):  50  (5.7%)
  Language-name pass-through (term contains target language name):  78  (8.9%)
  Unexpected rationale language (English variant produced non-Latin rationale): 244  (27.7%)

Signal groups:


,signal_group,n_languages,pct_languages,signals
0,pipeline_integrity,556,63.1,missing_rationale; unexpected_rationale_language
1,output_failure,719,81.6,mixed_script; placeholder_term; repetition_loop; extreme_term_leng...
2,curation_search_complication,781,88.6,romanization; source_term; short_translation; language_name_term; ...
3,analytical_signal,411,46.7,script_disagreement; any_mixing



Languages by number of automated review signals:


,n_signals,n_languages
7,0,33
5,1,73
4,2,103
1,3,160
0,4,163
2,5,136
3,6,116
6,7,66
8,8,21
10,9,4



Top 10 flagged languages:
         language_name                                                                                                                                                                      automated_review_signals  flag_count
American Sign Language  script_disagreement;source_term;extreme_term_length;unicode_escape;short_translation;any_mixing;refusal_rationale;transliteration_rationale;language_name_term;unexpected_rationale_language          10
           Blissymbols                 missing_rationale;romanization;script_disagreement;source_term;repetition_loop;short_translation;any_mixing;refusal_rationale;transliteration_rationale;placeholder_rationale          10
             Khotanese             missing_rationale;mixed_script;romanization;script_disagreement;source_term;placeholder_term;any_mixing;refusal_rationale;transliteration_rationale;unexpected_rationale_language          10
    Large Flowery Miao missing_rationale;romanization;script_disagreement

### Error Categories × Automated Review Signals

This cross-tab asks how the automated review signals overlap with logged error categories. It helps distinguish failures that are visible in both places from failures that only appear after inspecting generated text. The useful cases are often the mismatches: a language can have no API error but still produce source leakage or mixed-script output, and a logged failure can still leave other services with usable translations.


In [27]:
# Error categories × automated review signals cross-tab
qf = pd.read_csv(
    os.path.join(DATA_DIR, "translated_terms", TARGET_TERMS[0].lower().replace(" ", "_"),
                 "evaluation", "automated_review_signals.csv"),
    converters={"language_code": str},
)

SIGNAL_METADATA = [
    {"flag": "has_missing_rationale", "label": "Missing rationale", "role": "Pipeline integrity", "what_it_flags": "Raw LLM output had a translation/rationale pairing mismatch."},
    {"flag": "has_unexpected_rationale_language", "label": "Unexpected rationale language", "role": "Pipeline integrity", "what_it_flags": "English-prompt rationale appears mostly non-Latin; fluent_speaker is excluded."},
    {"flag": "has_mixed_script", "label": "Mixed-script failure", "role": "Output failure", "what_it_flags": "Classifier marked interleaved scripts/noise as not salvageable."},
    {"flag": "has_placeholder_term", "label": "Placeholder term", "role": "Output failure", "what_it_flags": "Translation cell is a refusal, note, or placeholder rather than a term."},
    {"flag": "has_repetition_loop", "label": "Repetition loop", "role": "Output failure", "what_it_flags": "Repeated-token loop typical of generation failure."},
    {"flag": "has_extreme_term_length", "label": "Extreme term length", "role": "Output failure", "what_it_flags": "Output is too long for a term; often disclaimer text or looping."},
    {"flag": "has_unicode_escape", "label": "Unicode escape", "role": "Output failure", "what_it_flags": "Literal \\uXXXX escape appears instead of rendered script."},
    {"flag": "has_refusal_rationale", "label": "Refusal rationale", "role": "Output failure", "what_it_flags": "Rationale says no real translation/equivalent is available."},
    {"flag": "has_placeholder_rationale", "label": "Placeholder rationale", "role": "Output failure", "what_it_flags": "Rationale admits the model used a placeholder or stand-in."},
    {"flag": "has_romanization", "label": "Stripped wrapper/romanization", "role": "Curation/search complication", "what_it_flags": "Classifier stripped romanization, copied source prefix, equation wrapper, or slash wrapper."},
    {"flag": "has_source_term", "label": "Source-term leakage", "role": "Curation/search complication", "what_it_flags": "Non-English output still contains Digital Humanities or DH."},
    {"flag": "has_short_translation", "label": "Very short output", "role": "Curation/search complication", "what_it_flags": "Output has fewer than four codepoints after trimming."},
    {"flag": "has_language_name_term", "label": "Language-name pass-through", "role": "Curation/search complication", "what_it_flags": "Output contains the target language name rather than just the translated term."},
    {"flag": "has_transliteration_rationale", "label": "Transliteration rationale", "role": "Curation/search complication", "what_it_flags": "Rationale says output is phonetic/transliterated rather than translated."},
    {"flag": "has_script_disagreement", "label": "Script disagreement", "role": "Analytical signal", "what_it_flags": "LLM services disagree about the dominant script for a language/prompt."},
    {"flag": "has_any_mixing", "label": "Any script mixing", "role": "Analytical signal", "what_it_flags": "Any secondary-script characters appear, including below the exclusion threshold."},
]
SIGNAL_LABELS = {row["flag"]: row["label"] for row in SIGNAL_METADATA}
SIGNAL_ROLES = {row["flag"]: row["role"] for row in SIGNAL_METADATA}
signal_order = [row["label"] for row in SIGNAL_METADATA if row["flag"] in qf.columns]

def _truthy_series(series):
    return series.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

signal_explanation_rows = []
for row in SIGNAL_METADATA:
    flag = row["flag"]
    if flag not in qf.columns:
        continue
    n = int(_truthy_series(qf[flag]).sum())
    signal_explanation_rows.append({
        "role": row["role"],
        "signal": row["label"],
        "flag_column": flag,
        "languages_flagged": n,
        "pct_languages": round(n / len(qf) * 100, 1) if len(qf) else 0,
        "what_it_flags": row["what_it_flags"],
    })
signal_explanation_df = pd.DataFrame(signal_explanation_rows)
print("Automated review signal glossary:")
display(signal_explanation_df)

flag_cols = [row["flag"] for row in SIGNAL_METADATA if row["flag"] in qf.columns]
_err_cats  = ["max_tokens", "api_error", "empty_translation", "ollama_timeout",
              "extinct_ancient", "knowledge_gap", "generic_refusal",
              "repetition_loop", "parse_format", "other"]

# Per-language error-category set
lang_errcats = (
    llm_err.groupby("language_code")["category"]
    .apply(set)
    .reset_index()
    .rename(columns={"category": "err_cats"})
)
_cross = qf.merge(lang_errcats, on="language_code", how="left")
_cross["err_cats"] = _cross["err_cats"].apply(lambda x: x if isinstance(x, set) else set())

_rows = []
for flag in flag_cols:
    flagged = _cross[_truthy_series(_cross[flag])]
    for ecat in _err_cats:
        n = flagged["err_cats"].apply(lambda s: ecat in s).sum()
        _rows.append({
            "flag": flag.replace("has_", ""),
            "flag_column": flag,
            "flag_label": SIGNAL_LABELS.get(flag, flag.replace("has_", "")),
            "role": SIGNAL_ROLES.get(flag, "Other"),
            "error_category": ecat,
            "n": int(n),
        })

xtab = pd.DataFrame(_rows)
xtab_wide = xtab.pivot(index="flag_label", columns="error_category", values="n").fillna(0).astype(int)
print("Automated review signals × error categories (count = languages with both signals)")
print(xtab_wide[_err_cats].to_string())

# Visualise as a heatmap.
xtab_long = xtab.copy()
xtab_nonzero = xtab_long[xtab_long["n"] > 0].copy()
xtab_hm_base = alt.Chart(xtab_nonzero).mark_rect().encode(
    x=alt.X("error_category:N", sort=_err_cats, title=None,
            axis=alt.Axis(labelAngle=-35)),
    y=alt.Y("flag_label:N", sort=signal_order, title=None),
    color=alt.Color("n:Q", scale=alt.Scale(scheme="oranges"), title="languages"),
    tooltip=["role:N", "flag_label:N", "error_category:N", "n:Q"],
)
xtab_hm_text = alt.Chart(xtab_nonzero).mark_text(fontSize=9).encode(
    x=alt.X("error_category:N", sort=_err_cats),
    y=alt.Y("flag_label:N", sort=signal_order),
    text="n:Q",
    color=alt.condition(
        alt.datum.n > xtab_nonzero["n"].max() * 0.45,
        alt.value("white"),
        alt.value("black"),
    ),
)
automated_signals_error_categories_chart = (xtab_hm_base + xtab_hm_text).properties(
    width=560,
    height=360,
    title=alt.Title(
        "Automated Review Signals × Error Categories",
        subtitle="Counts are languages that have both the automated signal and at least one logged LLM error in the category.",
    ),
)

article_figures_dir = FIGURE_DIR
automated_signals_error_categories_png = article_figures_dir / "automated_review_signals_error_categories.png"
save_chart(automated_signals_error_categories_chart, str(automated_signals_error_categories_png), scale_factor=2)
print("Saved article chart.")
display(automated_signals_error_categories_chart)

# ── Honest-no doubles ────────────────────────────────────────────────────────
honest_rows = []
ph_svc = qf[_truthy_series(qf["has_placeholder_term"])][["language_code", "language_name", "placeholder_term_services"]]
for _, row in ph_svc.iterrows():
    lang = row["language_code"]
    ph_svcs = {svc.strip() for svc in str(row["placeholder_term_services"]).split(";") if svc.strip()}
    honest = llm_err[(llm_err["language_code"] == lang) &
                     (llm_err["category"].isin(["extinct_ancient", "knowledge_gap"])) &
                     (~llm_err["service"].isin(ph_svcs))]
    if not honest.empty:
        honest_rows.append({
            "language_code": lang,
            "language_name": row["language_name"],
            "placeholder_services": "; ".join(sorted(ph_svcs)),
            "honest_no_services": "; ".join(sorted(honest["service"].unique())),
            "honest_no_categories": "; ".join(sorted(honest["category"].unique())),
        })

honest_no_df = pd.DataFrame(honest_rows)
print(f"\nHonest-no doubles: {len(honest_no_df)} languages")
if not honest_no_df.empty:
    display(honest_no_df)

# ── Fabricate-vs-passthrough ─────────────────────────────────────────────────
fabricate_passthrough_df = qf[_truthy_series(qf["has_repetition_loop"]) & _truthy_series(qf["has_source_term"])][
    ["language_code", "language_name", "repetition_loop_services", "source_term_services"]
].copy()
print(f"\nFabricate-vs-passthrough: {len(fabricate_passthrough_df)} languages")
if not fabricate_passthrough_df.empty:
    display(fabricate_passthrough_df)

# ── Paper-facing signal table ────────────────────────────────────────────────
# Frequencies are non-exclusive language-level counts: at least one output for
# the language triggered the signal somewhere in the service × prompt grid.
from great_tables import GT

# Count every triggering output separately. The saved signal CSV intentionally
# collapses these occurrences to one row per language, so output frequency must
# be recomputed from the same source rows and classifier functions.
signal_output_counts = Counter()
for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        value = row.get(col)
        if not isinstance(value, str) or not value.strip():
            continue
        _, action = curate_translation(value)
        if action == "nulled":
            signal_output_counts["has_mixed_script"] += 1
        elif action == "stripped":
            signal_output_counts["has_romanization"] += 1
        elif action == "placeholder":
            signal_output_counts["has_placeholder_term"] += 1
        language_code = str(row.get("language_code", ""))
        if language_code != "en" and has_source_leakage(value, term):
            signal_output_counts["has_source_term"] += 1
        if is_repetition_loop(value):
            signal_output_counts["has_repetition_loop"] += 1
        if has_extreme_term_length(value):
            signal_output_counts["has_extreme_term_length"] += 1
        if has_unicode_escape(value):
            signal_output_counts["has_unicode_escape"] += 1
        language_name = lang_name_lookup.get(language_code, "")
        if language_name and is_language_name_term(value, language_name):
            signal_output_counts["has_language_name_term"] += 1

# Pairing and rationale-language checks use raw LLM files because the merged
# dataframe has already enforced translation/rationale pairing.
for service, trans_col in LLM_SERVICES.items():
    rat_col = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        file_path = os.path.join(prompt_dir, f"{file_key}_{variant}_translations.csv")
        if not os.path.exists(file_path):
            continue
        raw_variant = pd.read_csv(file_path)
        if trans_col in raw_variant.columns and rat_col in raw_variant.columns:
            has_translation = (
                raw_variant[trans_col].notna()
                & ~raw_variant[trans_col].astype(str).str.strip().isin(["", "nan"])
            )
            has_rationale = raw_variant[rat_col].apply(_has_rat)
            signal_output_counts["has_missing_rationale"] += int(
                ((has_translation & ~has_rationale) | (~has_translation & has_rationale)).sum()
            )
        if variant != "fluent_speaker" and rat_col in raw_variant.columns:
            signal_output_counts["has_unexpected_rationale_language"] += int(
                raw_variant[rat_col].apply(
                    lambda rationale: (
                        isinstance(rationale, str)
                        and has_unexpected_rationale_language(rationale)
                    )
                ).sum()
            )

PAPER_SIGNAL_ROWS = [
    {"flag": "has_source_term", "signal": "Source-term leakage", "detects": "English source phrase or DH remains in a non-English output", "example": "Acoli / Google: Digital Humanities", "response": "Retain as evidence; exclude from distinct multilingual search terms"},
    {"flag": "has_missing_rationale", "signal": "Missing response pair", "detects": "Translation without a rationale, or rationale without a translation", "example": "Eteocypriot / Claude: rationale present; translation empty", "response": "Treat response as incomplete"},
    {"flag": "has_unexpected_rationale_language", "signal": "Unexpected rationale language", "detects": "English-prompt rationale is predominantly in another script", "example": "Rajbanshi / Gemini rationale: রাজবংশী ভাষা...", "response": "Flag for review"},
    {"flag": "has_mixed_script", "signal": "Mixed-script failure", "detects": "Scripts are interleaved beyond the classifier threshold", "example": "Brahui / Qwen: デジタル هنسنۍ", "response": "Flag as structurally unsalvageable; review"},
    {"flag": "has_romanization", "signal": "Wrapper or romanization", "detects": "Usable term is accompanied by a source prefix, equation, or romanization", "example": "Classical Chinese / Gemma: 數人文 (Shù rénwén)", "response": "Strip wrapper; preserve cleaned term and flag"},
    {"flag": "has_language_name_term", "signal": "Language-name pass-through", "detects": "Output substitutes or embeds the target language name", "example": "Amo / DeepSeek: Amo", "response": "Flag as a possible non-translation"},
    {"flag": "has_placeholder_term", "signal": "Placeholder or refusal", "detects": "Response states that no translation was produced", "example": "Hittite / OpenAI: Translation not possible", "response": "Flag as structurally unusable"},
    {"flag": "has_extreme_term_length", "signal": "Extreme length", "detects": "A disclaimer, explanation, or loop appears in the term field", "example": "Arawak / Gemini: Arawak does not have a direct equivalent...", "response": "Flag for review"},
    {"flag": "has_repetition_loop", "signal": "Repetition loop", "detects": "One token repeats mechanically through the response", "example": "Lycian / Gemma: ḫāšḫa ḫāšḫa ḫāšḫa ḫāšḫa", "response": "Flag as generation failure"},
    {"flag": "has_unicode_escape", "signal": "Literal Unicode escape", "detects": "Character codes appear instead of rendered script", "example": r"Min Nan / Llama: \u5e73\u516c\u4f1a...", "response": "Flag as malformed output"},
]
paper_signal_table_df = pd.DataFrame(PAPER_SIGNAL_ROWS)
paper_signal_table_df["languages"] = paper_signal_table_df["flag"].map(
    lambda flag: int(_truthy_series(qf[flag]).sum())
)
paper_signal_table_df["flagged_outputs"] = paper_signal_table_df["flag"].map(
    lambda flag: int(signal_output_counts[flag])
)
paper_signal_table_df["frequency"] = paper_signal_table_df["languages"].map(
    lambda n: f"{n} ({n / len(qf):.1%})"
)
paper_signal_display = paper_signal_table_df[[
    "signal", "detects", "example", "flagged_outputs", "frequency"
]].copy()

automated_review_signal_table = (
    GT(paper_signal_display)
    .tab_header(
        title="Automated Review Signals for Translation Outputs",
        subtitle=(
            "Outputs count triggering cells; languages count any trigger among 881 rows. "
            "Signals are non-exclusive and prioritize review rather than determine accuracy."
        ),
    )
    .cols_label(
        signal="Signal", detects="What it detects", example="Representative pattern",
        flagged_outputs="Flagged outputs, n",
        frequency="Languages affected, n (%)",
    )
    .cols_width(
        signal="165px", detects="310px", example="285px",
        flagged_outputs="125px", frequency="150px",
    )
    .tab_options(
        column_labels_font_weight="bold",
        column_labels_vlines_style="solid", column_labels_vlines_width="1px",
        column_labels_vlines_color="#A9A9A9",
        table_body_vlines_style="solid", table_body_vlines_width="1px",
        table_body_vlines_color="#D0D0D0",
    )
    .tab_source_note(
        source_note=(
            "Source: automated_review_signals.csv and underlying service-prompt files. "
            "Outputs are triggering cells; languages count any trigger among 881 rows. "
            "Rationale checks apply only to LLM outputs. Signals are non-exclusive."
        )
    )
)
display(automated_review_signal_table)
automated_review_signal_table_path = article_figures_dir / "automated_review_signal_glossary_table.png"
automated_review_signal_table.gtsave(str(automated_review_signal_table_path))
print("Saved article table figure.")


Automated review signal glossary:


,role,signal,flag_column,languages_flagged,pct_languages,what_it_flags
0,Pipeline integrity,Missing rationale,has_missing_rationale,407,46.2,Raw LLM output had a translation/rationale pairing mismatch.
1,Pipeline integrity,Unexpected rationale language,has_unexpected_rationale_language,244,27.7,English-prompt rationale appears mostly non-Latin; fluent_speaker ...
2,Output failure,Mixed-script failure,has_mixed_script,94,10.7,Classifier marked interleaved scripts/noise as not salvageable.
3,Output failure,Placeholder term,has_placeholder_term,28,3.2,"Translation cell is a refusal, note, or placeholder rather than a ..."
4,Output failure,Repetition loop,has_repetition_loop,14,1.6,Repeated-token loop typical of generation failure.
5,Output failure,Extreme term length,has_extreme_term_length,21,2.4,Output is too long for a term; often disclaimer text or looping.
6,Output failure,Unicode escape,has_unicode_escape,10,1.1,Literal \uXXXX escape appears instead of rendered script.
7,Output failure,Refusal rationale,has_refusal_rationale,687,78.0,Rationale says no real translation/equivalent is available.
8,Output failure,Placeholder rationale,has_placeholder_rationale,50,5.7,Rationale admits the model used a placeholder or stand-in.
9,Curation/search complication,Stripped wrapper/romanization,has_romanization,89,10.1,"Classifier stripped romanization, copied source prefix, equation w..."


Automated review signals × error categories (count = languages with both signals)
error_category                 max_tokens  api_error  empty_translation  ollama_timeout  extinct_ancient  knowledge_gap  generic_refusal  repetition_loop  parse_format  other
flag_label                                                                                                                                                                    
Any script mixing                       6          5                  7               0                8             32               34               17             3     24
Extreme term length                     2          1                  1               0                4              7                5                1             0      4
Language-name pass-through              4          1                  1               0                2             21               13                5             0     15
Missing rationale                      32  

alt.LayerChart(...)


Honest-no doubles: 15 languages


,language_code,language_name,placeholder_services,honest_no_services,honest_no_categories
0,egy,Ancient Egyptian,Gemini,Llama,extinct_ancient
1,cad,Caddo,OpenAI,Llama,knowledge_gap
2,elx,Elamite,Gemini,Llama; Mistral; OpenAI,extinct_ancient
3,ecy,Eteocypriot,Claude; Gemini,Llama; OpenAI,extinct_ancient; knowledge_gap
4,was,Washo,OpenAI,Llama,knowledge_gap
5,aro,Araona,DeepSeek; Gemini,OpenAI,knowledge_gap
6,kro,Kru languages,Claude,OpenAI,knowledge_gap
7,kut,Kutenai,Claude,Llama; OpenAI,knowledge_gap
8,clc,Chilcotin,OpenAI,Llama,knowledge_gap
9,hit,Hittite,Gemini; OpenAI,Llama,extinct_ancient; knowledge_gap



Fabricate-vs-passthrough: 12 languages


,language_code,language_name,repetition_loop_services,source_term_services
1,zbl,Blissymbols,Mistral,Claude
9,yav,Yangben,DeepSeek,Claude;DeepSeek;OpenAI
24,xsa,Sabaean,Gemma,Claude;DeepSeek
65,xlc,Lycian,Gemma,Claude;DeepSeek
71,xmr,Meroitic,Gemma,Claude;DeepSeek;Mistral
86,ter,Tereno,Gemma,Claude;DeepSeek;Gemini;OpenAI
137,hnj,Hmong Njua,Gemma,DeepSeek
153,kro,Kru languages,Llama,Claude;DeepSeek;OpenAI
179,ttm,Northern Tutchone,DeepSeek,Claude;OpenAI
194,tce,Southern Tutchone,DeepSeek,Claude;OpenAI


GT(_tbl_data=                          signal  \
0            Source-term leakage   
1          Missing response pair   
2  Unexpected rationale language   
3           Mixed-script failure   
4        Wrapper or romanization   
5     Language-name pass-through   
6         Placeholder or refusal   
7                 Extreme length   
8                Repetition loop   
9         Literal Unicode escape   

                                                                 detects  \
0            English source phrase or DH remains in a non-English output   
1    Translation without a rationale, or rationale without a translation   
2            English-prompt rationale is predominantly in another script   
3                Scripts are interleaved beyond the classifier threshold   
4  Usable term is accompanied by a source prefix, equation, or romani...   
5                  Output substitutes or embeds the target language name   
6                       Response states that no translation was produced   
7           A disclaimer, explanation, or loop appears in the term field   
8                    One token repeats mechanically through the response   
9                      Character codes appear instead of rendered script   

                                                        example  \
0                            Acoli / Google: Digital Humanities   
1    Eteocypriot / Claude: rationale present; translation empty   
2                 Rajbanshi / Gemini rationale: রাজবংশী ভাষা...   
3                                     Brahui / Qwen: デジタル هنسنۍ   
4                   Classical Chinese / Gemma: 數人文 (Shù rénwén)   
5                                           Amo / DeepSeek: Amo   
6                    Hittite / OpenAI: Translation not possible   
7  Arawak / Gemini: Arawak does not have a direct equivalent...   
8                       Lycian / Gemma: ḫāšḫa ḫāšḫa ḫāšḫa ḫāšḫa   
9                        Min Nan / Llama: \u5e73\u516c\u4f1a...   

   flagged_outputs    frequency  
0             1398  420 (47.7%)  
1              647  407 (46.2%)  
2              330  244 (27.7%)  
3              145   94 (10.7%)  
4              139   89 (10.1%)  
5              169    78 (8.9%)  
6               38    28 (3.2%)  
7               29    21 (2.4%)  
8               15    14 (1.6%)  
9               11    10 (1.1%)  , _body=<great_tables._gt_data.Body object at 0x160fef0b0>, _boxhead=Boxhead([ColInfo(var='signal', type=<ColInfoTypeEnum.default: 1>, column_label='Signal', column_align='left', column_width='165px'), ColInfo(var='detects', type=<ColInfoTypeEnum.default: 1>, column_label='What it detects', column_align='left', column_width='310px'), ColInfo(var='example', type=<ColInfoTypeEnum.default: 1>, column_label='Representative pattern', column_align='left', column_width='285px'), ColInfo(var='flagged_outputs', type=<ColInfoTypeEnum.default: 1>, column_label='Flagged outputs, n', column_align='right', column_width='125px'), ColInfo(var='frequency', type=<ColInfoTypeEnum.default: 1>, column_label='Languages affected, n (%)', column_align='right', column_width='150px')]), _stub=<great_tables._gt_data.Stub object at 0x11c79ee90>, _spanners=Spanners([]), _heading=Heading(title='Automated Review Signals for Translation Outputs', subtitle='Outputs count triggering cells; languages count any trigger among 881 rows. Signals are non-exclusive and prioritize review rather than determine accuracy.', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11f34f110>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11f36cfc0>, _source_notes=['Source: automated_review_signals.csv and underlying service-prompt files. Outputs are triggering cells; languages count any trigger among 881 rows. Rationale checks apply only to LLM outputs. Signals are non-exclusive.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11f34f250>, _formats=[], _substitutions=[], _

Saved article table figure.


## 2.3 Manual Review & Exclusion Summary

The automated review signals in §2.2 surface potential problems. The authoritative exclusion decisions come from manual review in the HTML explorer and are saved in `manual_exclusions.csv`.

This section summarizes the current manual decisions, checks how often the automated likely-error tier would have caught analysis exclusions, and characterizes the kinds of term corrections I made by hand.


In [28]:
from scripts.review.apply_review_filters import load_manual_exclusions

term = TARGET_TERMS[0]
eval_dir = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "evaluation")
analysis_terms, search_terms, corrections = load_manual_exclusions(eval_dir)

excl_df = pd.read_csv(os.path.join(eval_dir, "manual_exclusions.csv"), dtype=str, keep_default_na=False)

print("=== Manual Exclusion Counts ===")
print(f"  analysis_exclusion : {len(analysis_terms):>4} (language, term) pairs")
print(f"  search_exclusion   : {len(search_terms):>4} (language, term) pairs  "
      f"({excl_df['service'].eq('search_exclusion').sum()} rows, "
      f"{excl_df[excl_df['service']=='search_exclusion']['language_code'].nunique()} unique langs)")
print(f"  term_correction    : {len(corrections):>4} corrections")
print()

# Overlap: languages that contain both analysis- and search-excluded terms
ae_langs = set(excl_df[excl_df["service"]=="analysis_exclusion"]["language_code"])
se_langs = set(excl_df[excl_df["service"]=="search_exclusion"]["language_code"])
print(f"  Overlap (both analysis + search exclusion): {len(ae_langs & se_langs)} languages")
print(f"  Languages containing ≥1 excluded term      : {len(ae_langs | se_langs)} languages")
print(f"  Languages with no excluded terms                       : {len(qf['language_code'].unique()) - len(ae_langs | se_langs)}")

=== Manual Exclusion Counts ===
  analysis_exclusion :  122 (language, term) pairs
  search_exclusion   :  383 (language, term) pairs  (383 rows, 226 unique langs)
  term_correction    :   94 corrections

  Overlap (both analysis + search exclusion): 37 languages
  Languages containing ≥1 excluded term      : 285 languages
  Languages with no excluded terms                       : 596


### What the Manual Decisions Exclude

The three row types in `manual_exclusions.csv` operate at the level of translation strings, not languages. An `analysis_exclusion` removes a reviewed `(language_code, term)` cluster from translation analysis wherever that exact string appears. A `search_exclusion` leaves the term available for translation analysis but prevents it from becoming a search query. A `term_correction` replaces one recorded string with a reviewed form. None of these decisions removes every translation associated with a language.

This distinction became important when I audited the exclusion loader. An earlier implementation reduced `analysis_exclusion` rows to language codes and therefore removed all data for any language containing one rejected term. The corrected loader preserves all 881 language rows and masks only the 122 reviewed translation clusters across 96 languages. The chart compares the number of saved decisions with the number of languages those decisions affect; the sample table makes the term-level unit visible without printing the full review file.


In [29]:
# Summarize manual decisions at both the saved-row and affected-language levels.
decision_specs = [
    ("Analysis exclusion", "analysis_exclusion"),
    ("Search exclusion", "search_exclusion"),
    ("Term correction", "term_correction"),
]
decision_rows = []
for label, service_key in decision_specs:
    subset = excl_df[excl_df["service"].eq(service_key)].copy()
    decision_rows.extend([
        {"decision_type": label, "measure": "Saved term decisions", "count": len(subset)},
        {"decision_type": label, "measure": "Languages affected", "count": subset["language_code"].nunique()},
    ])
manual_decision_summary_df = pd.DataFrame(decision_rows)

manual_decision_chart = alt.Chart(manual_decision_summary_df).mark_bar().encode(
    y=alt.Y(
        "decision_type:N",
        sort=["Analysis exclusion", "Search exclusion", "Term correction"],
        title=None,
    ),
    x=alt.X("count:Q", title="count"),
    yOffset=alt.YOffset("measure:N"),
    color=alt.Color(
        "measure:N",
        title=None,
        scale=alt.Scale(
            domain=["Saved term decisions", "Languages affected"],
            range=["#2563a6", "#9ecae1"],
        ),
    ),
    tooltip=["decision_type:N", "measure:N", "count:Q"],
).properties(
    width=500,
    height=170,
    title=alt.Title(
        "Manual Review Decisions Are Term-Level",
        subtitle="Saved decisions can affect multiple terms within one language; no category removes the language row.",
    ),
)
manual_decision_labels = manual_decision_chart.mark_text(
    align="left", dx=4, fontSize=10, color="black"
).encode(text="count:Q")
manual_review_exclusion_chart = manual_decision_chart + manual_decision_labels

article_figures_dir = FIGURE_DIR
manual_review_exclusion_png = article_figures_dir / "manual_review_decisions_by_scope.png"
save_chart(manual_review_exclusion_chart, str(manual_review_exclusion_png), scale_factor=2)
print("Saved article chart.")
display(manual_review_exclusion_chart)

analysis_exclusion_df = excl_df[
    excl_df["service"].eq("analysis_exclusion")
].copy()
analysis_exclusion_df["excluded_terms_for_language"] = (
    analysis_exclusion_df.groupby("language_code")["original_term"].transform("nunique")
)
analysis_exclusion_sample = (
    analysis_exclusion_df[
        ["language_code", "language_name", "original_term", "excluded_terms_for_language"]
    ]
    .sort_values(["excluded_terms_for_language", "language_code"], ascending=[False, True])
    .drop_duplicates("language_code")
    .head(12)
    .rename(columns={
        "language_code": "Language code",
        "language_name": "Language",
        "original_term": "Excluded translation string",
        "excluded_terms_for_language": "Excluded strings for language",
    })
)
print("Sample of analysis-excluded translation clusters (12 of 122):")
display(analysis_exclusion_sample)


Saved article chart.


alt.LayerChart(...)

Sample of analysis-excluded translation clusters (12 of 122):


,Language code,Language,Excluded translation string,Excluded strings for language
141,elx,Elamite,Translation not possible,3
268,str,Straits Salish,Sxʷməlc̕ Ḵwulixw.getElementsByName('content').innerHTML=,3
276,tsi,Tsimshian,"Gét’ees’u’u’u X’aam”, “Gét’ees’u’u’u Ła’a”, or “Gét’ees’u’u’u Hí’a”,",3
112,xmr,Meroitic,Translation not possible,3
191,ae,Avestan,Translation not available,2
107,cad,Caddo,Translation not available,2
281,ccr,Cacaopera,Unable to provide translation,2
256,chb,Chibcha,Chibcha does not have a direct equivalent for 'Digital Humanities'...,2
146,ett,Etruscan,Cannot be accurately translated,2
212,hit,Hittite,Translation not possible,2


### Auto-Pipeline Miss Rate

For each language containing a manually excluded translation term, I check whether the automated likely-error tier would have caught it. I count a language as caught if it has one of the same high-confidence flags used by `filter_for_analysis(mode="quality")` and the review explorer's `xall` hints.

- **Caught**: at least one likely-error signal was already present.
- **Missed**: no likely-error signal was present, so the automated layer did not surface the issue.

A high language-level miss rate would mean manual review is still doing essential work even after automated filtering. A low miss rate would mean the automated signals are capturing most of the exclusions, though still not replacing judgment.


In [30]:
qf = pd.read_csv(os.path.join(eval_dir, "automated_review_signals.csv"), converters={"language_code": str})

AUTO_CATCH_FLAGS = [
    "has_repetition_loop", "has_mixed_script", "has_placeholder_term",
    "has_unicode_escape", "has_extreme_term_length",
]
flag_cols_present = [c for c in AUTO_CATCH_FLAGS if c in qf.columns]

analysis_term_langs = {code for code, _ in analysis_terms}
ae_qf = qf[qf["language_code"].isin(analysis_term_langs)].copy()
ae_qf["any_auto_flag"] = ae_qf[flag_cols_present].apply(
    lambda row: any(str(v).strip().lower() == "true" for v in row), axis=1
)
_caught = ae_qf["any_auto_flag"].sum()
missed = len(ae_qf) - _caught
miss_rate = missed / len(ae_qf) if len(ae_qf) else 0

print(f"Languages containing analysis-excluded terms: {len(ae_qf)}")
print(f"  Caught by likely-error signals : {_caught}  ({_caught/len(ae_qf):.0%})")
print(f"  Missed by likely-error signals : {missed}  ({miss_rate:.0%})")
print()
print("Missed languages (zero likely-error signals):")
missed_df = ae_qf[~ae_qf["any_auto_flag"]][["language_code", "language_name"] + flag_cols_present].reset_index(drop=True)
print(missed_df.to_string(index=False))
print()

# Which auto-flag types most often co-occur with analysis exclusions?
print("Likely-error signal co-occurrence with languages containing analysis-excluded terms (among caught):")
for col in flag_cols_present:
    n = ae_qf[col].apply(lambda v: str(v).strip().lower() == "true").sum()
    print(f"  {col:<35} {n:>3}  ({n/len(ae_qf):.0%})")

Languages containing analysis-excluded terms: 96
  Caught by likely-error signals : 50  (52%)
  Missed by likely-error signals : 46  (48%)

Missed languages (zero likely-error signals):
language_code         language_name  has_repetition_loop  has_mixed_script  has_placeholder_term  has_unicode_escape  has_extreme_term_length
          shn                  Shan                False             False                 False               False                    False
          xna Ancient North Arabian                False             False                 False               False                    False
          bbj               Ghomala                False             False                 False               False                    False
          lab              Linear A                False             False                 False               False                    False
          nsk               Naskapi                False             False                 False        

### Term Correction Patterns

Manual term corrections are different from exclusions. They are cases where I keep the language row but revise the search/query term: stripping punctuation, removing a copied source wrapper, dropping parenthetical explanations, or replacing a malformed output with the usable part. This final section summarizes those edit patterns so later search-term generation has a clear provenance trail.


In [31]:
import re

corr_df = excl_df[excl_df["service"] == "term_correction"].copy()
corr_df = corr_df[corr_df["corrected_term"].str.strip() != ""].reset_index(drop=True)

def classify_edit(orig, corr):
    orig, corr = str(orig).strip(), str(corr).strip()
    src_term = term  # "Digital Humanities"
    # Leading / trailing punctuation stripped
    if corr == re.sub(r'^[\s\W]+|[\s\W]+$', '', orig):
        return "strip_punctuation"
    # Source term wrapper removed  (e.g. "Digital Humanities (X)" → "X")
    if src_term in orig and src_term not in corr:
        return "remove_source_wrapper"
    # Leading slash removed  (e.g. "/Dkotan" → "Dkotan")
    if orig.startswith("/") and corr == orig[1:]:
        return "remove_leading_slash"
    # Parenthetical stripped from end
    if re.search(r'\(.+\)$', orig) and corr == re.sub(r'\s*\(.+\)$', '', orig).strip():
        return "strip_trailing_parenthetical"
    # Transliteration / script cleanup (corrected has different script)
    return "other"

corr_df["edit_type"] = corr_df.apply(
    lambda r: classify_edit(r["original_term"], r["corrected_term"]), axis=1
)

print("Term correction edit types:")
print(corr_df["edit_type"].value_counts().to_string())
print(f"\nTotal corrections: {len(corr_df)}")
print()
print("Examples per edit type:")
for etype, grp in corr_df.groupby("edit_type"):
    ex = grp[["language_code", "original_term", "corrected_term"]].head(3)
    print(f"\n  [{etype}]")
    for _, r in ex.iterrows():
        print(f"    {r['language_code']}: {repr(r['original_term'][:60])} → {repr(r['corrected_term'][:60])}")

# ── Paper-facing summary: automated signal definitions + correction patterns ──
# The signal glossary is rebuilt here so this final section can stand alone if
# only the manual-review cells are rerun.
try:
    signal_explanation_df
except NameError:
    qf = pd.read_csv(os.path.join(eval_dir, "automated_review_signals.csv"), converters={"language_code": str})
    SIGNAL_METADATA = [
        {"flag": "has_missing_rationale", "label": "Missing rationale", "role": "Pipeline integrity", "what_it_flags": "Raw LLM output had a translation/rationale pairing mismatch."},
        {"flag": "has_unexpected_rationale_language", "label": "Unexpected rationale language", "role": "Pipeline integrity", "what_it_flags": "English-prompt rationale appears mostly non-Latin; fluent_speaker is excluded."},
        {"flag": "has_mixed_script", "label": "Mixed-script failure", "role": "Output failure", "what_it_flags": "Classifier marked interleaved scripts/noise as not salvageable."},
        {"flag": "has_placeholder_term", "label": "Placeholder term", "role": "Output failure", "what_it_flags": "Translation cell is a refusal, note, or placeholder rather than a term."},
        {"flag": "has_repetition_loop", "label": "Repetition loop", "role": "Output failure", "what_it_flags": "Repeated-token loop typical of generation failure."},
        {"flag": "has_extreme_term_length", "label": "Extreme term length", "role": "Output failure", "what_it_flags": "Output is too long for a term; often disclaimer text or looping."},
        {"flag": "has_unicode_escape", "label": "Unicode escape", "role": "Output failure", "what_it_flags": "Literal \\uXXXX escape appears instead of rendered script."},
        {"flag": "has_refusal_rationale", "label": "Refusal rationale", "role": "Output failure", "what_it_flags": "Rationale says no real translation/equivalent is available."},
        {"flag": "has_placeholder_rationale", "label": "Placeholder rationale", "role": "Output failure", "what_it_flags": "Rationale admits the model used a placeholder or stand-in."},
        {"flag": "has_romanization", "label": "Stripped wrapper/romanization", "role": "Curation/search complication", "what_it_flags": "Classifier stripped romanization, copied source prefix, equation wrapper, or slash wrapper."},
        {"flag": "has_source_term", "label": "Source-term leakage", "role": "Curation/search complication", "what_it_flags": "Non-English output still contains Digital Humanities or DH."},
        {"flag": "has_short_translation", "label": "Very short output", "role": "Curation/search complication", "what_it_flags": "Output has fewer than four codepoints after trimming."},
        {"flag": "has_language_name_term", "label": "Language-name pass-through", "role": "Curation/search complication", "what_it_flags": "Output contains the target language name rather than just the translated term."},
        {"flag": "has_transliteration_rationale", "label": "Transliteration rationale", "role": "Curation/search complication", "what_it_flags": "Rationale says output is phonetic/transliterated rather than translated."},
        {"flag": "has_script_disagreement", "label": "Script disagreement", "role": "Analytical signal", "what_it_flags": "LLM services disagree about the dominant script for a language/prompt."},
        {"flag": "has_any_mixing", "label": "Any script mixing", "role": "Analytical signal", "what_it_flags": "Any secondary-script characters appear, including below the exclusion threshold."},
    ]
    def _truthy_series(series):
        return series.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
    signal_explanation_rows = []
    for row in SIGNAL_METADATA:
        flag = row["flag"]
        if flag not in qf.columns:
            continue
        n = int(_truthy_series(qf[flag]).sum())
        signal_explanation_rows.append({
            "role": row["role"],
            "signal": row["label"],
            "flag_column": flag,
            "languages_flagged": n,
            "pct_languages": round(n / len(qf) * 100, 1) if len(qf) else 0,
            "what_it_flags": row["what_it_flags"],
        })
    signal_explanation_df = pd.DataFrame(signal_explanation_rows)

correction_label_map = {
    "strip_punctuation": "Strip punctuation",
    "remove_source_wrapper": "Remove source wrapper",
    "remove_leading_slash": "Remove leading slash",
    "strip_trailing_parenthetical": "Strip trailing parenthetical",
    "other": "Other/manual cleanup",
}
correction_explanation_map = {
    "strip_punctuation": "Remove punctuation wrapped around an otherwise usable term.",
    "remove_source_wrapper": "Remove copied English source text and keep the target-language portion.",
    "remove_leading_slash": "Remove slash delimiter added by the model.",
    "strip_trailing_parenthetical": "Remove explanatory parenthetical from an otherwise usable term.",
    "other": "Manual correction that does not fit the common regex patterns.",
}
correction_counts = corr_df["edit_type"].value_counts().rename_axis("edit_type").reset_index(name="n")
correction_explanation_df = correction_counts.copy()
correction_explanation_df["pattern"] = correction_explanation_df["edit_type"].map(correction_label_map).fillna(correction_explanation_df["edit_type"])
correction_explanation_df["what_it_changes"] = correction_explanation_df["edit_type"].map(correction_explanation_map).fillna("Manual correction pattern.")
correction_explanation_df = correction_explanation_df[["pattern", "edit_type", "n", "what_it_changes"]]

print("\nPaper-facing automated signal glossary:")
display(signal_explanation_df[["role", "signal", "languages_flagged", "pct_languages", "what_it_flags"]])
print("Term correction pattern glossary:")
display(correction_explanation_df)

signal_plot_df = signal_explanation_df.rename(columns={"signal": "item", "languages_flagged": "n"}).copy()
signal_plot_df["layer"] = "Automated review signal"
signal_plot_df["detail"] = signal_plot_df["role"]
signal_plot_df = signal_plot_df[["layer", "detail", "item", "n"]]

correction_plot_df = correction_explanation_df.rename(columns={"pattern": "item"}).copy()
correction_plot_df["layer"] = "Manual term correction"
correction_plot_df["detail"] = "Term correction pattern"
correction_plot_df = correction_plot_df[["layer", "detail", "item", "n"]]

review_signal_correction_plot = pd.concat([signal_plot_df, correction_plot_df], ignore_index=True)
review_signal_correction_plot["item_label"] = review_signal_correction_plot["item"] + "  (" + review_signal_correction_plot["n"].astype(str) + ")"
layer_order = ["Automated review signal", "Manual term correction"]

review_signal_correction_base = alt.Chart(review_signal_correction_plot).mark_bar().encode(
    y=alt.Y("item:N", sort=alt.EncodingSortField("n", order="descending"), title=None),
    x=alt.X("n:Q", title="languages / correction rows"),
    color=alt.Color("detail:N", title="Type", scale=alt.Scale(scheme="tableau10")),
    tooltip=["layer:N", "detail:N", "item:N", "n:Q"],
)
review_signal_correction_text = review_signal_correction_base.mark_text(
    align="left", dx=4, fontSize=9
).encode(text="n:Q", color=alt.value("black"))

automated_signal_and_correction_patterns_chart = (review_signal_correction_base + review_signal_correction_text).properties(
    width=520,
    height=220,
).facet(
    row=alt.Row("layer:N", sort=layer_order, title=None,
                header=alt.Header(labelAngle=0, labelAlign="left", labelFontSize=12, labelFontWeight="bold")),
).properties(
    title=alt.Title(
        "Automated Review Signals and Manual Term-Correction Patterns",
        subtitle="Signals are language-level flags; term corrections are manual rows that preserve a language while revising the search term.",
    ),
).resolve_scale(y="independent", x="independent")

article_figures_dir = FIGURE_DIR
signal_correction_png = article_figures_dir / "automated_review_signals_and_term_correction_patterns.png"
save_chart(automated_signal_and_correction_patterns_chart, str(signal_correction_png), scale_factor=2)
print("Saved article chart.")
display(automated_signal_and_correction_patterns_chart)


Term correction edit types:
edit_type
other                           53
strip_trailing_parenthetical    20
strip_punctuation               12
remove_source_wrapper            7
remove_leading_slash             2

Total corrections: 94

Examples per edit type:

  [other]
    ain: 'ペーシ・ウシ・イプキ・アプ・シケペ' → 'Dkara Usaamrra'
    brx: 'digitāl ḥumnāt' → 'ཐོགས་པའི་བདེན་ཆུང'
    brx: 'DIGITAL HUMANITIES' → 'ဗိုႯလ်ချမ်းဘာသာ'

  [remove_leading_slash]
    gld: '/Dkaw ᑕ+-+-+' → 'Dkaw ᑕ+-+-+'
    eky: '/Dkaw ၲေတာင္းမြန်းအစား' → 'Dkaw ၲေတာင္းမြန်းအစား'

  [remove_source_wrapper]
    brx: 'Digital Humanities ≈ ཐོགས་པའི་བདེན་ཆུང' → 'ཐོགས་པའི་བདེན་ཆུང'
    brx: 'Digital Humanities ဗိုႯလ်ချမ်းဘာသာ (Dịch vụ người lập trình ' → 'ဗိုႯလ်ချမ်းဘာသာ'
    crg: 'Digital Humanities naskapiy-owinimakan-asiniy-iskwēwak kiske' → 'naskapiy-owinimakan-asiniy-iskwēwak kiskeya-mīna-oyawiyik'

  [strip_punctuation]
    ain: '/Dkara Usaamrra' → 'Dkara Usaamrra'
    ain: '/Dkiritik Huamuuni' → 'Dkiritik Huamuuni'
    lus: '

,role,signal,languages_flagged,pct_languages,what_it_flags
0,Pipeline integrity,Missing rationale,407,46.2,Raw LLM output had a translation/rationale pairing mismatch.
1,Pipeline integrity,Unexpected rationale language,244,27.7,English-prompt rationale appears mostly non-Latin; fluent_speaker ...
2,Output failure,Mixed-script failure,94,10.7,Classifier marked interleaved scripts/noise as not salvageable.
3,Output failure,Placeholder term,28,3.2,"Translation cell is a refusal, note, or placeholder rather than a ..."
4,Output failure,Repetition loop,14,1.6,Repeated-token loop typical of generation failure.
5,Output failure,Extreme term length,21,2.4,Output is too long for a term; often disclaimer text or looping.
6,Output failure,Unicode escape,10,1.1,Literal \uXXXX escape appears instead of rendered script.
7,Output failure,Refusal rationale,687,78.0,Rationale says no real translation/equivalent is available.
8,Output failure,Placeholder rationale,50,5.7,Rationale admits the model used a placeholder or stand-in.
9,Curation/search complication,Stripped wrapper/romanization,89,10.1,"Classifier stripped romanization, copied source prefix, equation w..."


Term correction pattern glossary:


,pattern,edit_type,n,what_it_changes
0,Other/manual cleanup,other,53,Manual correction that does not fit the common regex patterns.
1,Strip trailing parenthetical,strip_trailing_parenthetical,20,Remove explanatory parenthetical from an otherwise usable term.
2,Strip punctuation,strip_punctuation,12,Remove punctuation wrapped around an otherwise usable term.
3,Remove source wrapper,remove_source_wrapper,7,Remove copied English source text and keep the target-language por...
4,Remove leading slash,remove_leading_slash,2,Remove slash delimiter added by the model.


Saved article chart.


alt.FacetChart(...)

### From Automated Signals to Manual Review Decisions

This alluvial figure follows the 599 saved term-level decisions in `manual_exclusions.csv`. Because automated signals overlap, the first stage collapses them into two mutually exclusive review tiers: a language enters the likely-error tier if it triggered mixed-script failure, placeholder/refusal, repetition loop, literal Unicode escape, or extreme length; otherwise, any remaining signal places it in the other-review-signals tier. The second stage records the manual decision, and the third shows its downstream effect. Automated signals are language-level evidence while exclusions and corrections apply to individual strings, so the ribbons show review provenance rather than claiming that a signal automatically caused a decision.


In [ ]:
manual_lineage = excl_df.copy()
likely_error_flags = [
    "has_repetition_loop", "has_mixed_script", "has_placeholder_term",
    "has_unicode_escape", "has_extreme_term_length",
]
all_review_flags = [column for column in qf.columns if column.startswith("has_")]

def flag_is_true(value):
    return str(value).strip().lower() == "true"

signal_tiers = qf[["language_code"]].copy()
signal_tiers["has_likely_error"] = qf[likely_error_flags].apply(
    lambda row: any(flag_is_true(value) for value in row), axis=1
)
signal_tiers["has_any_review_signal"] = qf[all_review_flags].apply(
    lambda row: any(flag_is_true(value) for value in row), axis=1
)
signal_tiers["signal_tier"] = signal_tiers.apply(
    lambda row: (
        "Likely-error signal" if row["has_likely_error"]
        else "Other review signals only" if row["has_any_review_signal"]
        else "No automated signal"
    ), axis=1,
)
manual_lineage = manual_lineage.merge(
    signal_tiers[["language_code", "signal_tier"]], on="language_code", how="left"
)
manual_lineage["signal_tier"] = manual_lineage["signal_tier"].fillna("No automated signal")
manual_lineage["manual_decision"] = manual_lineage["service"].map({
    "analysis_exclusion": "Analysis exclusion",
    "search_exclusion": "Search exclusion",
    "term_correction": "Term correction",
})

def downstream_treatment(row):
    if row["service"] == "analysis_exclusion":
        return "Excluded from analysis"
    if row["service"] == "search_exclusion":
        return "Excluded from GitHub search"
    edit_type = classify_edit(row["original_term"], row["corrected_term"])
    return {
        "strip_punctuation": "Correction: strip punctuation",
        "remove_source_wrapper": "Correction: remove source wrapper",
        "remove_leading_slash": "Correction: remove leading slash",
        "strip_trailing_parenthetical": "Correction: remove parenthetical",
        "other": "Correction: substantive/other",
    }[edit_type]

manual_lineage["downstream_treatment"] = manual_lineage.apply(downstream_treatment, axis=1)

from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle

stage_columns = ["signal_tier", "manual_decision", "downstream_treatment"]
stage_orders = [
    ["Likely-error signal", "Other review signals only", "No automated signal"],
    ["Analysis exclusion", "Search exclusion", "Term correction"],
    ["Excluded from analysis", "Excluded from GitHub search",
     "Correction: substantive/other", "Correction: remove parenthetical",
     "Correction: strip punctuation", "Correction: remove source wrapper",
     "Correction: remove leading slash"],
]
node_colors = {
    "Likely-error signal": "#D55E00", "Other review signals only": "#0072B2", "No automated signal": "#999999",
    "Analysis exclusion": "#CC79A7", "Search exclusion": "#E69F00", "Term correction": "#009E73",
    "Excluded from analysis": "#CC79A7", "Excluded from GitHub search": "#E69F00",
    "Correction: substantive/other": "#009E73", "Correction: remove parenthetical": "#56B4E9",
    "Correction: strip punctuation": "#0072B2", "Correction: remove source wrapper": "#F0E442",
    "Correction: remove leading slash": "#999999",
}
stage_x = [0.0, 1.0, 2.0]
node_width, gap = 0.055, 20
positions = {}
for stage, (column, order) in enumerate(zip(stage_columns, stage_orders)):
    counts = manual_lineage[column].value_counts()
    y = 0
    for label in order:
        count = int(counts.get(label, 0))
        positions[(stage, label)] = (y, y + count)
        y += count + gap

fig, ax = plt.subplots(figsize=(15, 8))
def draw_review_ribbon(x0, x1, left, right, color):
    y0a, y0b = left; y1a, y1b = right; curve = (x1 - x0) * 0.48
    path = MplPath(
        [(x0, y0a), (x0 + curve, y0a), (x1 - curve, y1a), (x1, y1a),
         (x1, y1b), (x1 - curve, y1b), (x0 + curve, y0b), (x0, y0b), (x0, y0a)],
        [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
         MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CLOSEPOLY],
    )
    ax.add_patch(PathPatch(path, facecolor=color, edgecolor="none", alpha=0.34))

for stage in range(2):
    left_col, right_col = stage_columns[stage], stage_columns[stage + 1]
    flows = manual_lineage.groupby([left_col, right_col]).size().reset_index(name="count")
    left_offsets = {label: positions[(stage, label)][0] for label in stage_orders[stage]}
    right_offsets = {label: positions[(stage + 1, label)][0] for label in stage_orders[stage + 1]}
    flows[left_col] = pd.Categorical(flows[left_col], stage_orders[stage], ordered=True)
    flows[right_col] = pd.Categorical(flows[right_col], stage_orders[stage + 1], ordered=True)
    for _, flow in flows.sort_values([left_col, right_col]).iterrows():
        left_label, right_label, count = str(flow[left_col]), str(flow[right_col]), int(flow["count"])
        left = (left_offsets[left_label], left_offsets[left_label] + count)
        right = (right_offsets[right_label], right_offsets[right_label] + count)
        draw_review_ribbon(stage_x[stage] + node_width, stage_x[stage + 1], left, right, node_colors[left_label])
        left_offsets[left_label] += count; right_offsets[right_label] += count

for stage, order in enumerate(stage_orders):
    for label in order:
        y0, y1 = positions[(stage, label)]; count = int(y1 - y0)
        if count == 0:
            continue
        ax.add_patch(Rectangle((stage_x[stage], y0), node_width, count, color=node_colors[label], ec="white", lw=0.8))
        text_x = stage_x[stage] - 0.025 if stage == 2 else stage_x[stage] + node_width + 0.025
        ax.text(text_x, (y0 + y1) / 2, f"{label}  {count:,}", va="center", ha="right" if stage == 2 else "left", fontsize=9.5)

for x, heading in zip(stage_x, ["1. Automated review tier", "2. Manual decision", "3. Downstream treatment"]):
    ax.text(x + node_width / 2, -42, heading, ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_xlim(-0.05, 2.42); ax.set_ylim(-55, 760); ax.invert_yaxis(); ax.axis("off")
fig.tight_layout()
manual_review_lineage_titleless_png = FIGURE_DIR / "manual_review_decision_lineage_titleless.png"
fig.savefig(manual_review_lineage_titleless_png, dpi=240, bbox_inches="tight", facecolor="white")
fig.suptitle("From automated review signals to 599 manual term decisions", x=0.08, y=0.98, ha="left", fontsize=16, fontweight="bold")
fig.text(0.08, 0.94, "Overlapping signals were collapsed into mutually exclusive review tiers before manual decisions were traced.", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.9])
manual_review_lineage_png = FIGURE_DIR / "manual_review_decision_lineage.png"
fig.savefig(manual_review_lineage_png, dpi=240, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {manual_review_lineage_png.name}")

# Context version: preserve the term-decision alluvial while showing the larger response-cell universe.
fig.text(
    0.08, 0.895,
    f"Generation context: {total_translations_present:,} populated translation response cells",
    fontsize=11, fontweight="bold",
    bbox={"boxstyle": "round,pad=0.45", "facecolor": "#f2f2f2", "edgecolor": "#bdbdbd"},
)
fig.text(
    0.08, 0.86,
    f"Review artifact: {len(manual_lineage):,} saved term-level decisions; repeated response forms can share a decision.",
    fontsize=10, color="#444444",
)
manual_review_context_png = FIGURE_DIR / "manual_review_decision_lineage_with_generation_context.png"
fig.savefig(manual_review_context_png, dpi=240, bbox_inches="tight", facecolor="white")
print(f"Saved: {manual_review_context_png.name}")
